In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from scipy import stats
from scipy import integrate
from scipy.signal import savgol_filter
from scipy.optimize import curve_fit
from scipy.stats import linregress
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from sklearn.linear_model import LinearRegression
import os

In [ ]:
# Define all functions

def remove_samples(df, samples_to_remove):
    return df[~df['Sample'].isin(samples_to_remove)]

def convert_time_to_minutes(time_str):
        try:
            if 'min' in time_str:
                hours, minutes = time_str.split(' h ')
                minutes = int(minutes.split(' min')[0])
            else:
                hours = time_str.split(' h')[0]
                minutes = 0
            hours = int(hours)
            total_minutes = hours * 1 + minutes/60
            return total_minutes
        except ValueError:
            return pd.NA  # Return NaN if the time string is not in the expected format
        
def apply_savgol_filter(group, window_size, polyorder=1):
    """
    Apply Savitzky-Golay filter to smooth the 'OD' values in each group.
    
    Parameters:
        group (DataFrame): Grouped data by 'Sample'.
        window_size (int): The length of the filter window. Must be a positive odd integer.
        polyorder (int): The order of the polynomial used to fit the samples. Must be less than window_size.
    
    Returns:
        DataFrame: The original group with an additional 'Smoothed_OD' column.
    """
    # Apply Savitzky-Golay filter to the 'OD' column
    group['Smoothed_OD'] = savgol_filter(group['OD'], window_size, polyorder, mode='nearest')
    return group


def calculate_log2(group):
    """
    Calculate the log2 of 'Smoothed_OD_Blank_Sub' for a given group of rows.
    
    Parameters:
        group (DataFrame): A subset of the DataFrame corresponding to a specific 'Sample' group.
    
    Returns:
        DataFrame: The original group with an additional 'Log2' column.
    """
    group['Log2'] = np.log2(group['Smoothed_OD_Blank_Sub'])
    return group
import numpy as np
import pandas as pd
from scipy.signal import find_peaks

import numpy as np
import pandas as pd
from scipy.signal import find_peaks, peak_widths

from scipy.signal import find_peaks, peak_widths
import numpy as np
import pandas as pd

from scipy.signal import find_peaks, peak_widths
import numpy as np
import pandas as pd
def extract_metrics(group):
    t = group['Time'].values
    y = group['Smoothed_OD_Blank_Sub'].values  # already smoothed

    # --- Basic growth metrics ---
    auc = np.trapz(y, t)

    max_od_idx = np.argmax(y)
    max_od = y[max_od_idx]
    max_od_time = t[max_od_idx]
    last_od = y[-1]

    dy_dt = np.gradient(y, t)
    max_slope = np.max(dy_dt)

    decline_region = dy_dt[max_od_idx:]
    decline_rate = np.mean(decline_region) if len(decline_region) > 0 else 0

    # --- Peaks with all properties explicitly computed ---
    peaks_idx, peak_props = find_peaks(
        y,
        distance=5,
        height=(None, None),        # ✅ ensures 'peak_heights' is returned
        prominence=(None, None),    # ensures prominences and bases are computed
        width=(None, None),         # ensures width metrics are computed
        threshold=None,
        plateau_size=None
    )
    num_peaks = len(peaks_idx)

    # Compute widths (needed for width-based metrics)
    widths_res = peak_widths(y, peaks_idx, rel_height=0.5)

    # --- Helper function to compute max and mean safely ---
    def max_mean(arr):
        if arr is None or len(arr) == 0:
            return np.nan, np.nan
        return np.nanmax(arr), np.nanmean(arr)

    var_dODdt = np.var(dy_dt)

    # --- AUCs over specific time ranges ---
    x, y_time = 0, 10
    z, w = 10, 14

    group1 = group[(group['Time'] >= x) & (group['Time'] <= y_time)]
    group2 = group[(group['Time'] >= z) & (group['Time'] <= w)]

    auc1 = np.trapz(group1['Smoothed_OD_Blank_Sub'], group1['Time'])
    auc2 = np.trapz(group2['Smoothed_OD_Blank_Sub'], group2['Time'])

    # --- Peak property summaries (max + mean) ---
    peak_height_max, peak_height_mean = max_mean(peak_props.get('peak_heights', []))
    peak_prom_max, peak_prom_mean = max_mean(peak_props.get('prominences', []))
    peak_left_base_max, peak_left_base_mean = max_mean(peak_props.get('left_bases', []))
    peak_right_base_max, peak_right_base_mean = max_mean(peak_props.get('right_bases', []))
    peak_width_max, peak_width_mean = max_mean(widths_res[0])
    peak_width_height_max, peak_width_height_mean = max_mean(widths_res[1])
    peak_left_ip_max, peak_left_ip_mean = max_mean(widths_res[2])
    peak_right_ip_max, peak_right_ip_mean = max_mean(widths_res[3])

    # --- Collect all results ---
    result = {
        'AUC': auc,
        'AUC1': auc1,
        'AUC2': auc2,
        'Max_OD': max_od,
        'Max_OD_Time (h)': max_od_time,  # convert minutes to hours
        'Final_OD': last_od,
        'Max_Slope': max_slope,
        'Decline_Rate': decline_rate,
        'Num_Peaks': num_peaks,
        'Var_dODdt': var_dODdt,
        'Peak_Height_Max': peak_height_max,
        'Peak_Height_Mean': peak_height_mean,
        'Peak_Prominence_Max': peak_prom_max,
        'Peak_Prominence_Mean': peak_prom_mean,
        'Peak_Left_Base_Max': peak_left_base_max,
        'Peak_Left_Base_Mean': peak_left_base_mean,
        'Peak_Right_Base_Max': peak_right_base_max,
        'Peak_Right_Base_Mean': peak_right_base_mean,
        'Peak_Width_Max': peak_width_max,
        'Peak_Width_Mean': peak_width_mean,
        'Peak_Width_Height_Max': peak_width_height_max,
        'Peak_Width_Height_Mean': peak_width_height_mean,
        'Peak_Left_IP_Max': peak_left_ip_max,
        'Peak_Left_IP_Mean': peak_left_ip_mean,
        'Peak_Right_IP_Max': peak_right_ip_max,
        'Peak_Right_IP_Mean': peak_right_ip_mean
    }

    return pd.Series(result)


def calculate_sliding_regressions(df, window_regression):
    """
    Calculate linear regressions of Log2 values according to time for each 'Sample' over sliding windows of a specified size.

    Args:
        df (pandas.DataFrame): The input DataFrame containing 'Sample', 'Time', and 'Log2' columns.
        window_regression (int): The size of the sliding window for the regression.

    Returns:
        pandas.DataFrame: A DataFrame containing the regression results for each sliding window.
    """
    regression_results = []

    for sample, sample_df in df.groupby('Sample'):
        for i in range(len(sample_df) - window_regression + 1):
            window_df = sample_df.iloc[i:i+window_regression]
            x = window_df['Time'].values
            y = window_df['Log2'].values

            # Calculate linear regression using np.polyfit
            slope, intercept = np.polyfit(x, y, 1)

            # Calculate residuals and average residuals
            y_fit = slope * x + intercept
            residuals = y - y_fit
            avg_residuals = np.mean(np.abs(residuals))

            # Calculate R-squared
            ss_res = np.sum((y - y_fit) ** 2)
            ss_tot = np.sum((y - np.mean(y)) ** 2)
            r_squared = 1 - (ss_res / ss_tot)

            regression_results.append({
                'Sample': sample,
                'Start_Time': window_df['Time'].iloc[0],
                'End_Time': window_df['Time'].iloc[-1],
                'Start_Log2': window_df['Log2'].iloc[0],
                'End_Log2': window_df['Log2'].iloc[-1],
                'Slope': slope,
                'Intercept': intercept,
                'Avg_Residuals': avg_residuals,
                'R_squared': r_squared
            })

    return pd.DataFrame(regression_results)

def plot_sample_regressions(df_back, df_reg, sample_to_plot, plot_name):
    # Filter data for the specific sample
    sample_data = df_back[df_back['Sample'] == sample_to_plot]
    sample_regressions = df_reg[df_reg['Sample'] == sample_to_plot]

    # Create the plot
    plt.figure(figsize=(10, 6))

    # Plot the Log2 values
    plt.scatter(sample_data['Time'], sample_data['Log2'], facecolors='none', edgecolors='lightgrey', s=30, label='Log2 Values')

    # Create a color map
    cmap = mcolors.LinearSegmentedColormap.from_list("", ["grey", "red"])

    # Get the range of slopes
    min_slope, max_slope = sample_regressions['Slope'].min(), sample_regressions['Slope'].max()

    # Plot each regression line with color based on slope
    for _, row in sample_regressions.iterrows():
        start_time, end_time = row['Start_Time'], row['End_Time']
        slope, intercept = row['Slope'], row['Intercept']
        
        # Normalize the slope to get a value between 0 and 1 (if multiple slopes)
        if min_slope == max_slope:
            # Use red color for the slope
            norm_slope = slope  # or you could set it to a specific value like 1 or 0
            color = 'red'
        else:
            norm_slope = (slope - min_slope) / (max_slope - min_slope)
            color = cmap(norm_slope)
           
        x = [start_time, end_time]
        y = [intercept + slope * start_time, intercept + slope * end_time]
        plt.plot(x, y, color=color, alpha=0.7, linewidth=2)

    plt.xlabel('Time')
    plt.ylabel('Log2 of OD600nm')
    plt.title(f"{plot_name} {sample_to_plot}")
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_two_regressions(title_detail, df_log2, reg1_df, reg2_df, sample_to_plot):
    sample_data = df_log2[df_log2['Sample'] == sample_to_plot]
    reg1 = reg1_df[reg1_df['Sample'] == sample_to_plot]
    reg2 = reg2_df[reg2_df['Sample'] == sample_to_plot]

    plt.figure(figsize=(10, 6))
    plt.scatter(sample_data['Time'], sample_data['Log2'], facecolors='none', edgecolors='lightgrey', s=30, label='Log2 Values')

    # First exponential phase regression (black)
    if not reg1.empty:
        x1 = [reg1['Start_Time'].values[0], reg1['End_Time'].values[0]]
        y1 = [reg1['Intercept'].values[0] + reg1['Slope'].values[0] * x1[0],
              reg1['Intercept'].values[0] + reg1['Slope'].values[0] * x1[1]]
        plt.plot(x1, y1, color='black', linewidth=2, label='First Exponential Phase')

    # Secondary exponential phase regression (red)
    if not reg2.empty:
        x2 = [reg2['Start_Time'].values[0], reg2['End_Time'].values[0]]
        y2 = [reg2['Intercept'].values[0] + reg2['Slope'].values[0] * x2[0],
              reg2['Intercept'].values[0] + reg2['Slope'].values[0] * x2[1]]
        plt.plot(x2, y2, color='red', linewidth=2, label='Secondary Exponential Phase')

    plt.xlabel('Time')
    plt.ylabel('Log2 of OD600nm')
    plt.title(f'Best regressions {title_detail} for {sample_to_plot}')
    plt.legend()
    plt.grid(True)
    plt.show()

def refine_exponential_phase(top_rows_df, df_log2, thresh):
    refined_df = top_rows_df.copy()

    for idx, row in refined_df.iterrows():
        sample = row['Sample']
        residues_thresh = row['Avg_Residuals'] * thresh 

        sample_data = df_log2[df_log2['Sample'] == sample]
        current_start, current_end = row['Start_Time'], row['End_Time']

        # Expand window towards lower time values
        current_start = expand_window(sample_data, current_start, current_end, residues_thresh, direction='backward')

        # Expand window towards higher time values
        current_end = expand_window(sample_data, current_start, current_end, residues_thresh, direction='forward')

        # Update the refined dataframe
        refined_df.at[idx, 'Start_Time'] = current_start
        refined_df.at[idx, 'End_Time'] = current_end
        refined_df.at[idx, 'Start_Log2'] = get_log2_value(sample_data, current_start)
        refined_df.at[idx, 'End_Log2'] = get_log2_value(sample_data, current_end)
        refined_df.at[idx, 'Avg_Residuals'], refined_df.at[idx, 'Slope'], refined_df.at[idx, 'Intercept'] = calculate_regression_data(sample_data, current_start, current_end)

    return refined_df

def expand_window(data, current_start, current_end, threshold, direction):
    while True:
        if direction == 'backward':
            new_time = get_previous_time(data, current_start)
            if new_time is None or exceeds_threshold(data, new_time, current_end, threshold):
                break
            current_start = new_time
        elif direction == 'forward':
            new_time = get_next_time(data, current_end)
            if new_time is None or exceeds_threshold(data, current_start, new_time, threshold):
                break
            current_end = new_time
    return current_start if direction == 'backward' else current_end

def exceeds_threshold(data, start_time, end_time, threshold):
    return calculate_regression_data(data, start_time, end_time)[0] > threshold


def calculate_regression_data(data, start_time, end_time):
    window = data[(data['Time'] >= start_time) & (data['Time'] <= end_time)]
    slope, intercept, _, _, _ = stats.linregress(window['Time'], window['Log2'])
    residuals = window['Log2'] - (slope * window['Time'] + intercept)
    avg_residuals = np.mean(np.abs(residuals))
    return avg_residuals, slope, intercept


def get_log2_value(data, time):
    return data.loc[data['Time'] == time, 'Log2'].values[0]

def get_previous_time(data, current_time):
    previous_times = data[data['Time'] < current_time]['Time']
    return previous_times.max() if not previous_times.empty else None

def get_next_time(data, current_time):
    next_times = data[data['Time'] > current_time]['Time']
    return next_times.min() if not next_times.empty else None




In [ ]:
import pandas as pd
import re

# -------------------------------
# Define variables
# -------------------------------
sheet_name = 'Table All Cycles'
skiprows = 12
nrows = 170
time_format = 'XX h XX min'

# -------------------------------
# Define replicate files
# -------------------------------
files = [
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/crispri/plate1_rep1.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/crispri/plate1_rep2.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/crispri/plate1_rep3.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/xylose/plate1_xylose_rep1.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/xylose/plate1_xylose_rep2.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/xylose/plate1_xylose_rep3.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/crispri/plate2_rep1.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/crispri/plate2_rep2.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/crispri/plate2_rep3.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/xylose/plate2_xylose_rep1.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/xylose/plate2_xylose_rep2.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/xylose/plate2_xylose_rep3.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/crispri/plate3_rep1.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/crispri/plate3_rep2.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/crispri/plate3_rep3.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/xylose/plate3_xylose_rep1.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/xylose/plate3_xylose_rep2.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/xylose/plate3_xylose_rep3.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/crispri/plate4_rep1.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/crispri/plate4_rep2.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/crispri/plate4_rep3.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/xylose/plate4_xylose_rep1.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/xylose/plate4_xylose_rep2.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/xylose/plate4_xylose_rep3.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/251009_missingreps.xlsx",
    "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/Plate2_Rep4_andMissingrep.xlsx",
     "C:/Users/arnou/Documents/thesis/Resultaten/voorpythoncode/23_10_25Plate7missingreps.xlsx"
]

# -------------------------------
# Load metadata Excel with multiple sheets
# -------------------------------
file_path_meta = 'C:/Users/arnou/Documents/thesis/Resultaten/MetaData/MetaDataNew13.xlsx'  # full path including .xlsx
metadata_excel = pd.ExcelFile(file_path_meta)

dfs = []

for i, file_path in enumerate(files):
    # -------------------------------
    # Load each replicate
    # -------------------------------
    df_rep = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=[0, 1],
        index_col=None,
        skiprows=skiprows,
        nrows=nrows
    )

    # Flatten headers
    df_rep.columns = [col[0] for col in df_rep.columns]
    df_rep.columns.values[1] = "Time"
    df_rep = df_rep[df_rep.columns[1:]]

    # Melt to long format
    df_melted = pd.melt(df_rep, id_vars='Time', var_name='Sample', value_name='OD')

    
    df_melted["source_file"] = file_path

    # -------------------------------
    # Merge with corresponding metadata sheet
    # -------------------------------
    # Select the correct sheet (sheet1 → first file, sheet2 → second file, etc.)
    sheet_name_meta = metadata_excel.sheet_names[i]  # i-th sheet corresponds to i-th file
    df_meta = pd.read_excel(file_path_meta, sheet_name=sheet_name_meta, header=0, index_col=None)

    df_merged = df_melted.merge(
        df_meta,
        left_on="Sample",
        right_on="MetaData_Well",
        how="left"
    ).rename(columns={
        "Sample": "Well",
        "MetaData_gene": "Sample"
    })

    dfs.append(df_merged)

# -------------------------------
# Combine all replicates
# -------------------------------
df_final = pd.concat(dfs, ignore_index=True)

# -------------------------------
# Reformat time
# -------------------------------
if time_format == 'XX h XX min':
    df_final['Time'] = df_final['Time'].apply(convert_time_to_minutes)
elif time_format == 'decimal hour':
    df_final['Time'] = df_final['Time'] * 60
elif time_format == 'minutes':
    df_final = df_final.rename(columns={'Time': 'Time'})
else:
    print('Unknown time format.')

print("Final merged and reformatted data:")
print(df_final)
df_melted = df_final
print(df_melted)

# -------------------------------
# Select which plate(s) to analyze
# -------------------------------
# Options:
#   'all' → all plates together
#   ['Plaat_1'] → just plate 1
#   ['Plaat_1', 'Plaat_2'] → a combination of plates
plates_to_analyze = ['all']  # change this as needed

if plates_to_analyze != ['all']:
    df_filtered = df_final[df_final['MetaData_Plaat'].isin(plates_to_analyze)].copy()
else:
    df_filtered = df_final.copy()

print(f"Data selected for plates: {plates_to_analyze}")
print(df_filtered)

df_melted = df_filtered
print(df_melted)

In [ ]:
import matplotlib.pyplot as plt

# --- Step 1: Get unique samples alphabetically ---
samples = df_melted['Sample'].unique()  
n_rows = (len(samples) + 2) // 3  # Calculate number of rows for subplots

# --- Step 2: Create figure and subplots ---
fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5 * n_rows))
axes = axes.flatten()
print('melted')
print(df_melted)
# --- Step 3: Plot each sample ---
for i, sample in enumerate(samples):
    sample_data = df_melted[df_melted['Sample'] == sample]
    print('sample data')
    print(sample_data)

    # Plot each replicate in a different color
    for (rep,xylose), rep_data in sample_data.groupby(['rep','Xylose']):
        print(rep_data)
        axes[i].plot(rep_data['Time'], rep_data['OD'], label=f'Rep {rep} {xylose}', alpha=0.8)

    axes[i].set_title(sample)
    axes[i].set_xlabel('Time (min)')
    axes[i].set_ylabel('OD')
    axes[i].legend()

# --- Step 4: Remove any unused subplots ---
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

# Optional: print sample names alphabetically
print('List of sample names (alphabetical):')
print(samples)

In [ ]:
# Define variables for analysis

# List of specific sample–replicate combinations to exclude
# Format: ('Sample name', replicate_number)
# Example: [('Strain name:  CAG74399_1; Gene target: no_sgRNA', 2), ('Strain name:  BEC00920; Gene target: gltX', 1)]


samples_to_remove = [
    ('Strain name:  BEC14530; Gene target: rnjA', 1, 'no xylose', 'Plaat_1'),
    ('Strain name:  BEC14530; Gene target: rnjA', 2, 'no xylose', 'Plaat_1'),
    ('Strain name:  BEC14530; Gene target: rnjA', 3, 'no xylose', 'Plaat_1'),
    ('Strain name:  BEC25290; Gene target: era', 1, 'no xylose', 'Plaat_1'),
    ('Strain name:  BEC25290; Gene target: era', 2, 'no xylose', 'Plaat_1'),
    ('Strain name:  BEC25290; Gene target: era', 3, 'no xylose', 'Plaat_1'),
    ('Strain name:  BEC01500; Gene target: rpsI', 1, 'no xylose', 'Plaat_1'),
    ('Strain name:  BEC01500; Gene target: rpsI', 2, 'no xylose', 'Plaat_1'),
    ('Strain name:  BEC01500; Gene target: rpsI', 3, 'no xylose', 'Plaat_1'),
    ('Strain name:  BEC16750; Gene target: asd', 1, 'no xylose', 'Plaat_4'),
    ('Strain name:  BEC16750; Gene target: asd', 2, 'no xylose', 'Plaat_4'),
    ('Strain name:  BEC16750; Gene target: asd', 3, 'no xylose', 'Plaat_4'),
    ('Strain name:  BEC14530; Gene target: rnjA', 1, 'xylose', 'Plaat_1'),
    ('Strain name:  BEC14530; Gene target: rnjA', 2, 'xylose', 'Plaat_1'),
    ('Strain name:  BEC14530; Gene target: rnjA', 3, 'xylose', 'Plaat_1'),
    ('Strain name:  BEC25290; Gene target: era', 1, 'xylose', 'Plaat_1'),
    ('Strain name:  BEC25290; Gene target: era', 2, 'xylose', 'Plaat_1'),
    ('Strain name:  BEC25290; Gene target: era', 3, 'xylose', 'Plaat_1'),
    ('Strain name:  BEC01500; Gene target: rpsI', 1, 'xylose', 'Plaat_1'),
    ('Strain name:  BEC01500; Gene target: rpsI', 2, 'xylose', 'Plaat_1'),
    ('Strain name:  BEC01500; Gene target: rpsI', 3, 'xylose', 'Plaat_1'),
    ('Strain name:  BEC16750; Gene target: asd', 1, 'xylose', 'Plaat_4'),
    ('Strain name:  BEC16750; Gene target: asd', 2, 'xylose', 'Plaat_4'),
    ('Strain name:  BEC16750; Gene target: asd', 3, 'xylose', 'Plaat_4'),
    ('Strain name:  BEC17370; Gene target: nrdI', 1, 'xylose', 'Plaat_1'),
    ('Strain name:  BEC17370; Gene target: nrdI', 2, 'xylose', 'Plaat_1'),
    ('Strain name:  BEC17370; Gene target: nrdI', 3, 'xylose', 'Plaat_1'),
    ('Strain name:  BEC00920; Gene target: gltX', 1, 'xylose', 'Plaat_1'),
    ('Strain name:  BEC00920; Gene target: gltX', 2, 'xylose', 'Plaat_1'),
    ('Strain name:  BEC00920; Gene target: gltX', 3, 'xylose', 'Plaat_1'),
    ('Strain name:  BEC35740; Gene target: tagD', 1, 'xylose', 'Plaat_3'),
    ('Strain name:  BEC35740; Gene target: tagD', 2, 'xylose', 'Plaat_3'),
    ('Strain name:  BEC35740; Gene target: tagD', 3, 'xylose', 'Plaat_3'),
    ('Strain name:  CAG74399_15; Gene target: rpsN', 2, 'xylose', 'Plaat_1'),
('Strain name:  BEC15890; Gene target: plsX', 3, 'xylose', 'Plaat_1'),
('Strain name:  BEC18100; Gene target: parC', 1, 'xylose', 'Plaat_1'),
('Strain name:  BEC22840; Gene target: engA', 2, 'xylose', 'Plaat_1'),
('Strain name:  BEC28850; Gene target: rplT', 2, 'xylose', 'Plaat_1'),
('Strain name:  BEC15180; Gene target: murE', 3, 'xylose', 'Plaat_2'),
('Strain name:  BEC13300; Gene target: mgtE', 2, 'no xylose', 'Plaat_2'),
('Strain name:  BEC13300; Gene target: mgtE', 3, 'no xylose', 'Plaat_2'),
('Strain name:  BEC06070; Gene target: ydiP', 1, 'xylose', 'Plaat_2'),
('Strain name:  BEC22780; Gene target: folE', 2, 'xylose', 'Plaat_2'),
('Strain name:  BEC21810; Gene target: dfrA', 3, 'xylose', 'Plaat_2'),
('Strain name:  BEC01040; Gene target: rplJ', 2, 'xylose', 'Plaat_3'),
('Strain name:  BEC01080; Gene target: rpoC', 2, 'xylose', 'Plaat_3'),
('Strain name:  BEC15220; Gene target: murG', 2, 'xylose', 'Plaat_3'),
('Strain name:  BEC16040; Gene target: rplS', 2, 'xylose', 'Plaat_3'),
('Strain name:  BEC16580; Gene target: polC', 1, 'no xylose', 'Plaat_3'),
('Strain name:  BEC17910; Gene target: yneF', 2, 'xylose', 'Plaat_3'),
('Strain name:  BEC01360; Gene target: secY', 2, 'xylose', 'Plaat_4'),
('Strain name:  BEC01430; Gene target: rpoA', 3, 'no xylose', 'Plaat_4'),
('Strain name:  BEC15290.226; Gene target: ftsZ226', 2, 'xylose', 'Plaat_4'),
('Strain name:  BEC15230; Gene target: murB', 3, 'no xylose', 'Plaat_4'),
('Strain name:  BEC15940; Gene target: smc', 2, 'xylose', 'Plaat_4'),
('Strain name:  BEC15990; Gene target: rpsP', 3, 'xylose', 'Plaat_4'),
('Strain name:  BEC24310; Gene target: folD', 3, 'no xylose', 'Plaat_4'),
('Strain name:  CAG74399_89; Gene target: no_sgRNA', 1, 'xylose', 'Plaat_4'),
('Strain name:  CAG74399_12; Gene target: no_sgRNA', 1, 'xylose', 'Plaat_2'),
('Strain name:  CAG74399_12; Gene target: no_sgRNA', 2, 'xylose', 'Plaat_2'),
('Strain name:  CAG74399_12; Gene target: no_sgRNA', 3, 'xylose', 'Plaat_2'),
('Strain name:  CAG74399_12; Gene target: no_sgRNA', 1, 'no xylose', 'Plaat_2'),
('Strain name:  CAG74399_12; Gene target: no_sgRNA', 2, 'no xylose', 'Plaat_2'),
('Strain name:  CAG74399_12; Gene target: no_sgRNA', 3, 'no xylose', 'Plaat_2'),
('Strain name:  CAG74399_39; Gene target: no_sgRNA', 1, 'xylose', 'Plaat_4'),
('Strain name:  CAG74399_56; Gene target: no_sgRNA', 3, 'no xylose', 'Plaat_4'),
('Strain name:  CAG74399_59; Gene target: no_sgRNA', 3, 'no xylose', 'Plaat_4'),
('Strain name:  CAG74399_62; Gene target: no_sgRNA', 1, 'xylose', 'Plaat_2'),
('Strain name:  CAG74399_62; Gene target: no_sgRNA', 2, 'xylose', 'Plaat_2'),
('Strain name:  CAG74399_62; Gene target: no_sgRNA', 3, 'xylose', 'Plaat_2'),
('Strain name:  CAG74399_62; Gene target: no_sgRNA', 1, 'no xylose', 'Plaat_2'),
('Strain name:  CAG74399_62; Gene target: no_sgRNA', 2, 'no xylose', 'Plaat_2'),
('Strain name:  CAG74399_62; Gene target: no_sgRNA', 3, 'no xylose', 'Plaat_2'),
('Strain name:  CAG74399_75; Gene target: no_sgRNA', 3, 'xylose', 'Plaat_4'),
('Strain name:  CAG74399_75; Gene target: no_sgRNA', 2, 'xylose', 'Plaat_4'),
('Strain name:  CAG74399_75; Gene target: no_sgRNA', 1, 'xylose', 'Plaat_4'),
('Strain name:  CAG74399_77; Gene target: no_sgRNA', 1, 'no xylose', 'Plaat_4'),
('Strain name:  CAG74399_83; Gene target: no_sgRNA', 1, 'no xylose', 'Plaat_4'),
('Strain name:  CAG74399_84; Gene target: no_sgRNA', 3, 'no xylose', 'Plaat_4'),
('Strain name:  CAG74399_86; Gene target: no_sgRNA', 1, 'xylose', 'Plaat_4'),
('Strain name:  CAG74399_86; Gene target: no_sgRNA', 2, 'xylose', 'Plaat_4'),
('Strain name:  CAG74399_86; Gene target: no_sgRNA', 3, 'xylose', 'Plaat_4'),
('Strain name:  CAG74399_86; Gene target: no_sgRNA', 1, 'no xylose', 'Plaat_4'),
('Strain name:  CAG74399_86; Gene target: no_sgRNA', 2, 'no xylose', 'Plaat_4'),
('Strain name:  CAG74399_86; Gene target: no_sgRNA', 3, 'no xylose', 'Plaat_4'),
('Strain name:  BEC16550; Gene target: dxr', 1, 'no xylose', 'Plaat_1'),
('Strain name:  BEC16550; Gene target: dxr', 2, 'no xylose', 'Plaat_1'),
('Strain name:  BEC16550; Gene target: dxr', 3, 'no xylose', 'Plaat_1'),
('Strain name:  BEC28850; Gene target: rplT', 1, 'no xylose', 'Plaat_1'),
('Strain name:  BEC28850; Gene target: rplT', 2, 'no xylose', 'Plaat_1'),
('Strain name:  BEC28850; Gene target: rplT', 3, 'no xylose', 'Plaat_1'),
('Strain name:  BEC28870; Gene target: infC', 1, 'no xylose', 'Plaat_1'),
('Strain name:  BEC28870; Gene target: infC', 2, 'no xylose', 'Plaat_1'),
('Strain name:  BEC28870; Gene target: infC', 3, 'no xylose', 'Plaat_1'),
('Strain name:  BEC01500; Gene target: rpsI', 1, 'no xylose', 'Plaat_1'),
('Strain name:  BEC01500; Gene target: rpsI', 2, 'no xylose', 'Plaat_1'),
('Strain name:  BEC01500; Gene target: rpsI', 3, 'no xylose', 'Plaat_1'),
('Strain name:  BEC14530; Gene target: rnjA', 1, 'no xylose', 'Plaat_1'),
('Strain name:  BEC14530; Gene target: rnjA', 2, 'no xylose', 'Plaat_1'),
('Strain name:  BEC14530; Gene target: rnjA', 3, 'no xylose', 'Plaat_1'),
('Strain name:  BEC31650; Gene target: mrpF', 1, 'no xylose', 'Plaat_2'),
('Strain name:  BEC31650; Gene target: mrpF', 2, 'no xylose', 'Plaat_2'),
('Strain name:  BEC31650; Gene target: mrpF', 3, 'no xylose', 'Plaat_2'),
('Strain name:  BEC32670; Gene target: sufB', 1, 'no xylose', 'Plaat_2'),
('Strain name:  BEC32670; Gene target: sufB', 2, 'no xylose', 'Plaat_2'),
('Strain name:  BEC32670; Gene target: sufB', 3, 'no xylose', 'Plaat_2'),
('Strain name:  BEC01320.2; Gene target: rplR-2', 1, 'no xylose', 'Plaat_3'),
('Strain name:  BEC01320.2; Gene target: rplR-2', 2, 'no xylose', 'Plaat_3'),
('Strain name:  BEC01320.2; Gene target: rplR-2', 3, 'no xylose', 'Plaat_3'),
('Strain name:  BEC01340; Gene target: rpmD', 1, 'no xylose', 'Plaat_3'),
('Strain name:  BEC01340; Gene target: rpmD', 2, 'no xylose', 'Plaat_3'),
('Strain name:  BEC01340; Gene target: rpmD', 3, 'no xylose', 'Plaat_3'),
('Strain name:  BEC17380; Gene target: nrdE', 1, 'no xylose', 'Plaat_3'),
('Strain name:  BEC17380; Gene target: nrdE', 2, 'no xylose', 'Plaat_3'),
('Strain name:  BEC17380; Gene target: nrdE', 3, 'no xylose', 'Plaat_3'),
('Strain name:  BEC25200; Gene target: sigA', 1, 'no xylose', 'Plaat_4'),
('Strain name:  BEC25200; Gene target: sigA', 2, 'no xylose', 'Plaat_4'),
('Strain name:  BEC25200; Gene target: sigA', 3, 'no xylose', 'Plaat_4'),
('Strain name:  BEC01770; Gene target: glmM', 1, 'xylose', 'Plaat_1'),
('Strain name:  BEC01770; Gene target: glmM', 2, 'xylose', 'Plaat_1'),
('Strain name:  BEC01770; Gene target: glmM', 3, 'xylose', 'Plaat_1'),
('Strain name:  BEC15910; Gene target: fabG', 1, 'xylose', 'Plaat_1'),
('Strain name:  BEC15910; Gene target: fabG', 2, 'xylose', 'Plaat_1'),
('Strain name:  BEC15910; Gene target: fabG', 3, 'xylose', 'Plaat_1'),
('Strain name:  BEC25270; Gene target: glyQ', 1, 'xylose', 'Plaat_1'),
('Strain name:  BEC25270; Gene target: glyQ', 2, 'xylose', 'Plaat_1'),
('Strain name:  BEC25270; Gene target: glyQ', 3, 'xylose', 'Plaat_1'),
('Strain name:  BEC28870; Gene target: infC', 1, 'xylose', 'Plaat_1'),
('Strain name:  BEC28870; Gene target: infC', 2, 'xylose', 'Plaat_1'),
('Strain name:  BEC28870; Gene target: infC', 3, 'xylose', 'Plaat_1'),
('Strain name:  BEC_TRNA_23; Gene target: trnH', 1, 'xylose', 'Plaat_1'),
('Strain name:  BEC_TRNA_23; Gene target: trnH', 2, 'xylose', 'Plaat_1'),
('Strain name:  BEC_TRNA_23; Gene target: trnH', 3, 'xylose', 'Plaat_1'),
('Strain name:  BEC00460; Gene target: ipk', 1, 'xylose', 'Plaat_1'),
('Strain name:  BEC00460; Gene target: ipk', 2, 'xylose', 'Plaat_1'),
('Strain name:  BEC00460; Gene target: ipk', 3, 'xylose', 'Plaat_1'),
('Strain name:  BEC01500; Gene target: rpsI', 1, 'xylose', 'Plaat_2'),
('Strain name:  BEC01500; Gene target: rpsI', 2, 'xylose', 'Plaat_2'),
('Strain name:  BEC01500; Gene target: rpsI', 3, 'xylose', 'Plaat_2'),
('Strain name:  BEC00380; Gene target: metS', 1, 'xylose', 'Plaat_2'),
('Strain name:  BEC00380; Gene target: metS', 2, 'xylose', 'Plaat_2'),
('Strain name:  BEC00380; Gene target: metS', 3, 'xylose', 'Plaat_2'),
('Strain name:  BEC16530; Gene target: uppS', 1, 'xylose', 'Plaat_2'),
('Strain name:  BEC16530; Gene target: uppS', 2, 'xylose', 'Plaat_2'),
('Strain name:  BEC16530; Gene target: uppS', 3, 'xylose', 'Plaat_2'),
('Strain name:  BEC27560; Gene target: hisS', 1, 'xylose', 'Plaat_2'),
('Strain name:  BEC27560; Gene target: hisS', 2, 'xylose', 'Plaat_2'),
('Strain name:  BEC27560; Gene target: hisS', 3, 'xylose', 'Plaat_2'),
('Strain name:  BEC14190; Gene target: dapL', 1, 'xylose', 'Plaat_2'),
('Strain name:  BEC14190; Gene target: dapL', 2, 'xylose', 'Plaat_2'),
('Strain name:  BEC14190; Gene target: dapL', 3, 'xylose', 'Plaat_2'),
('Strain name:  BEC22420; Gene target: panC', 1, 'xylose', 'Plaat_2'),
('Strain name:  BEC22420; Gene target: panC', 2, 'xylose', 'Plaat_2'),
('Strain name:  BEC22420; Gene target: panC', 3, 'xylose', 'Plaat_2'),
('Strain name:  BEC16920; Gene target: pgsA', 1, 'xylose', 'Plaat_2'),
('Strain name:  BEC16920; Gene target: pgsA', 2, 'xylose', 'Plaat_2'),
('Strain name:  BEC16920; Gene target: pgsA', 3, 'xylose', 'Plaat_2'),
('Strain name:  BEC22490; Gene target: dapB', 1, 'xylose', 'Plaat_2'),
('Strain name:  BEC22490; Gene target: dapB', 2, 'xylose', 'Plaat_2'),
('Strain name:  BEC22490; Gene target: dapB', 3, 'xylose', 'Plaat_2'),
('Strain name:  BEC15930; Gene target: rnc', 1, 'xylose', 'Plaat_3'),
('Strain name:  BEC15930; Gene target: rnc', 2, 'xylose', 'Plaat_3'),
('Strain name:  BEC15930; Gene target: rnc', 3, 'xylose', 'Plaat_3'),
('Strain name:  BEC29660; Gene target: rpsD', 1, 'xylose', 'Plaat_4'),
('Strain name:  BEC29660; Gene target: rpsD', 2, 'xylose', 'Plaat_4'),
('Strain name:  BEC29660; Gene target: rpsD', 3, 'xylose', 'Plaat_4'),
('Strain name:  BEC32110; Gene target: yumC', 1, 'xylose', 'Plaat_4'),
('Strain name:  BEC32110; Gene target: yumC', 2, 'xylose', 'Plaat_4'),
('Strain name:  BEC32110; Gene target: yumC', 3, 'xylose', 'Plaat_4'),
('Strain name:  leeg; Gene target: nothing', 1, 'xylose', 'Plate_fout'),
('Strain name:  leeg; Gene target: nothing', 2, 'xylose', 'Plate_fout'),
('Strain name:  leeg; Gene target: nothing', 3, 'xylose', 'Plate_fout'),
('Strain name:  leeg; Gene target: nothing', 1, 'xylose', 'Plate_fout'),
('Strain name:  leeg; Gene target: nothing', 2, 'xylose', 'Plate_fout'),
('Strain name:  leeg; Gene target: nothing', 3, 'xylose', 'Plate_fout'),
('Strain name:  leeg; Gene target: nothing', 1, 'xylose', 'Plate_fout'),
('Strain name:  leeg; Gene target: nothing', 2, 'xylose', 'Plate_fout'),
('Strain name:  leeg; Gene target: nothing', 3, 'xylose', 'Plate_fout'),
('Strain name:  leeg; Gene target: nothing', 1, 'xylose', 'Plate_fout'),
('Strain name:  leeg; Gene target: nothing', 2, 'xylose', 'Plate_fout'),
('Strain name:  leeg; Gene target: nothing', 3, 'xylose', 'Plate_fout'),
('Strain name:  CAG74399_83; Gene target: no_sgRNA', 1, 'xylose', 'Plaat_3'),
('Strain name:  CAG74399_83; Gene target: no_sgRNA', 2, 'xylose', 'Plaat_3'),
('Strain name:  CAG74399_83; Gene target: no_sgRNA', 3, 'xylose', 'Plaat_3'),
('Strain name:  CAG74399_39; Gene target: no_sgRNA', 2, 'xylose', 'Plaat_4'),
('Strain name:  CAG74399_39; Gene target: no_sgRNA', 3, 'xylose', 'Plaat_4'),
('Strain name:  CAG74399_57; Gene target: no_sgRNA', 2, 'xylose', 'Plaat_4'),
('Strain name:  CAG74399_57; Gene target: no_sgRNA', 1, 'no xylose', 'Plaat_4'),
('Strain name:  CAG74399_57; Gene target: no_sgRNA', 2, 'no xylose', 'Plaat_4'),
('Strain name:  CAG74399_57; Gene target: no_sgRNA', 3, 'no xylose', 'Plaat_4'),
('Strain name:  CAG74399_89; Gene target: no_sgRNA', 2, 'xylose', 'Plaat_4'),
('Strain name:  CAG74399_89; Gene target: no_sgRNA', 3, 'xylose', 'Plaat_4'),

]




    
samples_to_remove_all = ["Strain name:  SGL999; Gene target: nothing","Strain name:  leeg; Gene target: nothing","Strain name:  xylosevergeten; Gene target: fout"]


# If you want to remove *all* replicates of a sample, you can also list it as just the sample name
#samples_to_remove_all = [
    # 'Strain name:  BAD_SAMPLE; Gene target: something'


# Example of setting diauxic samples (optional)
# samples_diauxic = ['Strain name:  ABC123; Gene target: foo']

# Example of a sample to plot in detail later
sample_to_plot = 'Strain name:  BEC22600; Gene target: aroE'

# Parameters
window_size = 50          # Smoothing window for Savitzky–Golay filter
window_regression = 23   # Window size for best linear regression fit


In [ ]:
#export_path = 'C:/Users/arnou/Documents/thesis/Resultaten/exportfiles/analysisy.xlsx'


#df_melted.to_excel(export_path, index=False, float_format='%.12g')
#print(f"File successfully exported to {export_path}")

df_melted = df_melted[
    ~df_melted.set_index(['Sample', 'rep', 'Xylose', 'MetaData_Plaat']).index.isin(samples_to_remove)
]
df_melted = df_melted[
    ~df_melted.apply(lambda row: (row["Sample"]) in samples_to_remove_all, axis=1)
]

#('Strain name:  CAG74399_12; Gene target: no_sgRNA', 2, 'xylose', 'Plaat_2')

#export_path = 'C:/Users/arnou/Documents/thesis/Resultaten/exportfiles/analysisyy.xlsx'


#df_melted.to_excel(export_path, index=False, float_format='%.12g')
#print(f"File successfully exported to {export_path}")



In [ ]:
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt
import numpy as np

# --- Smoothing using the Savitzky-Golay filter ---

# Group by both Sample and replicate
df_smoothed = df_melted.groupby(['Sample', 'rep','Xylose']).apply(
    apply_savgol_filter,
    window_size=window_size,
    polyorder=1
).reset_index(drop=True)

print(df_smoothed.head())

# --- Check-up plot for smoothing step ---

if sample_to_plot in df_smoothed['Sample'].unique():
    fig, ax = plt.subplots(figsize=(10, 6))

    for (rep,xylose), rep_data in df_smoothed[df_smoothed['Sample'] == sample_to_plot].groupby(['rep', 'Xylose']):
        ax.plot(rep_data['Time'], rep_data['OD'], linestyle='--', marker='o', alpha=0.6, label=f'Raw Rep {rep}{xylose}')
        ax.plot(rep_data['Time'], rep_data['Smoothed_OD'], marker='o', label=f'Smoothed Rep {rep}')

    ax.set_title(f'OD before and after smoothing for {sample_to_plot}')
    ax.set_xlabel('Time (minutes)')
    ax.set_ylabel('OD 600nm')
    ax.legend()
    plt.show()
else:
    print('The specified sample is not present in the smoothed data.')


# --- Blank calculation (per sample–replicate) ---

filtered_df = df_smoothed[(df_smoothed['Time'] >= 1) & (df_smoothed['Time'] <= 15)]

minimum_smoothed_od = (
    filtered_df.groupby(['Sample', 'rep','Xylose'])['Smoothed_OD'].min().rename('Blank_value')
)

df_smoothed = df_smoothed.merge(minimum_smoothed_od, on=['Sample', 'rep','Xylose'], how='left')

print("Blank values for each sample-replicate:")
print(minimum_smoothed_od)
print('-' * 65)

# --- Subtract blank values ---

df_blank_subs = df_smoothed.copy()
df_blank_subs['Smoothed_OD_Blank_Sub'] = df_blank_subs['Smoothed_OD'] - df_blank_subs['Blank_value']

negative_values_count = (df_blank_subs['Smoothed_OD_Blank_Sub'] < 0).sum()
print(f"Number of negative, blank-subtracted OD values: {negative_values_count}")

#all smoothed plots
# --- Plot curves after smoothing and blank subtraction ---

import matplotlib.pyplot as plt

# Get unique samples
samples = df_blank_subs['Sample'].unique()
n_rows = (len(samples) + 2) // 3  # Calculate number of rows for subplots

# Create figure and subplots
fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5 * n_rows))
axes = axes.flatten()

# Plot each sample
for i, sample in enumerate(samples):
    sample_data = df_blank_subs[df_blank_subs['Sample'] == sample]

    # Plot each replicate
    for (rep,xylose), rep_data in sample_data.groupby(['rep', 'Xylose']):
        axes[i].plot(
            rep_data['Time'],
            rep_data['Smoothed_OD_Blank_Sub'],
            label=f'Rep {rep}{xylose}',
            alpha=0.8
        )

    axes[i].set_title(f'{sample}')
    axes[i].set_xlabel('Time (hours)')
    axes[i].set_ylabel('OD ')
    axes[i].legend()
    #axes[i].set_xlim([0, 15])
    #axes[i].set_ylim([0, y_max])

# Remove unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

print('List of sample names (smoothed + blank-subtracted):')
print(list(samples))


# --- Check-up plot for blank subtraction ---


if sample_to_plot in df_blank_subs['Sample'].unique():
    fig, ax = plt.subplots(figsize=(10, 6))
    for (rep,xylose), rep_data in df_blank_subs[df_blank_subs['Sample'] == sample_to_plot].groupby(['rep', 'Xylose']):
        ax.plot(rep_data['Time'], rep_data['Smoothed_OD_Blank_Sub'], marker='o', label=f'Rep {rep}')
    ax.set_xlabel('Time (minutes)')
    ax.set_ylabel('Blank-subtracted OD')
    ax.set_title(f'Blank-subtracted OD for {sample_to_plot}')
    ax.legend()
    ax.grid(True)
    plt.show()
else:
    print('The specified sample is not present in this acquisition.')


In [ ]:
num_strains = df_smoothed['Sample'].nunique()
print(f"Number of different strains (unique Sample values): {num_strains}")


In [ ]:
num_strains = df_smoothed['Sample'].nunique()
print(f"Number of different strains (unique Sample values): {num_strains}")

# --- count no_sgRNA vs others ---
sample_list = df_smoothed['Sample'].unique()

num_no_sgRNA = sum("no_sgRNA".lower() in s.lower() for s in sample_list)
num_with_sgRNA = num_strains - num_no_sgRNA

print(f"Strains with no_sgRNA: {num_no_sgRNA}")
print(f"Strains with sgRNA (mutants): {num_with_sgRNA}")


In [ ]:
#met saving fig

In [ ]:
#excel maken die de relvante groeinnfo bevat om deze plotjes te maken

In [ ]:
import os
import pandas as pd

# --- SETUP DIRECTORY ---
base_path = r"E:\Thesis3april\GrowthResults"
excel_folder_path = os.path.join(base_path, "Excel_Export")

if not os.path.exists(excel_folder_path):
    os.makedirs(excel_folder_path)
    print(f"Created directory: {excel_folder_path}")

# --- EXPORT TO EXCEL ---
# Define the output path
excel_filename = os.path.join(excel_folder_path, "growth_data_blank_subtracted.xlsx")

# Export df_blank_subs to Excel
# We use index=False to keep the file clean
try:
    df_blank_subs.to_excel(excel_filename, index=False, engine='openpyxl')
    print(f"Successfully created Excel file at: {excel_filename}")
except Exception as e:
    print(f"Error creating Excel file: {e}")

In [ ]:
#dfblankc sub plotten van voorbeelden zonder zexcel

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

def plot_gene_targets_mean_std(df, gene_targets, time_limit=(0, 14), figsize=(12,8)):
    """
    Plots mean growth curves with SD shading using subtracted blank values.
    Features: 
    - Square plot area
    - OD600 with non-italic 'OD' and subscript '600'
    - Custom tick font size for axis numbers
    - Legend in top-left corner (no box)
    """
    
    # --- SETUP DIRECTORIES ---
    base_path = r"E:\Thesis3april\GrowthResults"
    subfolder_path = os.path.join(base_path, "growth curve blanksubs_differentclasses_samegrowth")
    
    if not os.path.exists(subfolder_path):
        os.makedirs(subfolder_path)

    if not isinstance(gene_targets, (list, tuple)):
        gene_targets = [gene_targets]
    
    # Helper function to ensure Xylose_flag column exists
    if 'Xylose_flag' not in df.columns:
        def _is_xylose_val(v):
            try:
                s = str(v).strip().lower()
                return s in ("true", "yes", "1", "xylose", "y")
            except Exception:
                return False
        df = df.copy()
        df['Xylose_flag'] = df['Xylose'].apply(_is_xylose_val)

    # --- FONT SIZE DEFINITIONS ---
    TITLE_FONT = 20
    LABEL_FONT = 20
    TICK_FONT = 20    # Adjust this to change the size of numbers on the axes
    LEGEND_FONT = 20

    for target in gene_targets:
        mask = df['Sample'].astype(str).str.contains(target)
        if mask.sum() == 0:
            print(f'No samples match gene target: {target}')
            continue
            
        sel = df[mask].copy()

        # Group data by Time and Xylose_flag
        grouped_data = sel.groupby(['Xylose_flag', 'Time'])['Smoothed_OD_Blank_Sub'].agg(['mean', 'std']).reset_index()

        # Separate data
        data_xylose = grouped_data[grouped_data['Xylose_flag'] == True]
        data_no_xylose = grouped_data[grouped_data['Xylose_flag'] == False]
        
        # Create figure
        fig, ax = plt.subplots(figsize=figsize)
        
        # --- Plot With Xylose (Blue) ---
        if not data_xylose.empty:
            time_xylose = data_xylose['Time']
            mean_xylose = data_xylose['mean']
            std_xylose = data_xylose['std'].fillna(0)
            
            ax.plot(time_xylose, mean_xylose, color='blue', linewidth=2.5)
            ax.fill_between(time_xylose, mean_xylose - std_xylose, mean_xylose + std_xylose, 
                            color='blue', alpha=0.2, edgecolor='none')

        # --- Plot Without Xylose (Green) ---
        if not data_no_xylose.empty:
            time_no_xylose = data_no_xylose['Time']
            mean_no_xylose = data_no_xylose['mean']
            std_no_xylose = data_no_xylose['std'].fillna(0)
            
            ax.plot(time_no_xylose, mean_no_xylose, color='green', linewidth=2.5)
            ax.fill_between(time_no_xylose, mean_no_xylose - std_no_xylose, mean_no_xylose + std_no_xylose, 
                            color='green', alpha=0.2, edgecolor='none')

        # --- STYLING ---
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_aspect('auto') 

        ax.set_xlabel('Time (hours)', fontsize=LABEL_FONT)
        
        # \mathrm{OD} ensures it is not italicized in the LaTeX environment
        ax.set_ylabel(r'$\mathrm{OD}_{600}$', fontsize=LABEL_FONT)
        
        ax.set_title(f'{target}', fontsize=TITLE_FONT, pad=15)
        ax.set_xlim(time_limit)
        
        # Apply the chosen font size to the axis numbers
        ax.tick_params(axis='both', which='major', labelsize=TICK_FONT)

        # --- LEGEND: INSIDE TOP LEFT ---
        legend_elements = [
            Line2D([0], [0], color='blue', lw=3, label='Mean xylose'),
            Patch(facecolor='blue', alpha=0.3, label='SD xylose'),
            Line2D([0], [0], color='green', lw=3, label='Mean no xylose'),
            Patch(facecolor='green', alpha=0.3, label='SD no xylose')
        ]
        
        ax.legend(handles=legend_elements, loc='upper left', 
                  fontsize=LEGEND_FONT, frameon=False)
        
        # --- SAVE ---
        safe_name = target.replace(" ", "_")
        file_base = os.path.join(subfolder_path, f"{safe_name}_growth_curve")
        
        plt.savefig(f"{file_base}.svg", format="svg", bbox_inches='tight')
        plt.savefig(f"{file_base}.png", format="png", dpi=500, bbox_inches='tight')
        
        plt.show()

# Run using the subtracted dataframe
targets = ['no_sgRNA', 'rpsNA', 'rpsK', 'rpsI', 'rpsJ', 'ftsZ', 'ftsL','ftsW', 'divIB', 'divIC']
plot_gene_targets_mean_std(df_blank_subs, targets)

In [ ]:
#excel gebruiken om te plotten (duurt wel langer om excel te laden)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# --- 1) LOAD THE DATA FROM EXCEL ---
base_path = r"E:\Thesis3april\GrowthResults"
excel_path = os.path.join(base_path, "Excel_Export", "growth_data_blank_subtracted.xlsx")           #doe #weg als je excel wilt gebruiken

# Load the dataframe
df_from_excel = pd.read_excel(excel_path)

def plot_gene_targets_mean_std(df, gene_targets, time_limit=(0, 14), figsize=(12,8)):
    """
    Plots mean growth curves from the loaded Excel data.
    """
    
    # --- SETUP PLOT DIRECTORY ---
    subfolder_path = os.path.join(base_path, "voorbeeldplots","linksboven")
    if not os.path.exists(subfolder_path):
        os.makedirs(subfolder_path)

    if not isinstance(gene_targets, (list, tuple)):
        gene_targets = [gene_targets]
    
    # Ensure Xylose_flag exists (in case it wasn't in the Excel)
    if 'Xylose_flag' not in df.columns:
        def _is_xylose_val(v):
            try:
                s = str(v).strip().lower()
                return s in ("true", "yes", "1", "xylose", "y")
            except Exception:
                return False
        df = df.copy()
        df['Xylose_flag'] = df['Xylose'].apply(_is_xylose_val)

    # --- FONT SIZE DEFINITIONS ---
    TITLE_FONT = 20
    LABEL_FONT = 20
    TICK_FONT = 18    
    LEGEND_FONT = 18

    for target in gene_targets:
        mask = df['Sample'].astype(str).str.contains(target)
        if mask.sum() == 0:
            print(f'No samples match gene target: {target}')
            continue
            
        sel = df[mask].copy()

        # Group data
        grouped_data = sel.groupby(['Xylose_flag', 'Time'])['Smoothed_OD_Blank_Sub'].agg(['mean', 'std']).reset_index()

        # Separate data
        data_xylose = grouped_data[grouped_data['Xylose_flag'] == True]
        data_no_xylose = grouped_data[grouped_data['Xylose_flag'] == False]
        
        # Create figure (Square aspect ratio)
        fig, ax = plt.subplots(figsize=figsize)
        
        # --- Plot With Xylose (Blue) ---
        if not data_xylose.empty:
            ax.plot(data_xylose['Time'], data_xylose['mean'], color='blue', linewidth=2.5)
            ax.fill_between(data_xylose['Time'], 
                            data_xylose['mean'] - data_xylose['std'].fillna(0), 
                            data_xylose['mean'] + data_xylose['std'].fillna(0), 
                            color='blue', alpha=0.2, edgecolor='none')

        # --- Plot Without Xylose (Green) ---
        if not data_no_xylose.empty:
            ax.plot(data_no_xylose['Time'], data_no_xylose['mean'], color='green', linewidth=2.5)
            ax.fill_between(data_no_xylose['Time'], 
                            data_no_xylose['mean'] - data_no_xylose['std'].fillna(0), 
                            data_no_xylose['mean'] + data_no_xylose['std'].fillna(0), 
                            color='green', alpha=0.2, edgecolor='none')

        # --- STYLING ---
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_aspect('auto') 

        ax.set_xlabel('Time (hours)', fontsize=LABEL_FONT)
        # Non-italic OD with subscript 600
        ax.set_ylabel(r'$\mathrm{OD}_{600}$', fontsize=LABEL_FONT)
        ax.set_title(f'{target}', fontsize=TITLE_FONT, pad=15)
        ax.set_xlim(time_limit)
        ax.tick_params(axis='both', which='major', labelsize=TICK_FONT)

        # --- LEGEND: TOP LEFT INSIDE ---
        legend_elements = [
            Line2D([0], [0], color='blue', lw=3, label='Mean xylose'),
            Patch(facecolor='blue', alpha=0.3, label='SD xylose'),
            Line2D([0], [0], color='green', lw=3, label='Mean no xylose'),
            Patch(facecolor='green', alpha=0.3, label='SD no xylose')
        ]
        ax.legend(handles=legend_elements, loc='upper left', fontsize=LEGEND_FONT, frameon=False)
        
        # --- SAVE ---
        safe_name = target.replace(" ", "_")
        file_base = os.path.join(subfolder_path, f"{safe_name}_growth_curve")
        plt.savefig(f"{file_base}.svg", format="svg", bbox_inches='tight')
        plt.savefig(f"{file_base}.png", format="png", dpi=500, bbox_inches='tight')
        plt.show()

# --- RUN THE FUNCTION ---
#targets = ['no_sgRNA', 'rpsNA', 'rpsK', 'rpsI', 'rpsJ', 'ftsZ', 'ftsL','ftsW', 'divIB', 'divIC'] #same growth curves, differnt class
#targets = ['no_sgRNA', 'xkdB','yjzJ']  

#targets = ['no_sgRNA', 'dnaE', 'walR', "pbpB", 'dapG', 'dapH', 'parE', 'tagD', 'tagB', 'tagF']
#targets =['no_sgRNA', 'mnaA', 'leuS','rpsD', 'rpsQ' , 'dapB', 'acpP','pgm','fabF'  ]  onder
targets = ['no_sgRNA', 'tagB','tagF','mnaA', 'rpsNA','sufS','pgm', 'rpsK', 'rpsI', 'rpsJ', 'ftsZ', 'ftsL','ftsW', 'divIB', 'divIC']
plot_gene_targets_mean_std(df_from_excel, targets)

In [ ]:
#na excel loading kan het korter
targets = ['sufS','pgm', 'no_sgRNA', 'dnaE', 'folC','folE', 'holA','rghRA']
plot_gene_targets_mean_std(df_from_excel, targets)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
from matplotlib.lines import Line2D

# ==========================================
# --- CONFIGURATION & FONT PARAMETERS ---
# ==========================================
BASE_PATH = r"E:\Thesis3april\GrowthResults"
EXCEL_PATH = os.path.join(BASE_PATH, "Excel_Export", "growth_data_blank_subtracted.xlsx")

# Font Sizes
TITLE_FONT = 22
LABEL_FONT = 23
TICK_FONT = 22      # Font size for axis numbers
LEGEND_FONT = 22

# Line and Axis Thickness
AXIS_LINE_WIDTH = 2.0  # <--- ADJUST THIS for X and Y axis thickness
MUTANT_LINE_WIDTH = 2
CONTROL_LINE_WIDTH = 2.5

# Plotting Parameters
TIME_LIMIT = (0, 14)
OD_LIMIT = (0, 1.35)
FIG_SIZE = (12, 8)   # Ensures a square-based figure
DATA_COL = 'Smoothed_OD_Blank_Sub'

# ==========================================
# --- DATA LOADING & PREPARATION ---
# ==========================================

if os.path.exists(EXCEL_PATH):
    df_plot = pd.read_excel(EXCEL_PATH)
else:
    print(f"Error: Excel file not found at {EXCEL_PATH}")
    # Fallback to df_blank_subs if already in memory
    try:
        df_plot = df_blank_subs.copy()
    except NameError:
        print("df_blank_subs not defined in environment.")

def _is_xylose_val(v):
    try:
        s = str(v).strip().lower()
        return s in ("true", "yes", "1", "xylose", "y")
    except Exception:
        return False

if 'Xylose_flag' not in df_plot.columns:
    df_plot['Xylose_flag'] = df_plot['Xylose'].apply(_is_xylose_val)

_no_sg_patterns = re.compile(r'no[_\-\s]?s?g?r?n?a?', flags=re.IGNORECASE)

def is_no_sgRNA(sample):
    return bool(_no_sg_patterns.search(str(sample)))

# ==========================================
# --- PLOTTING FUNCTION ---
# ==========================================

def plot_all_with_and_without_xylose(df):
    subfolder_path = os.path.join(BASE_PATH, "combined_growth_curves_blank_sub")
    if not os.path.exists(subfolder_path):
        os.makedirs(subfolder_path)

    groups = {
        'With Xylose': df[df['Xylose_flag'] == True],
        'No Xylose': df[df['Xylose_flag'] == False]
    }

    def color_by_substring(sample):
        s = str(sample).lower()
        if 'sgl999' in s:
            return 'orange'
        return 'gray'

    for title, subdf in groups.items():
        if subdf.empty:
            continue

        fig, ax = plt.subplots(figsize=FIG_SIZE)

        # --- 1) PLOT MUTANTS ---
        other_samples = [s for s in subdf['Sample'].unique() if not is_no_sgRNA(s)]
        for sample in other_samples:
            sample_data = subdf[subdf['Sample'] == sample]
            color = color_by_substring(sample)
            stats = sample_data.groupby('Time')[DATA_COL].mean().reset_index()
            ax.plot(stats['Time'], stats[DATA_COL], color=color, 
                    linewidth=MUTANT_LINE_WIDTH, alpha=0.5, zorder=1)

        # --- 2) PLOT no_sgRNA (Control) ---
        nosg_df = subdf[subdf['Sample'].apply(is_no_sgRNA)]
        if not nosg_df.empty:
            nosg_stats = nosg_df.groupby('Time')[DATA_COL].agg(['mean', 'std']).reset_index()
            ax.fill_between(nosg_stats['Time'], nosg_stats['mean'] - nosg_stats['std'].fillna(0), 
                            nosg_stats['mean'] + nosg_stats['std'].fillna(0),
                            color='red', alpha=0.2, edgecolor='none', zorder=3)
            ax.plot(nosg_stats['Time'], nosg_stats['mean'], color='red', 
                    linewidth=CONTROL_LINE_WIDTH, zorder=4)

        # --- STYLING ---
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        # Apply Axis Line Thickness
        ax.spines['left'].set_linewidth(AXIS_LINE_WIDTH)
        ax.spines['bottom'].set_linewidth(AXIS_LINE_WIDTH)
        
        ax.set_aspect('auto') 
        ax.set_xlabel('Time (hours)', fontsize=LABEL_FONT)
        # Non-italic OD with subscript 600
        ax.set_ylabel(r'$\mathrm{OD}_{600}$', fontsize=LABEL_FONT)
        ax.set_title(f'{title}', fontsize=TITLE_FONT, pad=15)
        ax.set_xlim(TIME_LIMIT)
        ax.set_ylim(OD_LIMIT)
        
        # Apply Tick Font and Tick thickness
        ax.tick_params(axis='both', which='major', labelsize=TICK_FONT, width=AXIS_LINE_WIDTH)

        # --- LEGEND ---
        legend_elements = [
            Line2D([0], [0], color='red', lw=CONTROL_LINE_WIDTH, label='no-sgRNA'),
            Line2D([0], [0], color='gray', lw=MUTANT_LINE_WIDTH, label='mutants')
        ]
        ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1, 1), 
                  fontsize=LEGEND_FONT, frameon=False)

        # --- SAVE ---
        file_name = title.replace(' ', '_').lower()
        file_base = os.path.join(subfolder_path, f"{file_name}_combined_growth")
        plt.savefig(f"{file_base}.svg", format="svg", bbox_inches='tight')
        plt.savefig(f"{file_base}.png", format="png", dpi=500, bbox_inches='tight')
        plt.show()

# Run the function
plot_all_with_and_without_xylose(df_plot)

In [ ]:
#plotting the diffent pathways growth curves

In [ ]:
#growht curves sperate grey

In [ ]:
#hoort bij de curves hieronder

# ==========================================
# --- CONFIGURATION & PATHS ---
# ==========================================
BASE_PATH = r"E:\Thesis3april\GrowthResults"
EXCEL_PATH = os.path.join(BASE_PATH, "Excel_Export", "growth_data_blank_subtracted.xlsx")
ANNOTATION_PATH = r"E:\Thesis3april\overlay\Pathway_annotation.xlsx"
OUTPUT_DIR = os.path.join(BASE_PATH, "7april", "Pathway_Growth_Curves_seperatcorrect")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)


In [ ]:
#individual pathways iwth indiual names

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
from matplotlib.lines import Line2D

# ==========================================
# --- CONFIGURATION & PATHS ---
# ==========================================
BASE_PATH = r"E:\Thesis3april\GrowthResults"
EXCEL_PATH = os.path.join(BASE_PATH, "Excel_Export", "growth_data_blank_subtracted.xlsx")
ANNOTATION_PATH = r"E:\Thesis3april\overlay\Pathway_annotation.xlsx"
OUTPUT_DIR = os.path.join(BASE_PATH, "7april", "Pathway_Growth_Curves_grey")
GENE_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "Pathway_with_Gene_Names")

for d in [OUTPUT_DIR, GENE_OUTPUT_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

# Font & Style Parameters
TITLE_FONT, LABEL_FONT, TICK_FONT, LEGEND_FONT = 22, 23, 22, 0
AXIS_LINE_WIDTH = 2.0
MUTANT_LINE_WIDTH = 2
CONTROL_LINE_WIDTH = 2.5
TIME_LIMIT = (0, 14)
OD_LIMIT = (0, 1.35)
FIG_SIZE = (15, 8) # Extra width for gene name legends
DATA_COL = 'Smoothed_OD_Blank_Sub'

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c", 
    "cell shape": "#d62728", 
    "biosynthesis of fatty acids": "#9467bd", 
    "DNA replication": "#8c564b", 
    "DNA condensation/ segregation": "#e377c2", 
    "biosynthesis of isoprenoids": "#7f7f7f", 
    "cell division": "#bcbd22", 
    "ribosomal proteins": "#17becf", 
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78", 
    "biosynthesis of menaquinone": "#98df8a"
}

# ==========================================
# --- DATA PREPARATION ---
# ==========================================
df_plot = pd.read_excel(EXCEL_PATH)
df_ann = pd.read_excel(ANNOTATION_PATH)

def get_pathway(row):
    for col in ['SubtiWiki Annotation 4', 'SubtiWiki Annotation 3']:
        val = str(row.get(col, '')).strip()
        if val not in ['nan', 'NA', 'None', '']: return val
    return "Unknown/Other"

df_ann['Pathway'] = df_ann.apply(get_pathway, axis=1)
pathway_map = dict(zip(df_ann['Treatment'].astype(str).str.strip(), df_ann['Pathway']))

_no_sg_patterns = re.compile(r'no[_\-\s]?s?g?r?n?a?', flags=re.IGNORECASE)
def is_no_sgRNA(sample): return bool(_no_sg_patterns.search(str(sample)))

def assign_pathway(sample):
    if is_no_sgRNA(sample): return "no-sgRNA"
    sample_str = str(sample)
    for target, pw in pathway_map.items():
        if target in sample_str:
            return pw
    return "Unknown/Other"

df_plot['Pathway'] = df_plot['Sample'].apply(assign_pathway)
df_xyl = df_plot[df_plot['Xylose'].astype(str).str.lower().isin(["true", "yes", "1", "xylose", "y"])]

# ==========================================
# --- HELPER FUNCTIONS ---
# ==========================================

def apply_style(ax, title):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(AXIS_LINE_WIDTH)
    ax.spines['bottom'].set_linewidth(AXIS_LINE_WIDTH)
    ax.set_xlabel('Time (hours)', fontsize=LABEL_FONT)
    ax.set_ylabel(r'$\mathrm{OD}_{600}$', fontsize=LABEL_FONT)
    ax.set_title(title, fontsize=TITLE_FONT, pad=15)
    ax.set_xlim(TIME_LIMIT)
    ax.set_ylim(OD_LIMIT)
    ax.tick_params(axis='both', labelsize=TICK_FONT, width=AXIS_LINE_WIDTH)

def sanitize_filename(name):
    return name.replace(' ', '_').replace('/', '_').replace('\\', '_').lower()

def save_and_show(fig, folder, base_name):
    fig.subplots_adjust(right=0.7) # Lock plotting region size
    clean_name = sanitize_filename(base_name)
    path = os.path.join(folder, clean_name)
    fig.savefig(f"{path}.png", dpi=500, bbox_inches='tight')
    fig.savefig(f"{path}.svg", format="svg", bbox_inches='tight')
    plt.show()

# ==========================================
# --- EXECUTION ---
# ==========================================

unique_pathways = [p for p in df_xyl['Pathway'].unique() if p not in ["no-sgRNA", "Unknown/Other"]]

# 1. Standard Individual Pathway Plots (Generic Mutants Legend)
for pw in unique_pathways:
    fig, ax = plt.subplots(figsize=FIG_SIZE)
    apply_style(ax, f"Pathway: {pw}")
    
    nosg_df = df_xyl[df_xyl['Pathway'] == "no-sgRNA"]
    if not nosg_df.empty:
        stats = nosg_df.groupby('Time')[DATA_COL].agg(['mean', 'std']).reset_index()
        ax.fill_between(stats['Time'], stats['mean'] - stats['std'].fillna(0), 
                        stats['mean'] + stats['std'].fillna(0), color='red', alpha=0.1, edgecolor='none')
        ax.plot(stats['Time'], stats['mean'], color='red', lw=CONTROL_LINE_WIDTH, label='no-sgRNA', zorder=5)
    
    pw_df = df_xyl[df_xyl['Pathway'] == pw]
    for sample in pw_df['Sample'].unique():
        s_data = pw_df[pw_df['Sample'] == sample].groupby('Time')[DATA_COL].mean()
        ax.plot(s_data.index, s_data.values, color="#47474d", lw=MUTANT_LINE_WIDTH, alpha=0.4)

    legend_elements = [
        Line2D([0], [0], color='red', lw=CONTROL_LINE_WIDTH, label='no-sgRNA'),
        Line2D([0], [0], color='gray', lw=MUTANT_LINE_WIDTH, label='Mutants')
    ]
    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False, fontsize=LEGEND_FONT)
    save_and_show(fig, OUTPUT_DIR, f"Pathway_{pw}")

# 2. Detailed Individual Pathway Plots (Showing Gene Names in Legend)
for pw in unique_pathways:
    fig, ax = plt.subplots(figsize=FIG_SIZE)
    apply_style(ax, f"Pathway Details: {pw}")
    
    # Control
    nosg_stats = df_xyl[df_xyl['Pathway'] == "no-sgRNA"].groupby('Time')[DATA_COL].mean()
    ax.plot(nosg_stats.index, nosg_stats.values, color='red', lw=CONTROL_LINE_WIDTH, label='no-sgRNA', zorder=10)
    
    # Mutants (labeled by sample/gene)
    pw_df = df_xyl[df_xyl['Pathway'] == pw]
    for sample in pw_df['Sample'].unique():
        s_data = pw_df[pw_df['Sample'] == sample].groupby('Time')[DATA_COL].mean()
        ax.plot(s_data.index, s_data.values, lw=MUTANT_LINE_WIDTH, alpha=0.7, label=str(sample))

    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False, fontsize=10) # Smaller font for many genes
    save_and_show(fig, GENE_OUTPUT_DIR, f"Detailed_{pw}")

# 3. Summary Plot (All Curves, Color-coded)
fig, ax = plt.subplots(figsize=FIG_SIZE)
apply_style(ax, "Summary: All Manual Pathway Curves")

nosg_stats = df_xyl[df_xyl['Pathway'] == "no-sgRNA"].groupby('Time')[DATA_COL].mean()
ax.plot(nosg_stats.index, nosg_stats.values, color='black', lw=CONTROL_LINE_WIDTH, label='no-sgRNA', zorder=15)

plotted_labels = set()
for pw, color in MANUAL_COLORS.items():
    pw_df = df_xyl[df_xyl['Pathway'] == pw]
    if pw_df.empty: continue
    
    for i, sample in enumerate(pw_df['Sample'].unique()):
        lbl = pw if pw not in plotted_labels else "_"
        s_data = pw_df[pw_df['Sample'] == sample].groupby('Time')[DATA_COL].mean()
        ax.plot(s_data.index, s_data.values, color=color, lw=MUTANT_LINE_WIDTH, alpha=0.5, label=lbl)
        plotted_labels.add(pw)

ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False, fontsize=11)
save_and_show(fig, OUTPUT_DIR, "Summary_Manual_All_Curves")

In [ ]:
#betere laytou indiviudal pathways individual names colored (previous was grey)

In [ ]:
#foutje bij vorige

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
from matplotlib.lines import Line2D

# ==========================================
# --- CONFIGURATION & PATHS ---
# ==========================================
BASE_PATH = r"E:\Thesis3april\GrowthResults"
EXCEL_PATH = os.path.join(BASE_PATH, "Excel_Export", "growth_data_blank_subtracted.xlsx")
ANNOTATION_PATH = r"E:\Thesis3april\overlay\Pathway_annotation.xlsx"
OUTPUT_DIR = os.path.join(BASE_PATH, "7april", "Pathway_Growth_Curves_2depoging")
GENE_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "Pathway_with_Gene_Names")

for d in [OUTPUT_DIR, GENE_OUTPUT_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

# --- THICKNESS & FONT SETTINGS ---
TITLE_FONT, LABEL_FONT, TICK_FONT = 22, 23, 22
LEGEND_FONT_DETAILED = 18  # As requested
AXIS_LINE_WIDTH = 2.0
MUTANT_LINE_WIDTH = 1.8    
CONTROL_LINE_WIDTH = 2.5
LEGEND_LINE_WIDTH = 5.0    

TIME_LIMIT = (0, 14)
OD_LIMIT = (0, 1.35)
FIG_SIZE = (15, 8) 
DATA_COL = 'Smoothed_OD_Blank_Sub'

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c", 
    "cell shape": "#d62728", 
    "biosynthesis of fatty acids": "#9467bd", 
    "DNA replication": "#8c564b", 
    "DNA condensation/ segregation": "#e377c2", 
    "biosynthesis of isoprenoids": "#7f7f7f", 
    "cell division": "#bcbd22", 
    "ribosomal proteins": "#17becf", 
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78", 
    "biosynthesis of menaquinone": "#98df8a"
}

# ==========================================
# --- DATA PREPARATION ---
# ==========================================
df_plot = pd.read_excel(EXCEL_PATH)
df_ann = pd.read_excel(ANNOTATION_PATH)

def get_pathway(row):
    for col in ['SubtiWiki Annotation 4', 'SubtiWiki Annotation 3']:
        val = str(row.get(col, '')).strip()
        if val not in ['nan', 'NA', 'None', '']: return val
    return "Unknown/Other"

df_ann['Pathway'] = df_ann.apply(get_pathway, axis=1)
pathway_map = dict(zip(df_ann['Treatment'].astype(str).str.strip(), df_ann['Pathway']))

_no_sg_patterns = re.compile(r'no[_\-\s]?s?g?r?n?a?', flags=re.IGNORECASE)
def is_no_sgRNA(sample): return bool(_no_sg_patterns.search(str(sample)))

def assign_pathway(sample):
    if is_no_sgRNA(sample): return "no-sgRNA"
    sample_str = str(sample)
    for target, pw in pathway_map.items():
        if target in sample_str:
            return pw
    return "Unknown/Other"

df_plot['Pathway'] = df_plot['Sample'].apply(assign_pathway)
df_xyl = df_plot[df_plot['Xylose'].astype(str).str.lower().isin(["true", "yes", "1", "xylose", "y"])]

# ==========================================
# --- HELPER FUNCTIONS ---
# ==========================================

def apply_style(ax, title):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(AXIS_LINE_WIDTH)
    ax.spines['bottom'].set_linewidth(AXIS_LINE_WIDTH)
    ax.set_xlabel('Time (hours)', fontsize=LABEL_FONT)
    ax.set_ylabel(r'$\mathrm{OD}_{600}$', fontsize=LABEL_FONT)
    ax.set_title(title, fontsize=TITLE_FONT, pad=15)
    ax.set_xlim(TIME_LIMIT)
    ax.set_ylim(OD_LIMIT)
    ax.tick_params(axis='both', labelsize=TICK_FONT, width=AXIS_LINE_WIDTH)

def sanitize_filename(name):
    return name.replace(' ', '_').replace('/', '_').replace('\\', '_').lower()

def save_and_show(fig, folder, base_name, thicken_legend=False):
    fig.subplots_adjust(right=0.7) 
    if thicken_legend:
        leg = plt.gca().get_legend()
        if leg:
            for line in leg.get_lines():
                line.set_linewidth(LEGEND_LINE_WIDTH)
    clean_name = sanitize_filename(base_name)
    path = os.path.join(folder, clean_name)
    fig.savefig(f"{path}.png", dpi=500, bbox_inches='tight')
    fig.savefig(f"{path}.svg", format="svg", bbox_inches='tight')
    plt.show()

# ==========================================
# --- EXECUTION ---
# ==========================================

unique_pathways = [p for p in df_xyl['Pathway'].unique() if p not in ["no-sgRNA", "Unknown/Other"]]

# 1. Detailed Individual Pathway Plots (Cleaned Gene Names)
for pw in unique_pathways:
    fig, ax = plt.subplots(figsize=FIG_SIZE)
    apply_style(ax, f"Pathway Details: {pw}")
    
    # --- Control Plotting (with SD) ---
    nosg_df = df_xyl[df_xyl['Pathway'] == "no-sgRNA"]
    if not nosg_df.empty:
        stats = nosg_df.groupby('Time')[DATA_COL].agg(['mean', 'std']).reset_index()
        ax.fill_between(stats['Time'], stats['mean'] - stats['std'].fillna(0), 
                        stats['mean'] + stats['std'].fillna(0), color='red', alpha=0.1, edgecolor='none')
        ax.plot(stats['Time'], stats['mean'], color='red', lw=CONTROL_LINE_WIDTH, label='no-sgRNA', zorder=10)
    
    # --- Mutants Plotting (Clean names) ---
    pw_df = df_xyl[df_xyl['Pathway'] == pw]
    for sample in pw_df['Sample'].unique():
        # Logic to extract only the gene target from the string
        match = re.search(r"Gene target:\s*(\S+)", str(sample))
        label_name = match.group(1) if match else str(sample)
        
        s_data = pw_df[pw_df['Sample'] == sample].groupby('Time')[DATA_COL].mean()
        ax.plot(s_data.index, s_data.values, lw=MUTANT_LINE_WIDTH, alpha=0.7, label=label_name)

    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False, fontsize=LEGEND_FONT_DETAILED)
    save_and_show(fig, GENE_OUTPUT_DIR, f"Detailed_{pw}", thicken_legend=True)

# 2. Summary Plot (All Curves, Color-coded)
fig, ax = plt.subplots(figsize=FIG_SIZE)
apply_style(ax, "Summary: All Manual Pathway Curves")

nosg_df = df_xyl[df_xyl['Pathway'] == "no-sgRNA"]
if not nosg_df.empty:
    stats = nosg_df.groupby('Time')[DATA_COL].agg(['mean', 'std']).reset_index()
    ax.fill_between(stats['Time'], stats['mean'] - stats['std'].fillna(0), 
                    stats['mean'] + stats['std'].fillna(0), color='black', alpha=0.1, edgecolor='none')
    ax.plot(stats['Time'], stats['mean'], color='black', lw=CONTROL_LINE_WIDTH, label='no-sgRNA', zorder=15)

plotted_labels = set()
for pw, color in MANUAL_COLORS.items():
    pw_df = df_xyl[df_xyl['Pathway'] == pw]
    if pw_df.empty: continue
    
    for sample in pw_df['Sample'].unique():
        lbl = pw if pw not in plotted_labels else "_"
        s_data = pw_df[pw_df['Sample'] == sample].groupby('Time')[DATA_COL].mean()
        ax.plot(s_data.index, s_data.values, color=color, lw=MUTANT_LINE_WIDTH, alpha=0.5, label=lbl)
        plotted_labels.add(pw)

ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False, fontsize=18)
save_and_show(fig, OUTPUT_DIR, "Summary_Manual_All_Curves", thicken_legend=True)

In [ ]:
#this code only to change the all together plot , 14 april isporeonoids weg

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
from matplotlib.lines import Line2D

# ==========================================
# --- CONFIGURATION & PATHS ---
# ==========================================
BASE_PATH = r"E:\Thesis3april\GrowthResults"
EXCEL_PATH = os.path.join(BASE_PATH, "Excel_Export", "growth_data_blank_subtracted.xlsx")
ANNOTATION_PATH = r"E:\Thesis3april\overlay\Pathway_annotation.xlsx"
OUTPUT_DIR = os.path.join(BASE_PATH, "14april", "Pathway_Growth_Curves_2depoging")
GENE_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "Pathway_with_Gene_Names")

for d in [OUTPUT_DIR, GENE_OUTPUT_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

# --- THICKNESS & FONT SETTINGS ---
TITLE_FONT, LABEL_FONT, TICK_FONT = 22, 23, 22
LEGEND_FONT_DETAILED = 18  # As requested
AXIS_LINE_WIDTH = 2.0
MUTANT_LINE_WIDTH = 1.8    
CONTROL_LINE_WIDTH = 2.5
LEGEND_LINE_WIDTH = 5.0    

TIME_LIMIT = (0, 14)
OD_LIMIT = (0, 1.35)
FIG_SIZE = (15, 8) 
DATA_COL = 'Smoothed_OD_Blank_Sub'

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c", 
    "cell shape": "#d62728", 
    "biosynthesis of fatty acids": "#9467bd", 
    "DNA replication": "#8c564b", 
    "DNA condensation/ segregation": "#e377c2", 
   
    "cell division": "#bcbd22", 
    "ribosomal proteins": "#17becf", 
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78", 
    "biosynthesis of menaquinone": "#98df8a"
}

# ==========================================
# --- DATA PREPARATION ---
# ==========================================
df_plot = pd.read_excel(EXCEL_PATH)
df_ann = pd.read_excel(ANNOTATION_PATH)

def get_pathway(row):
    for col in ['SubtiWiki Annotation 4', 'SubtiWiki Annotation 3']:
        val = str(row.get(col, '')).strip()
        if val not in ['nan', 'NA', 'None', '']: return val
    return "Unknown/Other"

df_ann['Pathway'] = df_ann.apply(get_pathway, axis=1)
pathway_map = dict(zip(df_ann['Treatment'].astype(str).str.strip(), df_ann['Pathway']))

_no_sg_patterns = re.compile(r'no[_\-\s]?s?g?r?n?a?', flags=re.IGNORECASE)
def is_no_sgRNA(sample): return bool(_no_sg_patterns.search(str(sample)))

def assign_pathway(sample):
    if is_no_sgRNA(sample): return "no-sgRNA"
    sample_str = str(sample)
    for target, pw in pathway_map.items():
        if target in sample_str:
            return pw
    return "Unknown/Other"

df_plot['Pathway'] = df_plot['Sample'].apply(assign_pathway)
df_xyl = df_plot[df_plot['Xylose'].astype(str).str.lower().isin(["true", "yes", "1", "xylose", "y"])]

# ==========================================
# --- HELPER FUNCTIONS ---
# ==========================================

def apply_style(ax, title):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(AXIS_LINE_WIDTH)
    ax.spines['bottom'].set_linewidth(AXIS_LINE_WIDTH)
    ax.set_xlabel('Time (hours)', fontsize=LABEL_FONT)
    ax.set_ylabel(r'$\mathrm{OD}_{600}$', fontsize=LABEL_FONT)
    ax.set_title(title, fontsize=TITLE_FONT, pad=15)
    ax.set_xlim(TIME_LIMIT)
    ax.set_ylim(OD_LIMIT)
    ax.tick_params(axis='both', labelsize=TICK_FONT, width=AXIS_LINE_WIDTH)

def sanitize_filename(name):
    return name.replace(' ', '_').replace('/', '_').replace('\\', '_').lower()

def save_and_show(fig, folder, base_name, thicken_legend=False):
    fig.subplots_adjust(right=0.7) 
    if thicken_legend:
        leg = plt.gca().get_legend()
        if leg:
            for line in leg.get_lines():
                line.set_linewidth(LEGEND_LINE_WIDTH)
    clean_name = sanitize_filename(base_name)
    path = os.path.join(folder, clean_name)
    fig.savefig(f"{path}.png", dpi=500, bbox_inches='tight')
    fig.savefig(f"{path}.svg", format="svg", bbox_inches='tight')
    plt.show()

# ==========================================
# --- EXECUTION ---
# ==========================================

unique_pathways = [p for p in df_xyl['Pathway'].unique() if p not in ["no-sgRNA", "Unknown/Other"]]

# 1. Detailed Individual Pathway Plots (Cleaned Gene Names)
for pw in unique_pathways:
    fig, ax = plt.subplots(figsize=FIG_SIZE)
    apply_style(ax, f"Pathway Details: {pw}")
    
    # --- Control Plotting (with SD) ---
    nosg_df = df_xyl[df_xyl['Pathway'] == "no-sgRNA"]
    if not nosg_df.empty:
        stats = nosg_df.groupby('Time')[DATA_COL].agg(['mean', 'std']).reset_index()
        ax.fill_between(stats['Time'], stats['mean'] - stats['std'].fillna(0), 
                        stats['mean'] + stats['std'].fillna(0), color='red', alpha=0.1, edgecolor='none')
        ax.plot(stats['Time'], stats['mean'], color='red', lw=CONTROL_LINE_WIDTH, label='no-sgRNA', zorder=10)
    
    # --- Mutants Plotting (Clean names) ---
    pw_df = df_xyl[df_xyl['Pathway'] == pw]
    for sample in pw_df['Sample'].unique():
        # Logic to extract only the gene target from the string
        match = re.search(r"Gene target:\s*(\S+)", str(sample))
        label_name = match.group(1) if match else str(sample)
        
        s_data = pw_df[pw_df['Sample'] == sample].groupby('Time')[DATA_COL].mean()
        ax.plot(s_data.index, s_data.values, lw=MUTANT_LINE_WIDTH, alpha=0.7, label=label_name)

    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False, fontsize=LEGEND_FONT_DETAILED)
    save_and_show(fig, GENE_OUTPUT_DIR, f"Detailed_{pw}", thicken_legend=True)

# 2. Summary Plot (All Curves, Color-coded)
fig, ax = plt.subplots(figsize=FIG_SIZE)
apply_style(ax, "Summary: All Manual Pathway Curves")

nosg_df = df_xyl[df_xyl['Pathway'] == "no-sgRNA"]
if not nosg_df.empty:
    stats = nosg_df.groupby('Time')[DATA_COL].agg(['mean', 'std']).reset_index()
    ax.fill_between(stats['Time'], stats['mean'] - stats['std'].fillna(0), 
                    stats['mean'] + stats['std'].fillna(0), color='black', alpha=0.1, edgecolor='none')
    ax.plot(stats['Time'], stats['mean'], color='black', lw=CONTROL_LINE_WIDTH, label='no-sgRNA', zorder=15)

plotted_labels = set()
for pw, color in MANUAL_COLORS.items():
    pw_df = df_xyl[df_xyl['Pathway'] == pw]
    if pw_df.empty: continue
    
    for sample in pw_df['Sample'].unique():
        lbl = pw if pw not in plotted_labels else "_"
        s_data = pw_df[pw_df['Sample'] == sample].groupby('Time')[DATA_COL].mean()
        ax.plot(s_data.index, s_data.values, color=color, lw=MUTANT_LINE_WIDTH, alpha=0.5, label=lbl)
        plotted_labels.add(pw)

ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False, fontsize=18)
save_and_show(fig, OUTPUT_DIR, "Summary_Manual_All_Curves", thicken_legend=True)

In [ ]:
#gene seleciton plotten

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
from matplotlib.lines import Line2D

# ==========================================
# --- CONFIGURATION & PATHS ---
# ==========================================
BASE_PATH = r"E:\Thesis3april\GrowthResults"
EXCEL_PATH = os.path.join(BASE_PATH, "Excel_Export", "growth_data_blank_subtracted.xlsx")
ANNOTATION_PATH = r"E:\Thesis3april\overlay\Pathway_annotation.xlsx"
OUTPUT_DIR = os.path.join(BASE_PATH, "7april", "Pathway_Growth_Curves")
SPECIFIC_GENES_DIR = os.path.join(OUTPUT_DIR, "Specific_Gene_SelectionsLinear")

for d in [OUTPUT_DIR, SPECIFIC_GENES_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

# --- THICKNESS & FONT SETTINGS ---
TITLE_FONT, LABEL_FONT, TICK_FONT = 22, 23, 22
LEGEND_FONT_DETAILED = 16 
AXIS_LINE_WIDTH = 2.0
MUTANT_LINE_WIDTH = 1.8    
CONTROL_LINE_WIDTH = 2.5
LEGEND_LINE_WIDTH = 5.0    

TIME_LIMIT = (0, 14)
OD_LIMIT = (0, 1.35)
FIG_SIZE = (15, 8) 
DATA_COL = 'Smoothed_OD_Blank_Sub'

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c", 
    "cell shape": "#d62728", 
    "biosynthesis of fatty acids": "#9467bd", 
    "DNA replication": "#8c564b", 
    "DNA condensation/ segregation": "#e377c2", 
    "biosynthesis of isoprenoids": "#7f7f7f", 
    "cell division": "#bcbd22", 
    "ribosomal proteins": "#17becf", 
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78", 
    "biosynthesis of menaquinone": "#98df8a"
}

# ==========================================
# --- DATA PREPARATION ---
# ==========================================
df_plot = pd.read_excel(EXCEL_PATH)
df_ann = pd.read_excel(ANNOTATION_PATH)

def get_pathway(row):
    for col in ['SubtiWiki Annotation 4', 'SubtiWiki Annotation 3']:
        val = str(row.get(col, '')).strip()
        if val not in ['nan', 'NA', 'None', '']: return val
    return "Unknown/Other"

df_ann['Pathway'] = df_ann.apply(get_pathway, axis=1)
pathway_map = dict(zip(df_ann['Treatment'].astype(str).str.strip(), df_ann['Pathway']))

_no_sg_patterns = re.compile(r'no[_\-\s]?s?g?r?n?a?', flags=re.IGNORECASE)
def is_no_sgRNA(sample): return bool(_no_sg_patterns.search(str(sample)))

def assign_pathway(sample):
    if is_no_sgRNA(sample): return "no-sgRNA"
    sample_str = str(sample)
    for target, pw in pathway_map.items():
        if target in sample_str:
            return pw
    return "Unknown/Other"

df_plot['Pathway'] = df_plot['Sample'].apply(assign_pathway)
df_xyl = df_plot[df_plot['Xylose'].astype(str).str.lower().isin(["true", "yes", "1", "xylose", "y"])]

# ==========================================
# --- HELPER FUNCTIONS ---
# ==========================================

def apply_style(ax, title):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(AXIS_LINE_WIDTH)
    ax.spines['bottom'].set_linewidth(AXIS_LINE_WIDTH)
    ax.set_xlabel('Time (hours)', fontsize=LABEL_FONT)
    ax.set_ylabel(r'$\mathrm{OD}_{600}$', fontsize=LABEL_FONT)
    ax.set_title(title, fontsize=TITLE_FONT, pad=15)
    ax.set_xlim(TIME_LIMIT)
    ax.set_ylim(OD_LIMIT)
    ax.tick_params(axis='both', labelsize=TICK_FONT, width=AXIS_LINE_WIDTH)

def sanitize_filename(name):
    return name.replace(' ', '_').replace('/', '_').replace('\\', '_').lower()

def save_and_show(fig, folder, base_name, thicken_legend=False):
    fig.subplots_adjust(right=0.7) 
    if thicken_legend:
        leg = plt.gca().get_legend()
        if leg:
            for line in leg.get_lines():
                line.set_linewidth(LEGEND_LINE_WIDTH)
    clean_name = sanitize_filename(base_name)
    path = os.path.join(folder, clean_name)
    fig.savefig(f"{path}.png", dpi=500, bbox_inches='tight')
    fig.savefig(f"{path}.svg", format="svg", bbox_inches='tight')
    plt.show()

# ==========================================
# --- PLOT SPECIFIC GENES ---
# ==========================================

def plot_gene_selection(df, gene_list, plot_title, filename):
    fig, ax = plt.subplots(figsize=FIG_SIZE)
    apply_style(ax, plot_title)

    # 1. Plot no-sgRNA with SD
    nosg_df = df[df['Pathway'] == "no-sgRNA"]
    if not nosg_df.empty:
        stats = nosg_df.groupby('Time')[DATA_COL].agg(['mean', 'std']).reset_index()
        ax.fill_between(stats['Time'], stats['mean'] - stats['std'].fillna(0), 
                        stats['mean'] + stats['std'].fillna(0), color='red', alpha=0.1, edgecolor='none')
        ax.plot(stats['Time'], stats['mean'], color='red', lw=CONTROL_LINE_WIDTH, label='no-sgRNA', zorder=10)

    # 2. Plot specific genes with pathway colors
    for gene in gene_list:
        # Find samples that contain the gene name
        gene_samples = df[df['Sample'].str.contains(gene, case=False, na=False)]
        
        if gene_samples.empty:
            print(f"Warning: Gene {gene} not found in data.")
            continue
            
        # Determine color based on pathway
        pathway = pathway_map.get(gene, "Unknown/Other")
        color = MANUAL_COLORS.get(pathway, "gray")
        
        # Plot mean curve for this gene
        s_data = gene_samples.groupby('Time')[DATA_COL].mean()
        ax.plot(s_data.index, s_data.values, color=color, lw=MUTANT_LINE_WIDTH, label=gene)

    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False, fontsize=LEGEND_FONT_DETAILED)
    save_and_show(fig, SPECIFIC_GENES_DIR, filename, thicken_legend=True)

# Run the specific plot requested
#my_genes = ['leuS', 'dapB', 'pgm', 'fabF', 'rpsD']
my_genes = ['ftsZ', 'ftsW', 'rpsJ', 'rpsK']
plot_gene_selection(df_xyl, my_genes, "Growth of Selected Pathway Genes", "Selected_Pathway_Comparison")

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import re
from matplotlib.lines import Line2D

# --- SETUP DIRECTORIES ---
base_path = r"E:\Thesis3april\GrowthResults"
# New subfolder for combined plots
subfolder_path = os.path.join(base_path, "combined_growth_curves_blank_sub")

if not os.path.exists(subfolder_path):
    os.makedirs(subfolder_path)
    print(f"Created directory: {subfolder_path}")

# Robust detection of Xylose presence
def _is_xylose_val(v):
    try:
        s = str(v).strip().lower()
        return s in ("true", "yes", "1", "xylose", "y")
    except Exception:
        return False

# Ensure flag exists in df_blank_subs
if 'Xylose_flag' not in df_blank_subs.columns:
    df_blank_subs['Xylose_flag'] = df_blank_subs['Xylose'].apply(_is_xylose_val)

# Regex patterns for identifying control samples
_no_sg_patterns = re.compile(r'no[_\-\s]?s?g?r?n?a?', flags=re.IGNORECASE)

def is_no_sgRNA(sample):
    return bool(_no_sg_patterns.search(str(sample)))

def plot_all_with_and_without_xylose(df, time_limit=(0,14), figsize=(12,8)):
    """
    Produces two figures using Smoothed_OD_Blank_Sub.
    Plotted region is square, legend is placed outside without a box.
    """
    DATA_COL = 'Smoothed_OD_Blank_Sub'
    
    # Font Sizes
    TITLE_FONT = 24
    LABEL_FONT = 22
    TICK_FONT = 22
    LEGEND_FONT = 22

    d = df.copy()
    if 'Xylose_flag' not in d.columns:
        d['Xylose_flag'] = d['Xylose'].apply(_is_xylose_val)

    groups = {
        'With Xylose': d[d['Xylose_flag'] == True],
        'No Xylose': d[d['Xylose_flag'] == False]
    }

    def color_by_substring(sample):
        s = str(sample).lower()
        if 'sgl999' in s:
            return 'orange'
        return 'gray'

    for title, subdf in groups.items():
        if subdf.empty:
            print(f'No samples found for group: {title}')
            continue

        # figsize (8,8) creates the square base
        fig, ax = plt.subplots(figsize=figsize)

        # --- 1) PLOT MUTANTS FIRST ---
        other_samples = [s for s in subdf['Sample'].unique() if not is_no_sgRNA(s)]

        for sample in other_samples:
            sample_data = subdf[subdf['Sample'] == sample]
            color = color_by_substring(sample)
            
            stats = sample_data.groupby('Time')[DATA_COL].mean().reset_index()

            ax.plot(
                stats['Time'], stats[DATA_COL],
                color=color, linewidth=1.7, alpha=0.5,
                zorder=1
            )

        # --- 2) PLOT no_sgRNA ON TOP ---
        nosg_df = subdf[subdf['Sample'].apply(is_no_sgRNA)]
        if not nosg_df.empty:
            nosg_stats = nosg_df.groupby('Time')[DATA_COL].agg(['mean', 'std']).reset_index()
            time = nosg_stats['Time']
            mean = nosg_stats['mean']
            std = nosg_stats['std'].fillna(0)

            # SD shaded area
            ax.fill_between(
                time, mean - std, mean + std,
                color='red', alpha=0.2, edgecolor='none',
                zorder=3
            )

            # Solid mean line
            ax.plot(
                time, mean,
                color='red', linewidth=2.5,
                zorder=4
            )

        # --- STYLING: SQUARE PLOT AREA ---
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        # Ensure the axis box itself remains square within the figure
        ax.set_aspect('auto') 

        ax.set_xlabel('Time (hours)', fontsize=LABEL_FONT)
        ax.set_ylabel(r'$\mathrm{OD}_{600}$', fontsize=LABEL_FONT)
        ax.set_title(f'{title}', fontsize=TITLE_FONT, pad=15)
        ax.set_xlim(time_limit)
        ax.set_ylim((0, 1.35))
        ax.tick_params(axis='both', which='major', labelsize=TICK_FONT)

        # --- LEGEND: NEXT TO PLOT, NO BOX, NO SGL ---
        legend_elements = [
            Line2D([0], [0], color='red', lw=2.5, label='no_sgRNA'),
            Line2D([0], [0], color='gray', lw=2, label='mutants')
        ]

        # bbox_to_anchor moves it outside; frameon=False removes the box
        ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1, 1), 
                  fontsize=LEGEND_FONT, frameon=False)

        # --- SAVE ---
        file_name = title.replace(' ', '_').lower()
        file_base = os.path.join(subfolder_path, f"{file_name}_combined_growth")
        
        # bbox_inches='tight' is crucial when the legend is outside the axes
        plt.savefig(f"{file_base}.svg", format="svg", bbox_inches='tight')
        plt.savefig(f"{file_base}.png", format="png", dpi=500, bbox_inches='tight')
        
        plt.show()

# Run the function
plot_all_with_and_without_xylose(df_blank_subs)

In [ ]:
#excel gebruiken om het samen te plotten

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
from matplotlib.lines import Line2D

# ==========================================
# --- CONFIGURATION & FONT PARAMETERS ---
# ==========================================
BASE_PATH = r"E:\Thesis3april\GrowthResults"
EXCEL_PATH = os.path.join(BASE_PATH, "Excel_Export", "growth_data_blank_subtracted.xlsx")

# Font Sizes
TITLE_FONT = 24
LABEL_FONT = 20
TICK_FONT = 20      # Font size for axis numbers
LEGEND_FONT = 20

# Plotting Parameters
TIME_LIMIT = (0, 14)
OD_LIMIT = (0, 1.35)
FIG_SIZE = (12, 8)   # Ensures a square-based figure
DATA_COL = 'Smoothed_OD_Blank_Sub'

# ==========================================
# --- DATA LOADING & PREPARATION ---
# ==========================================

# Load the data from the exported Excel
if os.path.exists(EXCEL_PATH):
    df_plot = pd.read_excel(EXCEL_PATH)
else:
    print(f"Error: Excel file not found at {EXCEL_PATH}")
    # Fallback to df_blank_subs if already in memory
    df_plot = df_blank_subs.copy()

def _is_xylose_val(v):
    try:
        s = str(v).strip().lower()
        return s in ("true", "yes", "1", "xylose", "y")
    except Exception:
        return False

# Ensure Xylose flag exists
if 'Xylose_flag' not in df_plot.columns:
    df_plot['Xylose_flag'] = df_plot['Xylose'].apply(_is_xylose_val)

# Regex for control detection
_no_sg_patterns = re.compile(r'no[_\-\s]?s?g?r?n?a?', flags=re.IGNORECASE)

def is_no_sgRNA(sample):
    return bool(_no_sg_patterns.search(str(sample)))

# ==========================================
# --- PLOTTING FUNCTION ---
# ==========================================

def plot_all_with_and_without_xylose(df):
    """
    Produces two figures: With Xylose and No Xylose.
    Saves to the 'combined_growth_curves_blank_sub' subfolder.
    """
    
    subfolder_path = os.path.join(BASE_PATH, "combined_growth_curves_blank_sub")
    if not os.path.exists(subfolder_path):
        os.makedirs(subfolder_path)

    groups = {
        'With Xylose': df[df['Xylose_flag'] == True],
        'No Xylose': df[df['Xylose_flag'] == False]
    }

    def color_by_substring(sample):
        s = str(sample).lower()
        if 'sgl999' in s:
            return 'orange'
        return 'gray'

    for title, subdf in groups.items():
        if subdf.empty:
            print(f'No samples found for group: {title}')
            continue

        fig, ax = plt.subplots(figsize=FIG_SIZE)

        # --- 1) PLOT MUTANTS ---
        other_samples = [s for s in subdf['Sample'].unique() if not is_no_sgRNA(s)]

        for sample in other_samples:
            sample_data = subdf[subdf['Sample'] == sample]
            color = color_by_substring(sample)
            
            stats = sample_data.groupby('Time')[DATA_COL].mean().reset_index()

            ax.plot(
                stats['Time'], stats[DATA_COL],
                color=color, linewidth=1, alpha=0.5,
                zorder=1
            )

        # --- 2) PLOT no_sgRNA (Control) ---
        nosg_df = subdf[subdf['Sample'].apply(is_no_sgRNA)]
        if not nosg_df.empty:
            nosg_stats = nosg_df.groupby('Time')[DATA_COL].agg(['mean', 'std']).reset_index()
            time = nosg_stats['Time']
            mean = nosg_stats['mean']
            std = nosg_stats['std'].fillna(0)

            ax.fill_between(
                time, mean - std, mean + std,
                color='red', alpha=0.2, edgecolor='none', zorder=3
            )
            ax.plot(time, mean, color='red', linewidth=2.5, zorder=4)

        # --- STYLING ---
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_aspect('auto') 

        ax.set_xlabel('Time (hours)', fontsize=LABEL_FONT)
        # Non-italic OD with subscript 600
        ax.set_ylabel(r'$\mathrm{OD}_{600}$', fontsize=LABEL_FONT)
        
        ax.set_title(f'{title}', fontsize=TITLE_FONT, pad=15)
        ax.set_xlim(TIME_LIMIT)
        ax.set_ylim(OD_LIMIT)
        
        # Apply Tick Font Size (Axis Numbers)
        ax.tick_params(axis='both', which='major', labelsize=TICK_FONT)

        # --- LEGEND: OUTSIDE PLOT, NO BOX ---
        legend_elements = [
            Line2D([0], [0], color='red', lw=2.5, label='no_sgRNA'),
            Line2D([0], [0], color='gray', lw=1, label='mutants')
        ]

        ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1, 1), 
                  fontsize=LEGEND_FONT, frameon=False)

        # --- SAVE ---
        file_name = title.replace(' ', '_').lower()
        file_base = os.path.join(subfolder_path, f"{file_name}_combined_growth")
        
        plt.savefig(f"{file_base}.svg", format="svg", bbox_inches='tight')
        plt.savefig(f"{file_base}.png", format="png", dpi=500, bbox_inches='tight')
        
        plt.show()

# Run the function
plot_all_with_and_without_xylose(df_plot)

In [ ]:
# Additional plotting helpers: specify gene targets and combined xylose/no-xylose plots
import matplotlib.pyplot as plt
import numpy as np
import re

# Robust detection of Xylose presence (handles booleans, ints and several strings)
def _is_xylose_val(v):
    try:
        s = str(v).strip().lower()
        return s in ("true", "yes", "1", "xylose", "y")
    except Exception:
        return False

# Add a boolean column that flags the presence of xylose if not already present
if 'Xylose_flag' not in df_blank_subs.columns:
    df_blank_subs['Xylose_flag'] = df_blank_subs['Xylose'].apply(_is_xylose_val)

# Helper to pick color by sample name (detects no_sgRNA and SGL999 variants)
_no_sg_patterns = re.compile(r'no[_\-\s]?s?g?r?n?a?', flags=re.IGNORECASE)  # matches many variants of "no_sgRNA"
_sgl999_pattern = re.compile(r'sgl999', flags=re.IGNORECASE)

def _sample_color_by_name(sample_name):
    s = str(sample_name)
    if _no_sg_patterns.search(s):
        return 'red'
    if _sgl999_pattern.search(s):
        return 'orange'
    return 'gray'



def plot_all_with_and_without_xylose(df, time_limit=(0,14), figsize=(12,8)):
    """
    Produces two figures:
      - all samples WITH xylose
      - all samples WITHOUT xylose
    Only the mean for each sample is plotted (thin lines).
    Color scheme:
      - no_sgRNA (substring, case-insensitive) -> red
      - SGL999 (substring, case-insensitive) -> orange
      - other samples -> gray
    """
    d = df.copy()
    if 'Xylose_flag' not in d.columns:
        def _is_xylose_val(v):
            try:
                s = str(v).strip().lower()
                return s in ("true", "yes", "1", "xylose", "y")
            except Exception:
                return False
        d['Xylose_flag'] = d['Xylose'].apply(_is_xylose_val)

    groups = {
        'xylose': d[d['Xylose_flag'] == True],
        'no xylose': d[d['Xylose_flag'] == False]
    }

    def color_by_substring(sample):
        s = str(sample).lower()
        if 'no_sgrna' in s:
            return 'red'
        if 'sgl999' in s:
            return 'orange'
        return 'gray'

    for title, subdf in groups.items():
        if subdf.empty:
            print(f'No samples found for group: {title}')
            continue

        plt.figure(figsize=figsize)
        for sample in subdf['Sample'].unique():
            sample_data = subdf[subdf['Sample'] == sample]
            color = color_by_substring(sample)
            # plot only the mean (thin line)
            mean_df = sample_data.groupby('Time')['Smoothed_OD'].mean().reset_index()
            plt.plot(mean_df['Time'], mean_df['Smoothed_OD'], color=color, linewidth=1, label=sample)
        # Create legend entries only for color coding (no_sgRNA, SGL999, other)
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], color='red', lw=1, label='no_sgRNA'),
           
            Line2D([0], [0], color='gray', lw=1, label='mutants')
        ]
        plt.xlabel('Time (hours)')
        plt.ylabel('OD')
        plt.title(f'{title}')
        plt.xlim(time_limit)
        
        plt.legend(handles=legend_elements, loc='upper right')
        plt.savefig(f"{title.replace(' ', '_')}.svg", format="svg")
        plt.show()

# Example usage:


plot_all_with_and_without_xylose(df_smoothed)



In [ ]:
print(df_smoothed)
import pandas as pd
import re

# NOTE: Replace this section with your actual 'df_smoothed' DataFrame loading
# or ensure the following code is applied directly to your existing 'df_smoothed'.

# 1. Extract Strain Name and Gene Target using regular expressions
# Extract Strain name: captures everything between 'Strain name: ' and the following ';'
df_smoothed['Strain_Name'] = df_smoothed['Sample'].str.extract(r'Strain name:\s*(.*?);')
# Extract Gene target: captures everything after 'Gene target: '
df_smoothed['Gene_Target'] = df_smoothed['Sample'].str.extract(r'Gene target:\s*(.*)')

# Clean up any potential leading/trailing whitespace
df_smoothed['Strain_Name'] = df_smoothed['Strain_Name'].str.strip()
df_smoothed['Gene_Target'] = df_smoothed['Gene_Target'].str.strip()

# 2. Total number of unique samples (unique Strain Names)
unique_strains_count = df_smoothed['Strain_Name'].nunique()

# 3. List of all unique sample names (Strain Names)
unique_strain_names = df_smoothed['Strain_Name'].unique().tolist()

# 4. Count of unique strains that contain 'no_sgRNA' as gene target
# First, create a temporary DataFrame with only unique Strain/Target combinations
unique_strains_targets = df_smoothed[['Strain_Name', 'Gene_Target']].drop_duplicates()

# Then, filter this unique list for 'no_sgRNA' and count the strains
no_sgRNA_count = unique_strains_targets[unique_strains_targets['Gene_Target'] == 'no_sgRNA']['Strain_Name'].nunique()

print(f"Total Unique Strains Count: {unique_strains_count}")
print(f"List of Unique Strain Names: {unique_strain_names}")
print(f"Unique Strains with 'no_sgRNA' Gene Target Count: {no_sgRNA_count}")

In [ ]:
# Additional plotting helpers: specify gene targets and combined xylose/no-xylose plots
import matplotlib.pyplot as plt
import numpy as np
import re

# Robust detection of Xylose presence (handles booleans, ints and several strings)
def _is_xylose_val(v):
    try:
        s = str(v).strip().lower()
        return s in ("true", "yes", "1", "xylose", "y")
    except Exception:
        return False

# Add a boolean column that flags the presence of xylose if not already present
if 'Xylose_flag' not in df_blank_subs.columns:
    df_blank_subs['Xylose_flag'] = df_blank_subs['Xylose'].apply(_is_xylose_val)

# Regex patterns
_no_sg_patterns = re.compile(r'no[_\-\s]?s?g?r?n?a?', flags=re.IGNORECASE)
_sgl999_pattern = re.compile(r'sgl999', flags=re.IGNORECASE)

def is_no_sgRNA(sample):
    return bool(_no_sg_patterns.search(str(sample)))


def plot_all_with_and_without_xylose(df, time_limit=(0,14), figsize=(12,8)):
    """
    Produces two figures:
      - all samples WITH xylose
      - all samples WITHOUT xylose

    Behavior:
      - no_sgRNA samples are combined into ONE group → mean + SD shaded area (red)
      - all mutants plotted individually (gray or orange)
      - no_sgRNA plotted LAST (higher zorder) so it overlays the others
    """
    d = df.copy()
    if 'Xylose_flag' not in d.columns:
        d['Xylose_flag'] = d['Xylose'].apply(_is_xylose_val)

    groups = {
        'xylose': d[d['Xylose_flag'] == True],
        'no xylose': d[d['Xylose_flag'] == False]
    }

    def color_by_substring(sample):
        s = str(sample).lower()
        if 'sgl999' in s:
            return 'orange'
        return 'gray'

    for title, subdf in groups.items():
        if subdf.empty:
            print(f'No samples found for group: {title}')
            continue

        plt.figure(figsize=figsize)

        # --- 1) PLOT ALL OTHER SAMPLES FIRST (lower zorder = underneath) ---
        other_samples = [s for s in subdf['Sample'].unique() if not is_no_sgRNA(s)]

        for sample in other_samples:
            sample_data = subdf[subdf['Sample'] == sample]
            color = color_by_substring(sample)

            stats = sample_data.groupby('Time')['Smoothed_OD'].mean().reset_index()

            plt.plot(
                stats['Time'], stats['Smoothed_OD'],
                color=color, linewidth=1,
                zorder=1   # draw below no_sgRNA
            )

        # --- 2) PLOT no_sgRNA LAST (on top) ---
        nosg_df = subdf[subdf['Sample'].apply(is_no_sgRNA)]
        if not nosg_df.empty:
            nosg_stats = nosg_df.groupby('Time')['Smoothed_OD'].agg(['mean', 'std']).reset_index()
            time = nosg_stats['Time']
            mean = nosg_stats['mean']
            std = nosg_stats['std'].fillna(0)

            # SD shaded area (under the line but still above mutants)
            plt.fill_between(
                time, mean - std, mean + std,
                color='red',
                alpha=0.2,
                edgecolor='none',
                zorder=3   # above mutants but under the no_sgRNA line
            )

            # Solid mean line (drawn on top)
            plt.plot(
                time, mean,
                color='red', linewidth=2,
                zorder=4   # very top
            )

        # Legend
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], color='red', lw=2, label='no_sgRNA (mean ± SD)'),
            Line2D([0], [0], color='gray', lw=1, label='mutants'),
           
        ]

        plt.xlabel('Time (hours)')
        plt.ylabel('OD')
        plt.title(f'{title}')
        plt.xlim(time_limit)
        plt.ylim((0,1.35))
        plt.legend(handles=legend_elements, loc='upper right')

        plt.savefig(f"{title.replace(' ', '_')}.svg", format="svg")
        plt.show()


# Example usage:
plot_all_with_and_without_xylose(df_smoothed)


In [ ]:
#dubbele namen
#strains_and_genes_full_name = [
   # 'Strain name: BEC14840.27.2; Gene target: ylaN27_2',
    #'Strain name: BEC14840.87; Gene target: ylaN87',
   # 'Strain name: BEC14840.189; Gene target: ylaN189',
   # 'Strain name: BEC14840.192; Gene target: ylaN192',
   # 'Strain name: BEC14840.27.1; Gene target: ylaN27_1',
   # 'Strain name: BEC06811.130; Gene target: yezG130',
    #'Strain name: CAG74399_47; Gene target: yezG',
    #'Strain name: CAG74399_45; Gene target: yqaE',
    #'Strain name: BEC26350.101; Gene target: yqaE101',
    #'Strain name: BEC04620.50.1; Gene target: acpS50_1',
    #'Strain name: BEC04620.50.2; Gene target: acpS50_2',
    #'Strain name: CAG74399_66; Gene target: ftsZ',
   # 'Strain name: BEC15290.226; Gene target: ftsZ226',
   # 'Strain name: BEC01320.2; Gene target: rplR-2',
   # 'Strain name: BEC01320.105; Gene target: rplR105'
#]

# Use the new function to plot mean and standard deviation for specific targets
plot_gene_targets_mean_std(df_smoothed,  [
    'ylaN27_2',
    'ylaN87',
    'ylaN189',
    'ylaN192',
    'ylaN27_1',
    'yezG130',
    'yezG',
    'yqaE',
    'yqaE101',
    'acpS50_1',
    'acpS50_2',
    'ftsZ',
    'ftsZ226',
    'rplR-2',
    'rplR105'
])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import re

# Robust detection of Xylose presence (handles booleans, ints and several strings)
def _is_xylose_val(v):
    try:
        s = str(v).strip().lower()
        return s in ("true", "yes", "1", "xylose", "y")
    except Exception:
        return False

# Ensure a consistent OD column is chosen
def _choose_od_column(df):
    for col in ('Smoothed_OD_Blank_Sub', 'Smoothed_OD', 'OD'):
        if col in df.columns:
            return col
    raise ValueError("No OD column found. Expected one of: Smoothed_OD_Blank_Sub, Smoothed_OD, OD")

# Add Xylose_flag column if not present
if 'Xylose_flag' not in df_blank_subs.columns:
    df_blank_subs = df_blank_subs.copy()
    df_blank_subs['Xylose_flag'] = df_blank_subs['Xylose'].apply(_is_xylose_val)


def plot_gene_targets_with_and_without_xylose(df, gene_targets, time_limit=(0, 14), figsize=(10, 6)):
    """
    For each gene target, plot all replicates with and without xylose on the same plot.
    Matches samples by substring (case-insensitive) using the approach:
        mask = df['Sample'].astype(str).str.lower().str.contains(target)
    - df: dataframe with columns ['Sample','rep','Time', <OD column>, 'Xylose' or 'Xylose_flag']
    - gene_targets: list or single string of substrings to match (case-insensitive)
    - time_limit: (xmin, xmax) for x-axis
    """
    if not isinstance(gene_targets, (list, tuple)):
        gene_targets = [gene_targets]
    targets_lower = [g.lower() for g in gene_targets]

    d = df.copy()
    if 'Xylose_flag' not in d.columns:
        d['Xylose_flag'] = d['Xylose'].apply(_is_xylose_val)

    od_col = _choose_od_column(d)

    for target in targets_lower:
        mask = d['Sample'].astype(str).str.lower().str.contains(target)
        if mask.sum() == 0:
            print(f'No samples match gene target: {target}')
            continue

        sel = d[mask].copy()
        plt.figure(figsize=figsize)

        # We'll plot replicates:
        # - With xylose -> blue
        # - Without xylose -> green
        # Make sure legend entries only appear once
        seen_labels = set()

        for (sample, rep), rep_data in sel[sel['Xylose_flag'] == True].groupby(['Sample', 'rep']):
            label = 'With xylose' if 'With xylose' not in seen_labels else None
            plt.plot(rep_data['Time'], rep_data[od_col], color='blue', alpha=0.7, linewidth=1, label=label)
            seen_labels.add('With xylose')

        for (sample, rep), rep_data in sel[sel['Xylose_flag'] == False].groupby(['Sample', 'rep']):
            label = 'Without xylose' if 'Without xylose' not in seen_labels else None
            plt.plot(rep_data['Time'], rep_data[od_col], color='green', alpha=0.7, linewidth=1, label=label)
            seen_labels.add('Without xylose')

        plt.xlabel('Time (hours)')
        plt.ylabel(od_col)
        plt.title(f'{target} — with and without xylose')
        plt.xlim(time_limit)
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()


def plot_all_with_and_without_xylose(df, time_limit=(0, 14), figsize=(12, 8), save_svg=False, outdir='plots'):
    """
    Produces two figures:
      - all samples WITH xylose
      - all samples WITHOUT xylose
    Only the mean for each sample is plotted (thin lines).
    Color scheme (substring, case-insensitive):
      - 'no_sgRNA' substring -> red
      - 'SGL999' substring   -> orange
      - other samples        -> gray

    If save_svg is True, saves each figure to `outdir` as an SVG.
    """
    d = df.copy()
    if 'Xylose_flag' not in d.columns:
        d['Xylose_flag'] = d['Xylose'].apply(_is_xylose_val)

    od_col = _choose_od_column(d)

    groups = {
        'With_xylose': d[d['Xylose_flag'] == True],
        'Without_xylose': d[d['Xylose_flag'] == False]
    }

    def _color_by_substring(sample_name):
        s = str(sample_name).lower()
        if 'no_sg' in s or 'no_sgrna' in s or 'no_sgRNA'.lower() in s:
            return 'red'
        if 'sgl999' in s:
            return 'orange'
        return 'gray'

    def _sanitize_filename(s):
        s = str(s).strip()
        s = re.sub(r'\s+', '_', s)
        s = re.sub(r'[^A-Za-z0-9_.-]', '', s)
        return s

    if save_svg:
        import os
        os.makedirs(outdir, exist_ok=True)

    for title, subdf in groups.items():
        if subdf.empty:
            print(f'No samples found for group: {title}')
            continue

        plt.figure(figsize=figsize)
        # For each sample, compute mean across replicates and plot thin line
        for sample in subdf['Sample'].unique():
            sample_data = subdf[subdf['Sample'] == sample]
            mean_df = sample_data.groupby('Time')[od_col].mean().reset_index()
            color = _color_by_substring(sample)
            plt.plot(mean_df['Time'], mean_df[od_col], color=color, linewidth=1, label=sample)

        # Build color legend
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], color='red', lw=1, label='no_sgRNA (substring)'),
            Line2D([0], [0], color='orange', lw=1, label='SGL999 (substring)'),
            Line2D([0], [0], color='gray', lw=1, label='other samples')
        ]

        plt.xlabel('Time (hours)')
        plt.ylabel(od_col)
        plt.title(f'{title.replace("_", " ")} — all samples (mean only)')
        plt.xlim(time_limit)
        plt.grid(True)
        plt.legend(handles=legend_elements, loc='upper right')
        plt.tight_layout()

        if save_svg:
            import os
            fname = f"{_sanitize_filename(title)}_mean_only.svg"
            path = os.path.join(outdir, fname)
            plt.savefig(path, format='svg')
            print(f"Saved SVG: {path}")

        plt.show()
        plt.close()

plot_gene_targets_with_and_without_xylose(df_smoothed, ['no_sgRNA', 'SGL999', 'rghRA','pgm','sufS','dxr','dapL'])
plot_all_with_and_without_xylose(df_smoothed)


# Example usage (adapt to the DataFrame you want to plot):
# plot_gene_targets_with_and_without_xylose(df_smoothed, ['rghRA', 'pgm'])
# plot_all_with_and_without_xylose(df_smoothed, save_svg=True, outdir='plots')

In [ ]:


# --- Log2 calculation per replicate ---

df_log2 = (
    df_blank_subs.groupby(['Sample', 'rep','Xylose'])
    .apply(calculate_log2)
    .drop(columns=['OD', 'Smoothed_OD', 'Blank_value'])
    .reset_index(drop=True)
)

print(df_log2.head())

# --- Extract metrics per replicate ---

df_metrics = df_blank_subs.groupby(['Sample', 'rep', 'Xylose']).apply(extract_metrics).reset_index()
print(df_metrics.head())

# --- Check-up plot for Log2 calculation ---

if sample_to_plot in df_log2['Sample'].unique():
    fig, ax = plt.subplots(figsize=(10, 6))
    for (rep,xylose), rep_data in df_log2[df_log2['Sample'] == sample_to_plot].groupby(['rep', 'Xylose']):
        ax.plot(rep_data['Time'], rep_data['Log2'], marker='o', linestyle='-', label=f'Rep {rep}{xylose}')
    ax.set_xlabel('Time (minutes)')
    ax.set_ylabel('Log2(OD)')
    ax.set_title(f'Log2(OD) for {sample_to_plot}')
    ax.legend()
    ax.grid(True)
    plt.show()
else:
    print('The specified sample is not present in the Log2-calculated data.')


# --- Drop invalid Log2 values ---

original_rows = len(df_log2)
df_log2 = df_log2.dropna(subset=['Log2'])
df_log2 = df_log2[df_log2['Log2'] != -np.inf]
new_rows = len(df_log2)

print(f"Number of rows dropped due to NaN or -inf in Log2 column: {original_rows - new_rows}")



In [ ]:
regression_df = (
    df_log2
    .groupby(['Sample','rep','Xylose'])
    .apply(lambda g: calculate_sliding_regressions(g, window_regression)
           .assign(
               Sample=g['Sample'].iloc[0],
               rep=g['rep'].iloc[0], Xylose=g['Xylose'].iloc[0],
           ))
    .reset_index(drop=True)
)

print(regression_df)

In [ ]:
# Define the filtering values to select the best candidate for visible exponential phase

crit1_linearity = 0.99 # Used in step B of the filtering to discard regressions with a good linear fit (exponential phase should have a bad linear fit). Typically = 0.99
crit2_ODincr =  0.002 # Used in step C of first exponential phase filtering to discard regressions with low OD increase. Typically = 0.002
crit3_fit =  0.97 # Used in step D of first exponential phase filtering to discard regressions with bad fit (R-squared >= crit3_fit, typically = 0.99)

In [ ]:
# Make a copy of blank-subtracted OD and rename column to 'Log2' for regression reuse
df_od_for_regression = df_blank_subs.copy().rename(columns={'Smoothed_OD_Blank_Sub': 'Log2'})

# Run calculate_sliding_regressions on the OD values per sample–replicate, keeping 'xylose'
regression_OD_df = (
    df_od_for_regression
    .groupby(['Sample','rep','Xylose'])
    .apply(lambda g: calculate_sliding_regressions(g, window_regression)
           .assign(
               Sample=g['Sample'].iloc[0],
               rep=g['rep'].iloc[0], Xylose=g['Xylose'].iloc[0]
           ))
    .reset_index(drop=True)
)
print(regression_OD_df)

# Merge on Sample, rep, Start_Time, End_Time
merged = regression_df.merge(
    regression_OD_df[['Sample', 'rep', 'Xylose','Start_Time', 'End_Time', 'R_squared']],
    on=['Sample', 'rep','Xylose', 'Start_Time', 'End_Time'],
    suffixes=('', '_OD')
)

# Filter out candidates where the OD linear fit is better than the exponential fit
filtered_regression_df = merged[merged['R_squared_OD'] < merged['R_squared']]

# Further filter by user-defined linearity threshold
filtered_regression_df = filtered_regression_df[filtered_regression_df['R_squared_OD'] < crit1_linearity]

# Drop the extra column if desired
filtered_regression_df = filtered_regression_df.drop(columns=['R_squared_OD'])

# --- Summary prints & plots per replicate ---
print('Number of candidate exponential phases before check for exponential fit:')
print(regression_df.groupby(['Sample', 'rep', 'Xylose']).size())

plot_sample_regressions(df_log2, regression_df, sample_to_plot, "Candidates before linear check for")

print('Number of candidate exponential phases after check for exponential fit:')
print(filtered_regression_df.groupby(['Sample', 'rep', 'Xylose']).size())

plot_sample_regressions(df_log2, filtered_regression_df, sample_to_plot, "Candidates after linear check for")





In [ ]:
# Filtering of the candidate regressions to extract the exponential phase
# Uncomment and change the sample name to plot a different sample
# sample_to_plot = 'Sample X6'

sorted_df = filtered_regression_df.copy()

# Step C: Filter out candidate regressions with low OD differences
sorted_df['Start_OD'] = np.power(2, sorted_df['Start_Log2'])
sorted_df['End_OD'] = np.power(2, sorted_df['End_Log2'])
sorted_df['difference_OD'] = sorted_df['End_OD'] - sorted_df['Start_OD']

# Filter per replicate
sorted_df = sorted_df[sorted_df['difference_OD'] >= crit2_ODincr].drop('difference_OD', axis=1)

# Plot after first filtering step
plot_sample_regressions(df_log2, sorted_df, sample_to_plot, 'After removing low OD differences for')

# Step D: Keep candidate regressions with high fit scores
sorted_df = sorted_df[sorted_df['R_squared'] >= crit3_fit]

# Plot after second filtering step
plot_sample_regressions(df_log2, sorted_df, sample_to_plot, 'After keeping R-squared >= ' + str(crit3_fit) + ' for')

# Save candidates for downstream secondary exponential phase detection
sorted_df_secondary_expo = sorted_df.copy()

# Step E: Extract candidate regression with the highest slope per replicate
sorted_df = sorted_df.sort_values(by='Slope', ascending=False).reset_index(drop=True)
sorted_df['Rank_S'] = sorted_df.groupby(['Sample','rep', 'Xylose']).cumcount() + 1

# Select the top regression for each sample-replicate
top_rows_df = sorted_df[sorted_df['Rank_S'] == 1].drop('Rank_S', axis=1)

# Plot the final selection
plot_sample_regressions(df_log2, top_rows_df, sample_to_plot, 'Final selection for')

# Print the final result
print(top_rows_df)

# Check the values extracted for one of the samples
if sample_to_plot in df_smoothed['Sample'].unique():
    print(top_rows_df[(top_rows_df['Sample'] == sample_to_plot)])
else:
    print('The indicated sample to plot is not present in this acquisition.')






In [ ]:
# Refinement of the exponential phases
authorized_diff = 2  # Threshold for refinement (e.g., 1.1–3)
print(top_rows_df)
# Apply refinement per replicate
final_first_expo_df = (
    top_rows_df
    .groupby(['Sample', 'rep','Xylose'])
    .apply(lambda g: refine_exponential_phase(g, df_log2, authorized_diff))
    .reset_index(drop=True)
)
print('dit is final expo df')
print(final_first_expo_df)

# Plot the best regression for a specific sample
# Uncomment and change the sample name to plot a different sample
# sample_to_plot = 'Sample X20'

# Plot the regressions for the specified sample after refinement
plot_sample_regressions(df_log2, final_first_expo_df, sample_to_plot, 'After refinement - Exponential phase of ')

# Display the refined data for the selected sample if it exists in the final dataset
if sample_to_plot in final_first_expo_df['Sample'].unique():
    print(final_first_expo_df[final_first_expo_df['Sample'] == sample_to_plot])
else:
    print(f"The sample '{sample_to_plot}' is not present in the final dataset.")


In [ ]:
import matplotlib.pyplot as plt

# Unique strains
samples = df_blank_subs['Sample'].unique()
nrows = len(samples)
ncols = 2  # no xylose vs xylose

fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(14, 6 * nrows), squeeze=False)

for i, sample in enumerate(samples):
    # Subset data for this strain
    sample_data = df_blank_subs[df_blank_subs['Sample'] == sample]
    sample_expo = final_first_expo_df[final_first_expo_df['Sample'] == sample]

    # --- Plot no xylose ---
    sample_no_xylose = sample_data[sample_data['Xylose'] == 'no xylose']
    for rep, rep_data in sample_no_xylose.groupby('rep'):
        axs[i, 0].plot(rep_data['Time'], rep_data['Smoothed_OD_Blank_Sub'], 
                       label=f'Rep {rep}', alpha=0.7)
    
    # Highlight exponential phases
    for _, row in sample_expo[sample_expo['Xylose'] == 'no xylose'].iterrows():
        axs[i, 0].axvspan(row['Start_Time'], row['End_Time'], alpha=0.3, color='green')

    axs[i, 0].set_title(f'{sample} - No xylose')
    axs[i, 0].set_ylabel('Smoothed OD (Blank Subtracted)')
    axs[i, 0].legend(loc='upper left')

    # --- Plot with xylose ---
    sample_xylose = sample_data[sample_data['Xylose'] != 'no xylose']
    for rep, rep_data in sample_xylose.groupby('rep'):
        axs[i, 1].plot(rep_data['Time'], rep_data['Smoothed_OD_Blank_Sub'], 
                       label=f'Rep {rep}', alpha=0.7)
    
    for _, row in sample_expo[sample_expo['Xylose'] != 'no xylose'].iterrows():
        axs[i, 1].axvspan(row['Start_Time'], row['End_Time'], alpha=0.3, color='green')

    axs[i, 1].set_title(f'{sample} - With xylose')
    axs[i, 1].set_ylabel('Smoothed OD (Blank Subtracted)')
    axs[i, 1].legend(loc='upper left')

# Set common x-labels
for ax in axs[-1, :]:
    ax.set_xlabel('Time (minutes)')

plt.subplots_adjust(hspace=0.6, wspace=0.3)
plt.show()



In [ ]:
# Add secondary exponential phase data to the final_first_expo_df dataframe
# If no secondary phase is available yet, just use first phase
growth_data_df = final_first_expo_df.copy()
growth_data_df = growth_data_df.drop(columns=['Start_Log2', 'End_Log2'], errors='ignore')
print(growth_data_df.columns)

In [ ]:
# --- Calculate and add growth rates & generation times of the first exponential phase ---
print(growth_data_df.head())
print(df_metrics)

# Growth rate per hour and generation time (min)
growth_data_df['Growth_rate (.h-1)'] = growth_data_df['Slope'] * 60  
growth_data_df['Generation_time (min)'] = 1 / growth_data_df['Slope']

# --- Optional: handle 2nd exponential phase if present ---
# if 'Slope_2nd' in growth_data_df.columns:
#     growth_data_df['Growth_rate (.h-1)_2nd'] = growth_data_df['Slope_2nd'] * 60
#     growth_data_df['Generation_time (min)_2nd'] = 1 / growth_data_df['Slope_2nd']

# --- Merge growth_data_df with df_metrics ---
# Ensure both have 'Xylose' if available

# Merge growth_data_df with df_metrics on Sample, rep, and Xylose


merge_keys = ['Sample', 'rep', 'Xylose']

# Merge all metrics from df_metrics, now including peak summaries
growth_data_df = growth_data_df.merge(
    df_metrics[[
        'Sample', 'rep', 'Xylose',
        # AUC metrics
        'AUC', 'AUC1', 'AUC2',
        # OD metrics
        'Max_OD', 'Max_OD_Time (h)', 'Final_OD',
        'Max_Slope', 'Decline_Rate', 'Var_dODdt','Num_Peaks',
        # Peak summary metrics
        'Peak_Height_Max', 'Peak_Height_Mean',
        'Peak_Prominence_Max', 'Peak_Prominence_Mean',
        'Peak_Left_Base_Max', 'Peak_Left_Base_Mean',
        'Peak_Right_Base_Max', 'Peak_Right_Base_Mean',
        'Peak_Width_Max', 'Peak_Width_Mean',
        'Peak_Width_Height_Max', 'Peak_Width_Height_Mean',
        'Peak_Left_IP_Max', 'Peak_Left_IP_Mean',
        'Peak_Right_IP_Max', 'Peak_Right_IP_Mean'
    ]],
    on=merge_keys,
    how='left'  # keeps all rows from growth_data_df even if df_metrics is missing
)


# --- Reorder columns for clarity ---
columns_order = [
    'Sample', 'rep', 'Xylose',
    'Growth_rate (.h-1)', 'Generation_time (min)',
    'Start_OD', 'Start_Time', 'End_OD', 'End_Time',
    'Slope', 'Intercept', 'R_squared', 'Avg_Residuals',
    'AUC', 'AUC1', 'AUC2',
    'Max_OD', 'Max_OD_Time (h)', 'Final_OD',
    'Max_Slope', 'Decline_Rate', 'Num_Peaks', 'Var_dODdt', 
    # Peak summaries
    'Peak_Height_Max', 'Peak_Height_Mean',
    'Peak_Prominence_Max', 'Peak_Prominence_Mean',
    'Peak_Left_Base_Max', 'Peak_Left_Base_Mean',
    'Peak_Right_Base_Max', 'Peak_Right_Base_Mean',
    'Peak_Width_Max', 'Peak_Width_Mean',
    'Peak_Width_Height_Max', 'Peak_Width_Height_Mean',
    'Peak_Left_IP_Max', 'Peak_Left_IP_Mean',
    'Peak_Right_IP_Max', 'Peak_Right_IP_Mean'
]

# Reorder dataframe
growth_data_df = growth_data_df[columns_order]

# Reorder DataFrame columns
growth_data_df = growth_data_df[columns_order]

#

# --- Display final per-replicate, per-condition growth data ---
print("✅ Final growth data (per replicate and xylose condition):")
print(growth_data_df)


In [ ]:
import pandas as pd

# Ensure correct dtype for grouping
growth_data_df["Xylose"] = growth_data_df["Xylose"].str.strip().str.lower()

# Select numeric columns automatically
numeric_cols = growth_data_df.select_dtypes(include="number").columns

# Group by Sample and Xylose to compute mean and std
grouped = growth_data_df.groupby(["Sample", "Xylose"], as_index=False)

mean_df = grouped[numeric_cols].mean()
std_df = grouped[numeric_cols].std()

# Add identifiers for rep
mean_df["rep"] = "mean"
std_df["rep"] = "sd"

# Merge back the non-numeric columns
# Keep Sample and Xylose as in the group, fill other columns with NaN
mean_df = mean_df.reindex(columns=growth_data_df.columns)
std_df = std_df.reindex(columns=growth_data_df.columns)

# Combine all: original + mean + sd
growth_data_with_avgs = pd.concat([growth_data_df, mean_df, std_df], ignore_index=True)

# Sort for neatness
growth_data_with_avgs = growth_data_with_avgs.sort_values(["Sample", "Xylose", "rep"]).reset_index(drop=True)

print(growth_data_with_avgs)


In [ ]:
# --- Extract Gene target from the Sample column ---
def extract_gene_target(sample_str):
    match = re.search(r"Gene target:\s*([\w\-]+)", str(sample_str))
    return match.group(1) if match else sample_str

growth_data_with_avgs["Gene_target"] = growth_data_with_avgs["Sample"].apply(extract_gene_target)
print(growth_data_df)

In [ ]:
# Load the Excel file into a DataFrame
file_path_meta = 'C:/Users/arnou/Documents/thesis/Resultaten/MetaData/growth_category2.xlsx'
growth_category = pd.read_excel(file_path_meta)

# Merge the two DataFrames on the 'gene target X' column
merged_df3 = pd.merge(
    growth_data_with_avgs,
    growth_category,
    on='Gene_target',
    how='left'  # keep all rows from growth_data_with_avg
)
print(merged_df3)
# Fill missing growth profile entries with 'Not present'
merged_df3['Growth_category'] = merged_df3['Growth_category'].fillna('Not present')
growth_data_with_avgs = merged_df3
# Optional: inspect the result
print(growth_data_with_avgs.head())

In [ ]:
#excel maken

In [ ]:
import pandas as pd
import numpy as np
import os
import re

# --- 1. SETTINGS & PATHS ---
# Define where you want to save your files
save_directory = r'E:\Thesis3april\GrowthResults' 

if not os.path.exists(save_directory):
    os.makedirs(save_directory)

# --- 2. PREPARE DATA ---
# Ensure consistent naming for xylose conditions
growth_data_with_avgs["Xylose"] = growth_data_with_avgs["Xylose"].astype(str).str.strip().str.lower()

# Extract the Mean and SD dataframes as independent copies
mean_df = growth_data_with_avgs[growth_data_with_avgs["rep"] == "mean"].copy()
sd_df = growth_data_with_avgs[growth_data_with_avgs["rep"] == "sd"].copy()

# --- 3. CALCULATE DERIVED METRICS (Ratios & Logs) ---

# A. AUC1 / AUC2 Ratio (Internal to each sample)
mean_df["AUC1/AUC2"] = mean_df["AUC1"] / mean_df["AUC2"]
mean_df["log(AUC1/AUC2)"] = np.log(mean_df["AUC1/AUC2"])

# Propagate SD for AUC1/AUC2
sd_df["AUC1/AUC2"] = mean_df["AUC1/AUC2"] * np.sqrt(
    (sd_df["AUC1"] / mean_df["AUC1"]).fillna(0)**2 + 
    (sd_df["AUC2"] / mean_df["AUC2"]).fillna(0)**2
)
# Propagate SD for the log: SD_log = SD_ratio / Mean_ratio
sd_df["log(AUC1/AUC2)"] = (sd_df["AUC1/AUC2"] / mean_df["AUC1/AUC2"]).fillna(0)

# B. AUC Ratio (Xylose / No Xylose)
# Align conditions to compare the same sample
m_xy = mean_df[mean_df["Xylose"] == "xylose"].set_index("Sample")
m_no = mean_df[mean_df["Xylose"] == "no xylose"].set_index("Sample")
s_xy = sd_df[sd_df["Xylose"] == "xylose"].set_index("Sample")
s_no = sd_df[sd_df["Xylose"] == "no xylose"].set_index("Sample")

common = m_xy.index.intersection(m_no.index)

ratio_val = m_xy.loc[common, "AUC"] / m_no.loc[common, "AUC"]
ratio_sd_val = ratio_val * np.sqrt(
    (s_xy.loc[common, "AUC"] / m_xy.loc[common, "AUC"]).fillna(0)**2 +
    (s_no.loc[common, "AUC"] / m_no.loc[common, "AUC"]).fillna(0)**2
)

# Create small dataframes for the Xylose/NoXylose ratio results
xy_ratio_means = pd.DataFrame({"Sample": common, "AUC_Ratio_Xyl_NoXyl": ratio_val.values})
xy_ratio_sds = pd.DataFrame({"Sample": common, "AUC_Ratio_Xyl_NoXyl": ratio_sd_val.values})

# --- 4. DEFINE COLUMNS & SAVE ---

export_cols = [
    'Sample', 'Gene_target', 'Growth_rate (.h-1)', 'AUC', 
    'AUC1', 'AUC2', 'AUC1/AUC2', 'log(AUC1/AUC2)', 
    'Max_OD', 'Max_OD_Time (h)', 'Final_OD', 'Peak_Prominence_Max'
]

# Ensure we only use columns that exist
final_cols = [c for c in export_cols if c in mean_df.columns]

def save_final_csv(df, condition, stat_type, ratio_df=None):
    # Filter by condition
    out_df = df[df["Xylose"] == condition][final_cols].copy()
    
    # Merge the Xyl/NoXyl ratio if applicable
    if ratio_df is not None:
        out_df = out_df.merge(ratio_df, on="Sample", how="left")
    
    filename = f"{condition.capitalize().replace(' ', '_')}_{stat_type}.csv"
    path = os.path.join(save_directory, filename)
    out_df.to_csv(path, index=False)
    print(f"✅ Saved: {filename}")

# Generate the 4 files
save_final_csv(mean_df, "xylose", "Means", xy_ratio_means)
save_final_csv(sd_df, "xylose", "Standard_Deviations", xy_ratio_sds)
save_final_csv(mean_df, "no xylose", "Means")
save_final_csv(sd_df, "no xylose", "Standard_Deviations")

print(f"\nAll files are located in: {save_directory}")

In [ ]:
#ecel statitics

In [ ]:
#raw replicates excel

In [ ]:
import pandas as pd
import os

# --- 1. Filter for Raw Replicates only ---
# We exclude the 'mean' and 'sd' rows to keep only the individual measurements
raw_reps_df = growth_data_with_avgs[~growth_data_with_avgs["rep"].isin(["mean", "sd"])].copy()

# --- 2. Clean up the columns for export ---
# We ensure the columns are in a logical order for manual inspection
raw_export_cols = [
    'Sample', 'Gene_target', 'Xylose', 'rep', 
    'Growth_rate (.h-1)', 'AUC', 'AUC1', 'AUC2', 
    'Max_OD', 'Max_OD_Time (h)', 'Final_OD', 'Peak_Prominence_Max'
]

# Only keep columns that actually exist in your dataframe
final_raw_cols = [c for c in raw_export_cols if c in raw_reps_df.columns]
raw_reps_export = raw_reps_df[final_raw_cols]

# --- 3. Save to CSV ---
raw_filename = "Growth_Metrics_Raw_Replicates.csv"
raw_path = os.path.join(save_directory, raw_filename)

raw_reps_export.to_csv(raw_path, index=False)

print(f"✅ Raw replicates saved to: {raw_path}")
print(f"📍 Total rows exported: {len(raw_reps_export)}")

In [ ]:
#extra code om excel te genereren 4 april

In [ ]:
#4april

In [ ]:
#######

In [ ]:
#color overlay met de rood zwart bovenaan

In [ ]:
!pip install plotly

In [ ]:
#excels maken

In [ ]:
#finale pca overlay (xyl ratio niet meegomen want is hetzelde als auc)

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import matplotlib.pyplot as plt
# import plotly.express as px  # Uncomment this if you install plotly
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform

# --- 1. FONT & STYLE CONFIGURATION ---
FONT_FAMILY = 'sans-serif' 

# Font Sizes
FONT_AXIS_LABEL = {'family': FONT_FAMILY, 'size': 16, 'weight': 'normal'}
FONT_TICKS = 12           
FONT_PLOT_LABELS = {'family': FONT_FAMILY, 'size': 10, 'alpha': 0.8}
FONT_LEGEND = {'family': FONT_FAMILY, 'size': 16}
FONT_CBAR_LABEL = {'family': FONT_FAMILY, 'size': 16}

# Thickness / Width Settings
AXIS_LINE_WIDTH = 1.5      
TICK_WIDTH = 1.5           
TICK_LENGTH = 5            

plt.rcParams['font.family'] = FONT_FAMILY

# --- 2. Load Data ---
save_directory = r'E:\Thesis3april\GrowthResults' 
df_xyl = pd.read_csv(os.path.join(save_directory, "Xylose_Means.csv"))

# --- 3. Prepare PCA Data ---
metrics_to_use = [
    "AUC", "log(AUC1/AUC2)", 
    "Max_OD", "Max_OD_Time (h)", "Final_OD", "Peak_Prominence_Max"
]

pca_df_clean = df_xyl.copy()
pca_df_clean[metrics_to_use] = pca_df_clean[metrics_to_use].fillna(0)
pca_df_clean.set_index("Sample", inplace=True)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_df_clean[metrics_to_use])
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_result_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"], index=pca_df_clean.index)
pca_result_df["Gene_target"] = pca_df_clean["Gene_target"]

# --- 4. Labeling Logic ---
min_distance = 0.8
max_cluster_size = 5
text_offset = 0.05
positions = pca_result_df[["PC1", "PC2"]].to_numpy()
labels = pca_result_df["Gene_target"].to_numpy()
dist_matrix = squareform(pdist(positions))
cluster_sizes = np.sum(dist_matrix < min_distance, axis=1) - 1

def apply_thesis_style(ax, pca_obj):
    ax.spines['bottom'].set_linewidth(AXIS_LINE_WIDTH)
    ax.spines['left'].set_linewidth(AXIS_LINE_WIDTH)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', which='major', labelsize=FONT_TICKS, width=TICK_WIDTH, length=TICK_LENGTH)
    ax.set_xlabel(f"PC1 ({pca_obj.explained_variance_ratio_[0]*100:.1f}% variance)", fontdict=FONT_AXIS_LABEL)
    ax.set_ylabel(f"PC2 ({pca_obj.explained_variance_ratio_[1]*100:.1f}% variance)", fontdict=FONT_AXIS_LABEL)
    ax.set_box_aspect(1)

# =========================================================
# PLOT 1: CATEGORICAL (Black Mutants / Red Control)
# =========================================================
plt.figure(figsize=(9, 7))
ax1 = plt.gca()

non_control = pca_result_df[pca_result_df["Gene_target"].str.lower() != "no_sgrna"]
plt.scatter(non_control["PC1"], non_control["PC2"], color='black', label='Mutant', s=45, alpha=0.8, zorder=1, edgecolors='none')

control = pca_result_df[pca_result_df["Gene_target"].str.lower() == "no_sgrna"]
if not control.empty:
    plt.scatter(control["PC1"], control["PC2"], color='red', label='no-sgRNA', s=45, alpha=0.9, zorder=2, edgecolors='none')

for i, (x, y) in enumerate(positions):
    if labels[i].lower() == "no_sgrna": continue
    if cluster_sizes[i] <= max_cluster_size:
        plt.text(x + text_offset, y + text_offset, labels[i], fontdict=FONT_PLOT_LABELS, zorder=3)

apply_thesis_style(ax1, pca)
plt.legend(bbox_to_anchor=(1, 1), loc="upper left", frameon=False, prop=FONT_LEGEND)
plt.tight_layout()
plt.savefig(os.path.join(save_directory, "PCA_Main_Categorical.svg"), format="svg")
plt.savefig(os.path.join(save_directory, "PCA_Main_Categorical.png"), dpi=500)
plt.show()

# =========================================================
# PLOTS 2-8: VIRIDIS GRADIENT OVERLAYS (Loop)
# =========================================================
for metric in metrics_to_use:
    plt.figure(figsize=(10, 7))
    ax_m = plt.gca()

    vals = pca_df_clean.loc[pca_result_df.index, metric]
    c_min, c_max = vals.min(), vals.quantile(0.99)

    scatter = plt.scatter(
        pca_result_df["PC1"], pca_result_df["PC2"],
        c=vals, cmap="viridis", vmin=c_min, vmax=c_max,
        s=45, alpha=0.9, zorder=2, edgecolors='none' # Removed black border
    )

    cbar = plt.colorbar(scatter, fraction=0.046, pad=0.04)
    cbar.set_label(metric, fontdict=FONT_CBAR_LABEL)
    cbar.ax.tick_params(labelsize=FONT_TICKS)
    cbar.outline.set_visible(False)

    for i, (x, y) in enumerate(positions):
        if labels[i].lower() == "no_sgrna": continue
        if cluster_sizes[i] <= max_cluster_size:
            plt.text(x + text_offset, y + text_offset, labels[i], fontdict=FONT_PLOT_LABELS, zorder=3)

    apply_thesis_style(ax_m, pca)
    plt.title(f"PCA Gradient: {metric}", pad=20, fontdict=FONT_AXIS_LABEL)
    plt.tight_layout()
    
    clean_name = metric.replace('/', '_').replace(' ', '_').replace('(', '').replace(')', '')
    plt.savefig(os.path.join(save_directory, f"PCA_Overlay_{clean_name}.svg"), format="svg")
    plt.savefig(os.path.join(save_directory, f"PCA_Overlay_{clean_name}.png"), dpi=300)
    plt.show()

    # --- Interactive Plotly Export (Optional) ---
    # fig_html = px.scatter(pca_result_df.reset_index(), x="PC1", y="PC2", color=vals.values,
    #                       hover_data=["Sample", "Gene_target"], range_color=[c_min, c_max],
    #                       color_continuous_scale="Viridis", labels={'color': metric}, template="simple_white")
    # fig_html.update_yaxes(scaleanchor="x", scaleratio=1)
    # fig_html.write_html(os.path.join(save_directory, f"PCA_Interactive_{clean_name}.html"))

# Final Print Stats
print("-" * 30)
print(f"✅ Processing complete.")
print(f"📍 Total dots on plot: {len(pca_result_df)}")
print(f"🔴 Number of no_sgRNA (control) dots: {len(control)}")
print("-" * 30)

In [ ]:
#finale overlay zonder genen erop

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform

# --- 1. FONT & STYLE CONFIGURATION ---
FONT_FAMILY = 'sans-serif' 

# Font Sizes
FONT_AXIS_LABEL = {'family': FONT_FAMILY, 'size': 16, 'weight': 'normal'}
FONT_TICKS = 12           
FONT_PLOT_LABELS = {'family': FONT_FAMILY, 'size': 10, 'alpha': 0}
FONT_LEGEND = {'family': FONT_FAMILY, 'size': 16}
FONT_CBAR_LABEL = {'family': FONT_FAMILY, 'size': 16}

# Thickness / Width Settings
AXIS_LINE_WIDTH = 1.5      
TICK_WIDTH = 1.5           
TICK_LENGTH = 5            

plt.rcParams['font.family'] = FONT_FAMILY

# --- 2. CUSTOM LABEL MAPPING ---
# Metrics not in this dictionary will default to their column name
metric_display_names = {
    "Peak_Prominence_Max": "Max Peak Prominence",
    "Final_OD": "Final OD",
    "Max_OD_Time (h)": "Max OD Time (h)",
    "Max_OD": "Max OD"
}

# --- 3. Load Data ---
save_directory = r'E:\Thesis3april\GrowthResults' 
df_xyl = pd.read_csv(os.path.join(save_directory, "Xylose_Means.csv"))

# --- 4. Prepare PCA Data ---
metrics_to_use = [
    "AUC", "log(AUC1/AUC2)", 
    "Max_OD", "Max_OD_Time (h)", "Final_OD", "Peak_Prominence_Max"
]

pca_df_clean = df_xyl.copy()
pca_df_clean[metrics_to_use] = pca_df_clean[metrics_to_use].fillna(0)
pca_df_clean.set_index("Sample", inplace=True)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_df_clean[metrics_to_use])
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_result_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"], index=pca_df_clean.index)
pca_result_df["Gene_target"] = pca_df_clean["Gene_target"]

# --- 5. Labeling Logic ---
min_distance = 0.8
max_cluster_size = 5
text_offset = 0.05
positions = pca_result_df[["PC1", "PC2"]].to_numpy()
labels = pca_result_df["Gene_target"].to_numpy()
dist_matrix = squareform(pdist(positions))
cluster_sizes = np.sum(dist_matrix < min_distance, axis=1) - 1

def apply_thesis_style(ax, pca_obj):
    ax.spines['bottom'].set_linewidth(AXIS_LINE_WIDTH)
    ax.spines['left'].set_linewidth(AXIS_LINE_WIDTH)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', which='major', labelsize=FONT_TICKS, width=TICK_WIDTH, length=TICK_LENGTH)
    ax.set_xlabel(f"PC1 ({pca_obj.explained_variance_ratio_[0]*100:.1f}% variance)", fontdict=FONT_AXIS_LABEL)
    ax.set_ylabel(f"PC2 ({pca_obj.explained_variance_ratio_[1]*100:.1f}% variance)", fontdict=FONT_AXIS_LABEL)
    ax.set_box_aspect(1) # Ensures the plotted region is a square

# =========================================================
# PLOT 1: CATEGORICAL (Black Mutants / Red Control)
# =========================================================
plt.figure(figsize=(9, 7))
ax1 = plt.gca()

non_control = pca_result_df[pca_result_df["Gene_target"].str.lower() != "no_sgrna"]
plt.scatter(non_control["PC1"], non_control["PC2"], color='black', label='Mutant', s=45, alpha=0.8, zorder=1, edgecolors='none')

control = pca_result_df[pca_result_df["Gene_target"].str.lower() == "no_sgrna"]
if not control.empty:
    plt.scatter(control["PC1"], control["PC2"], color='red', label='no-sgRNA', s=45, alpha=0.9, zorder=2, edgecolors='none')

for i, (x, y) in enumerate(positions):
    if labels[i].lower() == "no_sgrna": continue
    if cluster_sizes[i] <= max_cluster_size:
        plt.text(x + text_offset, y + text_offset, labels[i], fontdict=FONT_PLOT_LABELS, zorder=3)

apply_thesis_style(ax1, pca)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", frameon=False, prop=FONT_LEGEND)
plt.tight_layout()
plt.savefig(os.path.join(save_directory, "PCA_Main_Categorical.svg"), format="svg")
plt.savefig(os.path.join(save_directory, "PCA_Main_Categorical.png"), dpi=500)
plt.show()

# =========================================================
# PLOTS 2-8: VIRIDIS GRADIENT OVERLAYS (Loop)
# =========================================================
for metric in metrics_to_use:
    plt.figure(figsize=(10, 7))
    ax_m = plt.gca()

    # Determine display name (uses dict if exists, otherwise defaults to metric name)
    display_name = metric_display_names.get(metric, metric)

    vals = pca_df_clean.loc[pca_result_df.index, metric]
    c_min, c_max = vals.min(), vals.quantile(0.99)

    scatter = plt.scatter(
        pca_result_df["PC1"], pca_result_df["PC2"],
        c=vals, cmap="viridis", vmin=c_min, vmax=c_max,
        s=45, alpha=0.9, zorder=2, edgecolors='none'
    )

    # Colorbar configuration
    cbar = plt.colorbar(scatter, fraction=0.046, pad=0.04)
    cbar.set_label(display_name, fontdict=FONT_CBAR_LABEL)
    cbar.ax.tick_params(labelsize=FONT_TICKS)
    cbar.outline.set_visible(False)

    for i, (x, y) in enumerate(positions):
        if labels[i].lower() == "no_sgrna": continue
        if cluster_sizes[i] <= max_cluster_size:
            plt.text(x + text_offset, y + text_offset, labels[i], fontdict=FONT_PLOT_LABELS, zorder=3)

    apply_thesis_style(ax_m, pca)
    plt.title(f"PCA Gradient: {display_name}", pad=20, fontdict=FONT_AXIS_LABEL)
    plt.tight_layout()
    
    clean_name = metric.replace('/', '_').replace(' ', '_').replace('(', '').replace(')', '')
    plt.savefig(os.path.join(save_directory, f"nonamePCA_Overlay_{clean_name}.svg"), format="svg")
    plt.savefig(os.path.join(save_directory, f"nonamePCA_Overlay_{clean_name}.png"), dpi=300)
    plt.show()

print("-" * 30)
print(f"✅ Processing complete.")
print(f"📍 Total dots on plot: {len(pca_result_df)}")
print(f"🔴 Number of no_sgRNA (control) dots: {len(control)}")
print("-" * 30)

In [ ]:
#pathway overlay in same format 7 april: finaal

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform

# --- 1. DIRECTORY CONFIGURATION ---
save_directory = r'E:\Thesis3april\GrowthResults' 
OUTPUT_DIR = os.path.join(save_directory, "7april", "Finaal")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# --- 2. FONT & STYLE CONFIGURATION ---
FONT_FAMILY = 'sans-serif' 
FONT_AXIS_LABEL = {'family': FONT_FAMILY, 'size': 18, 'weight': 'normal'}
FONT_TICKS = 18            
FONT_PLOT_LABELS = {'family': FONT_FAMILY, 'size': 9, 'alpha': 0} #als je geen 
FONT_LEGEND = {'family': FONT_FAMILY, 'size': 16} 
AXIS_LINE_WIDTH = 1.5      
TICK_WIDTH = 1.5           
TICK_LENGTH = 5            

# --- DOT SIZE SETTINGS ---
DOT_SIZE = 60              # Size of dots on the actual plot
LEGEND_MARKER_SCALE = 1.3   # Scale factor for dots in the legend

plt.rcParams['font.family'] = FONT_FAMILY

# --- 3. PATHWAY & COLOR SETTINGS ---
annotation_path = r'E:\Thesis3april\overlay\Pathway_annotation.xlsx'

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c", "cell shape": "#d62728", 
    "biosynthesis of fatty acids": "#9467bd", "DNA replication": "#8c564b", 
    "DNA condensation/ segregation": "#e377c2", "biosynthesis of isoprenoids": "#7f7f7f", 
    "cell division": "#bcbd22", "ribosomal proteins": "#17becf", 
    "biosynthesis of iron-sulfur clusters": "#aec7e8", "glycolysis": "#ffbb78", 
    "biosynthesis of menaquinone": "#98df8a"
}
OTHER_LABEL, OTHER_LEGEND_COLOR = "Unknown/Other", "#D3D3D3"
CONTROL_LABEL, CONTROL_COLOR = "no-sgRNA", "#000000"

# --- 4. DATA PROCESSING ---
df_xyl = pd.read_csv(os.path.join(save_directory, "Xylose_Means.csv"))
df_ann = pd.read_excel(annotation_path)

metrics_to_use = ["AUC", "log(AUC1/AUC2)", "Max_OD", "Max_OD_Time (h)", "Final_OD", "Peak_Prominence_Max"]
pca_df_clean = df_xyl.copy()
pca_df_clean[metrics_to_use] = pca_df_clean[metrics_to_use].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_df_clean[metrics_to_use])
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_result_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"], index=pca_df_clean.index)
pca_result_df["Gene_target"] = pca_df_clean["Gene_target"].astype(str).str.strip()

def get_pathway(row):
    for col in ['SubtiWiki Annotation 4', 'SubtiWiki Annotation 3']:
        val = str(row.get(col, '')).strip()
        if val not in ['nan', 'NA', 'None', '']: return val
    return OTHER_LABEL

df_ann['Pathway_Temp'] = df_ann.apply(get_pathway, axis=1)
pathway_map = dict(zip(df_ann['Treatment'].astype(str).str.strip(), df_ann['Pathway_Temp']))

def assign_annotation(gene_name):
    if gene_name.lower() in ['no_sgrna', 'no-sgrna']: return CONTROL_LABEL
    return pathway_map.get(gene_name, OTHER_LABEL)

pca_result_df['Annotation'] = pca_result_df['Gene_target'].apply(assign_annotation)

unique_cats = sorted(pca_result_df['Annotation'].unique())
color_lookup = {**MANUAL_COLORS, CONTROL_LABEL: CONTROL_COLOR, OTHER_LABEL: OTHER_LEGEND_COLOR}
auto_cmap = plt.get_cmap('tab20b')
extra_cats = [c for c in unique_cats if c not in color_lookup]
for i, cat in enumerate(extra_cats):
    color_lookup[cat] = mcolors.to_hex(auto_cmap(i % 20))

min_distance, max_cluster_size, text_offset = 0.8, 5, 0.05
positions = pca_result_df[["PC1", "PC2"]].to_numpy()
dist_matrix = squareform(pdist(positions))
cluster_sizes = np.sum(dist_matrix < min_distance, axis=1) - 1

# --- 5. PLOTTING ---
def apply_thesis_style(ax, pca_obj):
    ax.spines['bottom'].set_linewidth(AXIS_LINE_WIDTH)
    ax.spines['left'].set_linewidth(AXIS_LINE_WIDTH)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', which='major', labelsize=FONT_TICKS, width=TICK_WIDTH, length=TICK_LENGTH)
    ax.set_xlabel(f"PC1 ({pca_obj.explained_variance_ratio_[0]*100:.1f}% variance)", fontdict=FONT_AXIS_LABEL)
    ax.set_ylabel(f"PC2 ({pca_obj.explained_variance_ratio_[1]*100:.1f}% variance)", fontdict=FONT_AXIS_LABEL)
    ax.set_box_aspect(1) # Keeps the plotted region a square

plt.figure(figsize=(12, 8))
ax = plt.gca()

# To get no-sgRNA at the top, we sort the plotting order
# We put CONTROL_LABEL first, then sorted manual colors, then others
priority_cats = [CONTROL_LABEL] + [c for c in sorted(unique_cats) if c != CONTROL_LABEL]

for cat in priority_cats:
    sub = pca_result_df[pca_result_df['Annotation'] == cat]
    
    # Check if category should be in legend
    if cat == CONTROL_LABEL or cat in MANUAL_COLORS:
        lbl = cat
    else:
        lbl = "_nolegend_"
    
    plt.scatter(
        sub["PC1"], sub["PC2"], 
        label=lbl, 
        color=color_lookup.get(cat, OTHER_LEGEND_COLOR),
        s=DOT_SIZE, 
        alpha=0.85, 
        zorder=10 if cat == CONTROL_LABEL else 3, # Control on top visually
        edgecolors='none'
    )

# Gene labels
for i, row in pca_result_df.reset_index().iterrows():
    if row["Annotation"] == CONTROL_LABEL: continue
    if cluster_sizes[i] <= max_cluster_size:
        plt.text(row["PC1"] + text_offset, row["PC2"] + text_offset, 
                 row["Gene_target"], fontdict=FONT_PLOT_LABELS, zorder=10)

apply_thesis_style(ax, pca)

# Clean up legend handles to ensure order and uniqueness
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))

plt.legend(by_label.values(), by_label.keys(), 
           bbox_to_anchor=(1.05, 1), loc="upper left", 
           frameon=False, prop=FONT_LEGEND, 
           markerscale=LEGEND_MARKER_SCALE)

plt.tight_layout()

# Save
plt.savefig(os.path.join(OUTPUT_DIR, "PCA_Pathway_Overlay.png"), dpi=500, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, "PCA_Pathway_Overlay.svg"), format="svg", bbox_inches='tight')
plt.show()

print(f"Success! Plot saved to: {OUTPUT_DIR}")

In [ ]:
#12 april other colors

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform

# --- 1. DIRECTORY CONFIGURATION ---
save_directory = r'E:\Thesis3april\GrowthResults' 
OUTPUT_DIR = os.path.join(save_directory, "12april", "Finaal")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# --- 2. FONT & STYLE CONFIGURATION ---
FONT_FAMILY = 'sans-serif' 
FONT_AXIS_LABEL = {'family': FONT_FAMILY, 'size': 18, 'weight': 'normal'}
FONT_TICKS = 18            
FONT_PLOT_LABELS = {'family': FONT_FAMILY, 'size': 9, 'alpha': 0} 
FONT_LEGEND = {'family': FONT_FAMILY, 'size': 16} 
AXIS_LINE_WIDTH = 1.5      
TICK_WIDTH = 1.5           
TICK_LENGTH = 5            

# --- DOT SIZE SETTINGS ---
DOT_SIZE = 60              # Size of dots on the actual plot
LEGEND_MARKER_SCALE = 1.3   # Scale factor for dots in the legend

plt.rcParams['font.family'] = FONT_FAMILY

# --- 3. PATHWAY & COLOR SETTINGS ---
annotation_path = r'E:\Thesis3april\overlay\Pathway_annotation.xlsx'

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c", "cell shape": "#d62728", 
    "biosynthesis of fatty acids": "#9467bd", "DNA replication": "#8c564b", 
    "DNA condensation/ segregation": "#e377c2", 
    "cell division": "#bcbd22", "ribosomal proteins": "#17becf", 
    "biosynthesis of iron-sulfur clusters": "#aec7e8", "glycolysis": "#ffbb78", 
    "biosynthesis of menaquinone": "#98df8a"
}
OTHER_LABEL, OTHER_LEGEND_COLOR = "Unknown/Other", "#736F6F"
CONTROL_LABEL, CONTROL_COLOR = "no-sgRNA", "#000000"

# --- 4. DATA PROCESSING ---
df_xyl = pd.read_csv(os.path.join(save_directory, "Xylose_Means.csv"))
df_ann = pd.read_excel(annotation_path)

metrics_to_use = ["AUC", "log(AUC1/AUC2)", "Max_OD", "Max_OD_Time (h)", "Final_OD", "Peak_Prominence_Max"]
pca_df_clean = df_xyl.copy()
pca_df_clean[metrics_to_use] = pca_df_clean[metrics_to_use].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_df_clean[metrics_to_use])
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_result_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"], index=pca_df_clean.index)
pca_result_df["Gene_target"] = pca_df_clean["Gene_target"].astype(str).str.strip()

def get_pathway(row):
    for col in ['SubtiWiki Annotation 4', 'SubtiWiki Annotation 3']:
        val = str(row.get(col, '')).strip()
        if val not in ['nan', 'NA', 'None', '']: return val
    return OTHER_LABEL

df_ann['Pathway_Temp'] = df_ann.apply(get_pathway, axis=1)
pathway_map = dict(zip(df_ann['Treatment'].astype(str).str.strip(), df_ann['Pathway_Temp']))

def assign_annotation(gene_name):
    if gene_name.lower() in ['no_sgrna', 'no-sgrna']: return CONTROL_LABEL
    return pathway_map.get(gene_name, OTHER_LABEL)

pca_result_df['Annotation'] = pca_result_df['Gene_target'].apply(assign_annotation)

unique_cats = sorted(pca_result_df['Annotation'].unique())

color_lookup = {**MANUAL_COLORS, CONTROL_LABEL: CONTROL_COLOR}
for cat in unique_cats:
    if cat not in color_lookup:
        color_lookup[cat] = OTHER_LEGEND_COLOR

min_distance, max_cluster_size, text_offset = 0.8, 5, 0.05
positions = pca_result_df[["PC1", "PC2"]].to_numpy()
dist_matrix = squareform(pdist(positions))
cluster_sizes = np.sum(dist_matrix < min_distance, axis=1) - 1

# --- 5. PLOTTING ---
def apply_thesis_style(ax, pca_obj):
    ax.spines['bottom'].set_linewidth(AXIS_LINE_WIDTH)
    ax.spines['left'].set_linewidth(AXIS_LINE_WIDTH)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', which='major', labelsize=FONT_TICKS, width=TICK_WIDTH, length=TICK_LENGTH)
    ax.set_xlabel(f"PC1 ({pca_obj.explained_variance_ratio_[0]*100:.1f}% variance)", fontdict=FONT_AXIS_LABEL)
    ax.set_ylabel(f"PC2 ({pca_obj.explained_variance_ratio_[1]*100:.1f}% variance)", fontdict=FONT_AXIS_LABEL)
    ax.set_box_aspect(1) 

plt.figure(figsize=(12, 8))
ax = plt.gca()

priority_cats = [CONTROL_LABEL] + [c for c in sorted(unique_cats) if c != CONTROL_LABEL]

# Keep track if we actually have "other" points to show in legend
has_others = False

for cat in priority_cats:
    sub = pca_result_df[pca_result_df['Annotation'] == cat]
    
    if cat == CONTROL_LABEL or cat in MANUAL_COLORS:
        lbl = cat
    else:
        lbl = "_nolegend_"
        has_others = True # Mark that we have points mapped to grey
    
    plt.scatter(
        sub["PC1"], sub["PC2"], 
        label=lbl, 
        color=color_lookup.get(cat, OTHER_LEGEND_COLOR),
        s=DOT_SIZE, 
        alpha=0.85, 
        zorder=10 if cat == CONTROL_LABEL else 3, 
        edgecolors='none'
    )

# Add a single "Other categories" entry at the end if needed
if has_others:
    plt.scatter([], [], color=OTHER_LEGEND_COLOR, label="other categories", s=DOT_SIZE, alpha=0.85, edgecolors='none')

for i, row in pca_result_df.reset_index().iterrows():
    if row["Annotation"] == CONTROL_LABEL: continue
    if cluster_sizes[i] <= max_cluster_size:
        plt.text(row["PC1"] + text_offset, row["PC2"] + text_offset, 
                 row["Gene_target"], fontdict=FONT_PLOT_LABELS, zorder=10)

apply_thesis_style(ax, pca)

# Unique handles and labels while preserving order
handles, labels = ax.get_legend_handles_labels()
by_label = {}
for h, l in zip(handles, labels):
    if l not in by_label:
        by_label[l] = h

plt.legend(by_label.values(), by_label.keys(), 
           bbox_to_anchor=(1.05, 1), loc="upper left", 
           frameon=False, prop=FONT_LEGEND, 
           markerscale=LEGEND_MARKER_SCALE)

plt.tight_layout()

plt.savefig(os.path.join(OUTPUT_DIR, "PCA_Pathway_Overlay.png"), dpi=500, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, "PCA_Pathway_Overlay.svg"), format="svg", bbox_inches='tight')
plt.show()

print(f"Success! Plot saved to: {OUTPUT_DIR}")

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform

# --- 1. DIRECTORY CONFIGURATION ---
save_directory = r'E:\Thesis3april\GrowthResults' 
OUTPUT_DIR = os.path.join(save_directory, "12april", "Finaal")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# --- 2. FONT & STYLE CONFIGURATION ---
FONT_FAMILY = 'sans-serif' 
FONT_AXIS_LABEL = {'family': FONT_FAMILY, 'size': 18, 'weight': 'normal'}
FONT_TICKS = 18            
FONT_PLOT_LABELS = {'family': FONT_FAMILY, 'size': 9, 'alpha': 0} 
FONT_LEGEND = {'family': FONT_FAMILY, 'size': 16} 
AXIS_LINE_WIDTH = 1.5      
TICK_WIDTH = 1.5           
TICK_LENGTH = 5            

# --- DOT SIZE SETTINGS ---
DOT_SIZE = 60              # Size of dots on the actual plot
LEGEND_MARKER_SCALE = 1.3   # Scale factor for dots in the legend

plt.rcParams['font.family'] = FONT_FAMILY

# --- 3. PATHWAY & COLOR SETTINGS ---
annotation_path = r'E:\Thesis3april\overlay\Pathway_annotation.xlsx'

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c", "cell shape": "#d62728", 
    "biosynthesis of fatty acids": "#9467bd", "DNA replication": "#8c564b", 
    "DNA condensation/ segregation": "#e377c2", 
    "cell division": "#bcbd22", "ribosomal proteins": "#17becf", 
    "biosynthesis of iron-sulfur clusters": "#aec7e8", "glycolysis": "#ffbb78", 
    "biosynthesis of menaquinone": "#98df8a"
}
OTHER_LABEL, OTHER_LEGEND_COLOR = "Unknown/Other", "#736F6F"
CONTROL_LABEL, CONTROL_COLOR = "no-sgRNA", "#000000"

# --- 4. DATA PROCESSING ---
df_xyl = pd.read_csv(os.path.join(save_directory, "Xylose_Means.csv"))
df_ann = pd.read_excel(annotation_path)

metrics_to_use = ["AUC", "log(AUC1/AUC2)", "Max_OD", "Max_OD_Time (h)", "Final_OD", "Peak_Prominence_Max"]
pca_df_clean = df_xyl.copy()
pca_df_clean[metrics_to_use] = pca_df_clean[metrics_to_use].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_df_clean[metrics_to_use])
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_result_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"], index=pca_df_clean.index)
pca_result_df["Gene_target"] = pca_df_clean["Gene_target"].astype(str).str.strip()

def get_pathway(row):
    for col in ['SubtiWiki Annotation 4', 'SubtiWiki Annotation 3']:
        val = str(row.get(col, '')).strip()
        if val not in ['nan', 'NA', 'None', '']: return val
    return OTHER_LABEL

df_ann['Pathway_Temp'] = df_ann.apply(get_pathway, axis=1)
pathway_map = dict(zip(df_ann['Treatment'].astype(str).str.strip(), df_ann['Pathway_Temp']))

def assign_annotation(gene_name):
    if gene_name.lower() in ['no_sgrna', 'no-sgrna']: return CONTROL_LABEL
    return pathway_map.get(gene_name, OTHER_LABEL)

pca_result_df['Annotation'] = pca_result_df['Gene_target'].apply(assign_annotation)

unique_cats = sorted(pca_result_df['Annotation'].unique())

# MODIFIED COLOR LOGIC: Map anything not in MANUAL_COLORS or CONTROL_LABEL to grey
color_lookup = {**MANUAL_COLORS, CONTROL_LABEL: CONTROL_COLOR}
for cat in unique_cats:
    if cat not in color_lookup:
        color_lookup[cat] = OTHER_LEGEND_COLOR

min_distance, max_cluster_size, text_offset = 0.8, 5, 0.05
positions = pca_result_df[["PC1", "PC2"]].to_numpy()
dist_matrix = squareform(pdist(positions))
cluster_sizes = np.sum(dist_matrix < min_distance, axis=1) - 1

# --- 5. PLOTTING ---
def apply_thesis_style(ax, pca_obj):
    ax.spines['bottom'].set_linewidth(AXIS_LINE_WIDTH)
    ax.spines['left'].set_linewidth(AXIS_LINE_WIDTH)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', which='major', labelsize=FONT_TICKS, width=TICK_WIDTH, length=TICK_LENGTH)
    ax.set_xlabel(f"PC1 ({pca_obj.explained_variance_ratio_[0]*100:.1f}% variance)", fontdict=FONT_AXIS_LABEL)
    ax.set_ylabel(f"PC2 ({pca_obj.explained_variance_ratio_[1]*100:.1f}% variance)", fontdict=FONT_AXIS_LABEL)
    ax.set_box_aspect(1) 

plt.figure(figsize=(12, 8))
ax = plt.gca()

priority_cats = [CONTROL_LABEL] + [c for c in sorted(unique_cats) if c != CONTROL_LABEL]

for cat in priority_cats:
    sub = pca_result_df[pca_result_df['Annotation'] == cat]
    
    if cat == CONTROL_LABEL or cat in MANUAL_COLORS:
        lbl = cat
    else:
        lbl = "_nolegend_"
    
    plt.scatter(
        sub["PC1"], sub["PC2"], 
        label=lbl, 
        color=color_lookup.get(cat, OTHER_LEGEND_COLOR),
        s=DOT_SIZE, 
        alpha=0.85, 
        zorder=10 if cat == CONTROL_LABEL else 3, 
        edgecolors='none'
    )

for i, row in pca_result_df.reset_index().iterrows():
    if row["Annotation"] == CONTROL_LABEL: continue
    if cluster_sizes[i] <= max_cluster_size:
        plt.text(row["PC1"] + text_offset, row["PC2"] + text_offset, 
                 row["Gene_target"], fontdict=FONT_PLOT_LABELS, zorder=10)

apply_thesis_style(ax, pca)

handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))

plt.legend(by_label.values(), by_label.keys(), 
           bbox_to_anchor=(1.05, 1), loc="upper left", 
           frameon=False, prop=FONT_LEGEND, 
           markerscale=LEGEND_MARKER_SCALE)

plt.tight_layout()

plt.savefig(os.path.join(OUTPUT_DIR, "PCA_Pathway_Overlay.png"), dpi=500, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, "PCA_Pathway_Overlay.svg"), format="svg", bbox_inches='tight')
plt.show()

print(f"Success! Plot saved to: {OUTPUT_DIR}")

In [ ]:
#met cnoturlijnen mislukt

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform
import scipy.stats as st

# --- 1. DIRECTORY CONFIGURATION ---
save_directory = r'E:\Thesis3april\GrowthResults' 
OUTPUT_DIR = os.path.join(save_directory, "7april", "FinaalContour")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# --- 2. FONT & STYLE CONFIGURATION ---
FONT_FAMILY = 'sans-serif' 
FONT_AXIS_LABEL = {'family': FONT_FAMILY, 'size': 18, 'weight': 'normal'}
FONT_TICKS = 18            
FONT_PLOT_LABELS = {'family': FONT_FAMILY, 'size': 9, 'alpha': 0} 
FONT_LEGEND = {'family': FONT_FAMILY, 'size': 16} 
AXIS_LINE_WIDTH = 1.5      
TICK_WIDTH = 1.5           
TICK_LENGTH = 5            

# --- DOT SIZE SETTINGS ---
DOT_SIZE = 60              
LEGEND_MARKER_SCALE = 1.3   

plt.rcParams['font.family'] = FONT_FAMILY

# --- 3. PATHWAY & COLOR SETTINGS ---
annotation_path = r'E:\Thesis3april\overlay\Pathway_annotation.xlsx'

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c", "cell shape": "#d62728", 
    "biosynthesis of fatty acids": "#9467bd", "DNA replication": "#8c564b", 
    "DNA condensation/ segregation": "#e377c2", "biosynthesis of isoprenoids": "#7f7f7f", 
    "cell division": "#bcbd22", "ribosomal proteins": "#17becf", 
    "biosynthesis of iron-sulfur clusters": "#aec7e8", "glycolysis": "#ffbb78", 
    "biosynthesis of menaquinone": "#98df8a"
}
OTHER_LABEL, OTHER_LEGEND_COLOR = "Unknown/Other", "#D3D3D3"
CONTROL_LABEL, CONTROL_COLOR = "no-sgRNA", "#000000"

# --- 4. DATA PROCESSING ---
df_xyl = pd.read_csv(os.path.join(save_directory, "Xylose_Means.csv"))
df_ann = pd.read_excel(annotation_path)

metrics_to_use = ["AUC", "log(AUC1/AUC2)", "Max_OD", "Max_OD_Time (h)", "Final_OD", "Peak_Prominence_Max"]
pca_df_clean = df_xyl.copy()
pca_df_clean[metrics_to_use] = pca_df_clean[metrics_to_use].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_df_clean[metrics_to_use])
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_result_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"], index=pca_df_clean.index)
pca_result_df["Gene_target"] = pca_df_clean["Gene_target"].astype(str).str.strip()

def get_pathway(row):
    for col in ['SubtiWiki Annotation 4', 'SubtiWiki Annotation 3']:
        val = str(row.get(col, '')).strip()
        if val not in ['nan', 'NA', 'None', '']: return val
    return OTHER_LABEL

df_ann['Pathway_Temp'] = df_ann.apply(get_pathway, axis=1)
pathway_map = dict(zip(df_ann['Treatment'].astype(str).str.strip(), df_ann['Pathway_Temp']))

def assign_annotation(gene_name):
    if gene_name.lower() in ['no_sgrna', 'no-sgrna']: return CONTROL_LABEL
    return pathway_map.get(gene_name, OTHER_LABEL)

pca_result_df['Annotation'] = pca_result_df['Gene_target'].apply(assign_annotation)

unique_cats = sorted(pca_result_df['Annotation'].unique())
color_lookup = {**MANUAL_COLORS, CONTROL_LABEL: CONTROL_COLOR, OTHER_LABEL: OTHER_LEGEND_COLOR}
auto_cmap = plt.get_cmap('tab20b')
extra_cats = [c for c in unique_cats if c not in color_lookup]
for i, cat in enumerate(extra_cats):
    color_lookup[cat] = mcolors.to_hex(auto_cmap(i % 20))

# Distance calc for labeling
positions = pca_result_df[["PC1", "PC2"]].to_numpy()
dist_matrix = squareform(pdist(positions))
min_distance, max_cluster_size, text_offset = 0.8, 5, 0.05
cluster_sizes = np.sum(dist_matrix < min_distance, axis=1) - 1

# --- 5. PLOTTING ---
def apply_thesis_style(ax, pca_obj):
    ax.spines['bottom'].set_linewidth(AXIS_LINE_WIDTH)
    ax.spines['left'].set_linewidth(AXIS_LINE_WIDTH)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', which='major', labelsize=FONT_TICKS, width=TICK_WIDTH, length=TICK_LENGTH)
    ax.set_xlabel(f"PC1 ({pca_obj.explained_variance_ratio_[0]*100:.1f}% variance)", fontdict=FONT_AXIS_LABEL)
    ax.set_ylabel(f"PC2 ({pca_obj.explained_variance_ratio_[1]*100:.1f}% variance)", fontdict=FONT_AXIS_LABEL)
    ax.set_box_aspect(1) 

plt.figure(figsize=(12, 8))
ax = plt.gca()

# --- A. SCATTER POINTS ---
priority_cats = [CONTROL_LABEL] + [c for c in sorted(unique_cats) if c != CONTROL_LABEL]

for cat in priority_cats:
    sub = pca_result_df[pca_result_df['Annotation'] == cat]
    lbl = cat if (cat == CONTROL_LABEL or cat in MANUAL_COLORS) else "_nolegend_"
    
    plt.scatter(
        sub["PC1"], sub["PC2"], 
        label=lbl, 
        color=color_lookup.get(cat, OTHER_LEGEND_COLOR),
        s=DOT_SIZE, 
        alpha=0.85, 
        zorder=10 if cat == CONTROL_LABEL else 3, 
        edgecolors='none'
    )

# --- B. CONTOUR LOGIC (Drawn ON TOP) ---
ctrl_data = pca_result_df[pca_result_df['Annotation'] == CONTROL_LABEL]
if not ctrl_data.empty:
    x_pts, y_pts = ctrl_data['PC1'].values, ctrl_data['PC2'].values
    kde = st.gaussian_kde(np.vstack([x_pts, y_pts]))
    
    # Grid for contour
    pad = 2
    x_grid = np.linspace(pca_result_df['PC1'].min()-pad, pca_result_df['PC1'].max()+pad, 100)
    y_grid = np.linspace(pca_result_df['PC2'].min()-pad, pca_result_df['PC2'].max()+pad, 100)
    X, Y = np.meshgrid(x_grid, y_grid)
    Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
    
    # Density levels
    densities = kde(np.vstack([x_pts, y_pts]))
    levels = [np.percentile(densities, p) for p in [5, 30, 55, 80]]
    
    # Plot contours with high zorder to be on top of everything
    ax.contour(X, Y, Z, levels=levels, colors='#808080', linewidths=1.5, alpha=1.0, zorder=20)

# --- C. GENE LABELS ---
for i, row in pca_result_df.reset_index().iterrows():
    if row["Annotation"] == CONTROL_LABEL: continue
    if cluster_sizes[i] <= max_cluster_size:
        plt.text(row["PC1"] + text_offset, row["PC2"] + text_offset, 
                 row["Gene_target"], fontdict=FONT_PLOT_LABELS, zorder=21)

apply_thesis_style(ax, pca)

# --- D. LEGEND HANDLING ---
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))

plt.legend(by_label.values(), by_label.keys(), 
           bbox_to_anchor=(1.05, 1), loc="upper left", 
           frameon=False, prop=FONT_LEGEND, 
           markerscale=LEGEND_MARKER_SCALE)

plt.tight_layout()

# Save
plt.savefig(os.path.join(OUTPUT_DIR, "PCA_Contour_OnTop.png"), dpi=500, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, "PCA_Contour_OnTop.svg"), format="svg", bbox_inches='tight')
plt.show()

print(f"Success! Plot with top-layer contours saved to: {OUTPUT_DIR}")

In [ ]:
#dit was contourlijnen maar geburik ik uitneilijk niet

In [ ]:
#pathya ovelray met keizen welke genes lablelne

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# --- 1. DIRECTORY & LABEL CONFIGURATION ---
save_directory = r'E:\Thesis3april\GrowthResults' 
OUTPUT_DIR = os.path.join(save_directory, "7april", "Pathwaykiezenlables")

# ONLY these genes will be labeled on the plot:
LABELED_GENES = [
    "tagD", "tagB", "tagF", "dnaE", "leuS", "dapB", 
    "pgm", "fabF", "rpsD", "ftsZ", "ftsW", "rpsJ", "rpsK"
]

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# --- 2. FONT & STYLE CONFIGURATION ---
FONT_FAMILY = 'sans-serif' 
FONT_AXIS_LABEL = {'family': FONT_FAMILY, 'size': 13, 'weight': 'normal'}
FONT_TICKS = 14            
FONT_PLOT_LABELS = {'family': FONT_FAMILY, 'size': 10, 'weight': 'bold', 'alpha': 0.9}
FONT_LEGEND = {'family': FONT_FAMILY, 'size': 11} 
AXIS_LINE_WIDTH = 1.5      
TICK_WIDTH = 1.5           
TICK_LENGTH = 5            

plt.rcParams['font.family'] = FONT_FAMILY

# --- 3. PATHWAY & COLOR SETTINGS ---
annotation_path = r'E:\Thesis3april\overlay\Pathway_annotation.xlsx'

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c", "cell shape": "#d62728", 
    "biosynthesis of fatty acids": "#9467bd", "DNA replication": "#8c564b", 
    "DNA condensation/ segregation": "#e377c2", "biosynthesis of isoprenoids": "#7f7f7f", 
    "cell division": "#bcbd22", "ribosomal proteins": "#17becf", 
    "biosynthesis of iron-sulfur clusters": "#aec7e8", "glycolysis": "#ffbb78", 
    "biosynthesis of menaquinone": "#98df8a"
}
OTHER_LABEL, OTHER_LEGEND_COLOR = "Unknown/Other", "#D3D3D3"
CONTROL_LABEL, CONTROL_COLOR = "Control Group", "#000000"

# --- 4. DATA PROCESSING ---
df_xyl = pd.read_csv(os.path.join(save_directory, "Xylose_Means.csv"))
df_ann = pd.read_excel(annotation_path)

metrics_to_use = ["AUC", "log(AUC1/AUC2)", "Max_OD", "Max_OD_Time (h)", "Final_OD", "Peak_Prominence_Max"]
pca_df_clean = df_xyl.copy()
pca_df_clean[metrics_to_use] = pca_df_clean[metrics_to_use].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_df_clean[metrics_to_use])
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_result_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"], index=pca_df_clean.index)
pca_result_df["Gene_target"] = pca_df_clean["Gene_target"].astype(str).str.strip()

# Pathway Mapping
def get_pathway(row):
    for col in ['SubtiWiki Annotation 4', 'SubtiWiki Annotation 3']:
        val = str(row.get(col, '')).strip()
        if val not in ['nan', 'NA', 'None', '']: return val
    return OTHER_LABEL

df_ann['Pathway_Temp'] = df_ann.apply(get_pathway, axis=1)
pathway_map = dict(zip(df_ann['Treatment'].astype(str).str.strip(), df_ann['Pathway_Temp']))

def assign_annotation(gene_name):
    if gene_name.lower() == 'no_sgrna': return CONTROL_LABEL
    return pathway_map.get(gene_name, OTHER_LABEL)

pca_result_df['Annotation'] = pca_result_df['Gene_target'].apply(assign_annotation)

# Color Logic
unique_cats = sorted(pca_result_df['Annotation'].unique())
color_lookup = {**MANUAL_COLORS, CONTROL_LABEL: CONTROL_COLOR, OTHER_LABEL: OTHER_LEGEND_COLOR}
auto_cmap = plt.get_cmap('tab20b')
extra_cats = [c for c in unique_cats if c not in color_lookup]
for i, cat in enumerate(extra_cats):
    color_lookup[cat] = mcolors.to_hex(auto_cmap(i % 20))

# --- 5. PLOTTING ---
def apply_thesis_style(ax, pca_obj):
    ax.spines['bottom'].set_linewidth(AXIS_LINE_WIDTH)
    ax.spines['left'].set_linewidth(AXIS_LINE_WIDTH)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', which='major', labelsize=FONT_TICKS, width=TICK_WIDTH, length=TICK_LENGTH)
    ax.set_xlabel(f"PC1 ({pca_obj.explained_variance_ratio_[0]*100:.1f}% variance)", fontdict=FONT_AXIS_LABEL)
    ax.set_ylabel(f"PC2 ({pca_obj.explained_variance_ratio_[1]*100:.1f}% variance)", fontdict=FONT_AXIS_LABEL)
    ax.set_box_aspect(1) 

plt.figure(figsize=(11, 8))
ax = plt.gca()

# Scatter Plot
for cat in unique_cats:
    sub = pca_result_df[pca_result_df['Annotation'] == cat]
    lbl = cat if (cat in MANUAL_COLORS or cat == CONTROL_LABEL) else "_nolegend_"
    
    plt.scatter(
        sub["PC1"], sub["PC2"], label=lbl, 
        color=color_lookup.get(cat, OTHER_LEGEND_COLOR),
        s=45, alpha=0.85, zorder=5 if cat == CONTROL_LABEL else 3, edgecolors='none'
    )

# Manual Labeling Logic
text_offset = 0.07
for _, row in pca_result_df.iterrows():
    # Only label if gene name is in your specified list
    if row["Gene_target"] in LABELED_GENES:
        plt.text(
            row["PC1"] + text_offset, 
            row["PC2"] + text_offset, 
            row["Gene_target"], 
            fontdict=FONT_PLOT_LABELS, 
            zorder=10
        )

apply_thesis_style(ax, pca)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", frameon=False, prop=FONT_LEGEND)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "PCA_Manual_Labels.png"), dpi=500, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, "PCA_Manual_Labels.svg"), format="svg", bbox_inches='tight')
plt.show()

print(f"Plotting complete. Outputs in: {OUTPUT_DIR}")

In [ ]:
#met circles erond

In [ ]:
#metsubplot

In [ ]:
#pathway annotation

In [ ]:
#pathwya overlay same colors as profiling

In [ ]:
#annotation met contours

In [ ]:
#pathway overlay met labelled HTMLs

In [ ]:
#7april goed

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import scipy.stats as st
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from matplotlib.lines import Line2D

# --- 1. CONFIGURATION & DIRECTORIES ---
save_directory = r'E:\Thesis3april\GrowthResults' 
output_subdir = os.path.join(save_directory, "7april", "PathwayPlots")
if not os.path.exists(output_subdir):
    os.makedirs(output_subdir)

# Specific pathway colors
MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c",      
    "cell shape": "#d62728",                          
    "biosynthesis of fatty acids": "#9467bd",     
    "DNA replication": "#8c564b",                  
    "DNA condensation/ segregation": "#e377c2",   
    "biosynthesis of isoprenoids": "#7f7f7f",     
    "cell division": "#bcbd22",                    
    "ribosomal proteins": "#17becf",              
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78",                        
    "biosynthesis of menaquinone": "#98df8a"
}

OTHER_LABEL = "Unknown/Other"
OTHER_LEGEND_COLOR = "#D3D3D3" 
CONTROL_LABEL = "Control Group"
CONTROL_COLOR = "#000000"

plt.rcParams['font.family'] = 'sans-serif'

# --- 2. LOAD & PREPARE DATA ---
df_xyl = pd.read_csv(os.path.join(save_directory, "Xylose_Means.csv"))
annotation_path = r'E:\Thesis3april\overlay\Pathway_annotation.xlsx'
df_ann = pd.read_excel(annotation_path)

metrics_to_use = ["AUC", "log(AUC1/AUC2)", "Max_OD", "Max_OD_Time (h)", "Final_OD", "Peak_Prominence_Max"]
pca_df_clean = df_xyl.copy()
pca_df_clean[metrics_to_use] = pca_df_clean[metrics_to_use].fillna(0)

# PCA Transformation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_df_clean[metrics_to_use])
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_result_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"], index=pca_df_clean.index)
pca_result_df["Gene_target"] = pca_df_clean["Gene_target"].astype(str).str.strip()

# --- 3. UPDATED MAPPING LOGIC (Control fix here) ---

# First: Create a helper dictionary from the Excel annotation file
def get_pathway_from_excel(row):
    ann4 = str(row.get('SubtiWiki Annotation 4', '')).strip()
    ann3 = str(row.get('SubtiWiki Annotation 3', '')).strip()
    if ann4 not in ['nan', 'NA', 'None', '']: return ann4
    if ann3 not in ['nan', 'NA', 'None', '']: return ann3
    return OTHER_LABEL

df_ann['Pathway_Temp'] = df_ann.apply(get_pathway_from_excel, axis=1)
# Map Treatment ID to the Pathway string
pathway_map = dict(zip(df_ann['Treatment'].astype(str).str.strip(), df_ann['Pathway_Temp']))

# Second: Apply the logic to identify Control vs Pathways based on Gene_target
def assign_final_annotation(gene_name):
    # CRITICAL FIX: Identify control if Gene_target is no_sgRNA
    if gene_name.lower() == 'no_sgrna':
        return CONTROL_LABEL
    # Otherwise, look up the pathway in the excel map
    return pathway_map.get(gene_name, OTHER_LABEL)

pca_result_df['Annotation'] = pca_result_df['Gene_target'].apply(assign_final_annotation)

# --- 4. SYNCED COLOR ASSIGNMENT ---
unique_cats = sorted(pca_result_df['Annotation'].unique())
color_lookup = MANUAL_COLORS.copy()
color_lookup[CONTROL_LABEL] = CONTROL_COLOR
color_lookup[OTHER_LABEL] = OTHER_LEGEND_COLOR

# Handle extra pathways not in MANUAL_COLORS
auto_cmap_b = plt.get_cmap('tab20b')
auto_cmap_c = plt.get_cmap('tab20c')
extra_hex_pool = [mcolors.to_hex(auto_cmap_b(i/20)) for i in range(20)] + \
                 [mcolors.to_hex(auto_cmap_c(i/20)) for i in range(20)]

extra_cats = [c for c in unique_cats if c not in color_lookup]
for i, cat in enumerate(extra_cats):
    color_lookup[cat] = extra_hex_pool[i % len(extra_hex_pool)]

# --- 5. STATIC PLOTTING ---
fig, ax = plt.subplots(figsize=(10, 8))

# Density Contours for Control
df_control = pca_result_df[pca_result_df['Annotation'] == CONTROL_LABEL]
if len(df_control) > 100:           #zet dit lager als je density contorus wil zien
    try:
        x, y = df_control["PC1"].values, df_control["PC2"].values
        kernel = st.gaussian_kde(np.vstack([x, y]))
        xmin, xmax = x.min()-1, x.max()+1
        ymin, ymax = y.min()-1, y.max()+1
        xx, yy = np.mgrid[xmin:xmax:100j, ymin:ymax:100j]
        f = np.reshape(kernel(np.vstack([xx.ravel(), yy.ravel()])).T, xx.shape)
        levels = sorted([np.percentile(kernel(np.vstack([x, y])), p) for p in [5, 30, 80]])
        ax.contour(xx, yy, f, levels=levels, colors=CONTROL_COLOR, linewidths=1, alpha=0.3, zorder=1)
    except: pass

# Plotting the points
for cat in unique_cats:
    sub = pca_result_df[pca_result_df['Annotation'] == cat]
    
    # Only show in legend if in Manual List or is the Control
    lbl = cat if (cat in MANUAL_COLORS or cat == CONTROL_LABEL) else "_nolegend_"

    ax.scatter(
        sub["PC1"], sub["PC2"], 
        label=lbl, 
        color=color_lookup.get(cat, OTHER_LEGEND_COLOR), 
        s=45, alpha=0.99, 
        edgecolors='none', 
        zorder=5 if cat == CONTROL_LABEL else 3
    )

# --- SQUARE ASPECT RATIO & STYLING ---
ax.set_box_aspect(1) 
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)", fontsize=12)
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)", fontsize=12)

# Legend Formatting - positioned to the right
handles, labels = ax.get_legend_handles_labels()
plt.legend(handles=handles, labels=labels, bbox_to_anchor=(1.05, 1), 
           loc="upper left", frameon=False, fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(output_subdir, "PCA_Vibrant_Square.png"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#pca  met enkel deel vd annotaties

In [ ]:
#pca with counts 

In [ ]:
######


statistics



#######

In [ ]:
#excels for statistics

In [ ]:
import pandas as pd
import os

# --- 1. Filter for Raw Replicates only ---
# We exclude the 'mean' and 'sd' rows to keep only the individual measurements
raw_reps_df = growth_data_with_avgs[~growth_data_with_avgs["rep"].isin(["mean", "sd"])].copy()

# --- 2. Clean up the columns for export ---
# We ensure the columns are in a logical order for manual inspection
raw_export_cols = [
    'Sample', 'Gene_target', 'Xylose', 'rep', 
    'Growth_rate (.h-1)', 'AUC', 'AUC1', 'AUC2', 
    'Max_OD', 'Max_OD_Time (h)', 'Final_OD', 'Peak_Prominence_Max'
]

# Only keep columns that actually exist in your dataframe
final_raw_cols = [c for c in raw_export_cols if c in raw_reps_df.columns]
raw_reps_export = raw_reps_df[final_raw_cols]

# --- 3. Save to CSV ---
raw_filename = "Growth_Metrics_Raw_Replicates.csv"
raw_path = os.path.join(save_directory, raw_filename)

raw_reps_export.to_csv(raw_path, index=False)

print(f"✅ Raw replicates saved to: {raw_path}")
print(f"📍 Total rows exported: {len(raw_reps_export)}")

In [ ]:
#tesgin auc and growht

In [ ]:
#Welch test

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import os

# --- 1. SETUP & DATA PREPARATION ---
save_directory = r'E:\Thesis3april\GrowthResults'
# Filtering for raw replicates (excluding mean/sd rows)
raw_reps_df = growth_data_with_avgs[~growth_data_with_avgs["rep"].isin(["mean", "sd"])].copy()

# Identifying Controls
control_names = ['no_sgrna', 'nosgrna', 'control', 'scr']
raw_reps_df['Is_Control'] = raw_reps_df['Gene_target'].str.lower().isin(control_names)

metrics = ['Growth_rate (.h-1)', 'AUC']
mutants = raw_reps_df[~raw_reps_df['Is_Control']]['Gene_target'].unique()

# Results Container
full_stats_results = []

# --- 2. STATISTICAL EXECUTION ---
for m in metrics:
    # A. CONTROL BASELINE (Xylose vs No Xylose)
    c_no = raw_reps_df[(raw_reps_df['Is_Control']) & (raw_reps_df['Xylose'] == 'no xylose')][m].dropna()
    c_yes = raw_reps_df[(raw_reps_df['Is_Control']) & (raw_reps_df['Xylose'] == 'xylose')][m].dropna()
    
    if len(c_no) >= 2 and len(c_yes) >= 2:
        # Two-tailed: Is it different?
        _, p_diff = stats.ttest_ind(c_yes, c_no, equal_var=False, alternative='two-sided')
        # One-tailed: Is it lower?
        _, p_lower = stats.ttest_ind(c_yes, c_no, equal_var=False, alternative='less')
        
        full_stats_results.append({'Metric': m, 'Test_Type': 'Control_Baseline', 'Condition': 'Xylose_vs_NoXylose', 
                                   'Result_Value': p_diff, 'Measurement': 'p-value (Different)'})
        full_stats_results.append({'Metric': m, 'Test_Type': 'Control_Baseline', 'Condition': 'Xylose_vs_NoXylose', 
                                   'Result_Value': p_lower, 'Measurement': 'p-value (Lower)'})

    # B. MUTANT INTERNAL SHIFT (Xylose vs No Xylose)
    sig_diff_m, sig_lower_m, total_m = 0, 0, 0
    for mutant in mutants:
        m_no = raw_reps_df[(raw_reps_df['Gene_target'] == mutant) & (raw_reps_df['Xylose'] == 'no xylose')][m].dropna()
        m_yes = raw_reps_df[(raw_reps_df['Gene_target'] == mutant) & (raw_reps_df['Xylose'] == 'xylose')][m].dropna()
        
        if len(m_no) >= 2 and len(m_yes) >= 2:
            total_m += 1
            if stats.ttest_ind(m_yes, m_no, equal_var=False, alternative='two-sided')[1] < 0.05: sig_diff_m += 1
            if stats.ttest_ind(m_yes, m_no, equal_var=False, alternative='less')[1] < 0.05: sig_lower_m += 1
            
    full_stats_results.append({'Metric': m, 'Test_Type': 'Mutant_Internal_Shift', 'Condition': 'Xylose_vs_NoXylose', 
                               'Result_Value': (sig_diff_m/total_m*100) if total_m > 0 else 0, 'Measurement': '% Mutants Different'})
    full_stats_results.append({'Metric': m, 'Test_Type': 'Mutant_Internal_Shift', 'Condition': 'Xylose_vs_NoXylose', 
                               'Result_Value': (sig_lower_m/total_m*100) if total_m > 0 else 0, 'Measurement': '% Mutants Lower'})

    # C. MUTANTS VS CONTROLS (Condition Specific)
    for cond in ['no xylose', 'xylose']:
        v_diff, v_lower, total_v = 0, 0, 0
        current_ctrls = raw_reps_df[(raw_reps_df['Is_Control']) & (raw_reps_df['Xylose'] == cond)][m].dropna()
        
        for mutant in mutants:
            m_vals = raw_reps_df[(raw_reps_df['Gene_target'] == mutant) & (raw_reps_df['Xylose'] == cond)][m].dropna()
            if len(m_vals) >= 2 and len(current_ctrls) >= 2:
                total_v += 1
                if stats.ttest_ind(m_vals, current_ctrls, equal_var=False, alternative='two-sided')[1] < 0.05: v_diff += 1
                if stats.ttest_ind(m_vals, current_ctrls, equal_var=False, alternative='less')[1] < 0.05: v_lower += 1
        
        full_stats_results.append({'Metric': m, 'Test_Type': 'Mutant_vs_Control', 'Condition': cond, 
                                   'Result_Value': (v_diff/total_v*100) if total_v > 0 else 0, 'Measurement': '% Mutants Different'})
        full_stats_results.append({'Metric': m, 'Test_Type': 'Mutant_vs_Control', 'Condition': cond, 
                                   'Result_Value': (v_lower/total_v*100) if total_v > 0 else 0, 'Measurement': '% Mutants Lower'})

# --- 3. SAVE & DISPLAY ---
summary_df = pd.DataFrame(full_stats_results)
output_file = os.path.join(save_directory, "PCA_Growth_Statistics_Full_Summary.csv")
summary_df.to_csv(output_file, index=False)

print("\n" + "="*60)
print("COMBINED GROWTH STATISTICS SUMMARY")
print("="*60)
print(summary_df.to_string(index=False))
print("\n" + "="*60)
print(f"✅ Full report saved to: {output_file}")

In [ ]:
#standard t test

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import os

# --- 1. SETUP & DATA PREPARATION ---
save_directory = r'E:\Thesis3april\GrowthResults'
# Filtering for raw replicates (excluding mean/sd rows)
raw_reps_df = growth_data_with_avgs[~growth_data_with_avgs["rep"].isin(["mean", "sd"])].copy()

# Identifying Controls
control_names = ['no_sgrna', 'nosgrna', 'control', 'scr']
raw_reps_df['Is_Control'] = raw_reps_df['Gene_target'].str.lower().isin(control_names)

metrics = ['Growth_rate (.h-1)', 'AUC']
mutants = raw_reps_df[~raw_reps_df['Is_Control']]['Gene_target'].unique()

# Results Container
standard_stats_results = []

# --- 2. STATISTICAL EXECUTION (Standard T-test: equal_var=True) ---
for m in metrics:
    # A. CONTROL BASELINE (Xylose vs No Xylose)
    c_no = raw_reps_df[(raw_reps_df['Is_Control']) & (raw_reps_df['Xylose'] == 'no xylose')][m].dropna()
    c_yes = raw_reps_df[(raw_reps_df['Is_Control']) & (raw_reps_df['Xylose'] == 'xylose')][m].dropna()
    
    if len(c_no) >= 2 and len(c_yes) >= 2:
        # Standard T-Test: equal_var=True
        _, p_diff = stats.ttest_ind(c_yes, c_no, equal_var=True, alternative='two-sided')
        _, p_lower = stats.ttest_ind(c_yes, c_no, equal_var=True, alternative='less')
        
        standard_stats_results.append({'Metric': m, 'Test_Type': 'Control_Baseline', 'Condition': 'Xylose_vs_NoXylose', 
                                      'Result_Value': p_diff, 'Measurement': 'p-value (Different)'})
        standard_stats_results.append({'Metric': m, 'Test_Type': 'Control_Baseline', 'Condition': 'Xylose_vs_NoXylose', 
                                      'Result_Value': p_lower, 'Measurement': 'p-value (Lower)'})

    # B. MUTANT INTERNAL SHIFT (Xylose vs No Xylose)
    sig_diff_m, sig_lower_m, total_m = 0, 0, 0
    for mutant in mutants:
        m_no = raw_reps_df[(raw_reps_df['Gene_target'] == mutant) & (raw_reps_df['Xylose'] == 'no xylose')][m].dropna()
        m_yes = raw_reps_df[(raw_reps_df['Gene_target'] == mutant) & (raw_reps_df['Xylose'] == 'xylose')][m].dropna()
        
        if len(m_no) >= 2 and len(m_yes) >= 2:
            total_m += 1
            if stats.ttest_ind(m_yes, m_no, equal_var=True, alternative='two-sided')[1] < 0.05: sig_diff_m += 1
            if stats.ttest_ind(m_yes, m_no, equal_var=True, alternative='less')[1] < 0.05: sig_lower_m += 1
            
    standard_stats_results.append({'Metric': m, 'Test_Type': 'Mutant_Internal_Shift', 'Condition': 'Xylose_vs_NoXylose', 
                                  'Result_Value': (sig_diff_m/total_m*100) if total_m > 0 else 0, 'Measurement': '% Mutants Different'})
    standard_stats_results.append({'Metric': m, 'Test_Type': 'Mutant_Internal_Shift', 'Condition': 'Xylose_vs_NoXylose', 
                                  'Result_Value': (sig_lower_m/total_m*100) if total_m > 0 else 0, 'Measurement': '% Mutants Lower'})

    # C. MUTANTS VS CONTROLS (Condition Specific)
    for cond in ['no xylose', 'xylose']:
        v_diff, v_lower, total_v = 0, 0, 0
        current_ctrls = raw_reps_df[(raw_reps_df['Is_Control']) & (raw_reps_df['Xylose'] == cond)][m].dropna()
        
        for mutant in mutants:
            m_vals = raw_reps_df[(raw_reps_df['Gene_target'] == mutant) & (raw_reps_df['Xylose'] == cond)][m].dropna()
            if len(m_vals) >= 2 and len(current_ctrls) >= 2:
                total_v += 1
                if stats.ttest_ind(m_vals, current_ctrls, equal_var=True, alternative='two-sided')[1] < 0.05: v_diff += 1
                if stats.ttest_ind(m_vals, current_ctrls, equal_var=True, alternative='less')[1] < 0.05: v_lower += 1
        
        standard_stats_results.append({'Metric': m, 'Test_Type': 'Mutant_vs_Control', 'Condition': cond, 
                                      'Result_Value': (v_diff/total_v*100) if total_v > 0 else 0, 'Measurement': '% Mutants Different'})
        standard_stats_results.append({'Metric': m, 'Test_Type': 'Mutant_vs_Control', 'Condition': cond, 
                                      'Result_Value': (v_lower/total_v*100) if total_v > 0 else 0, 'Measurement': '% Mutants Lower'})

# --- 3. SAVE & DISPLAY ---
summary_df = pd.DataFrame(standard_stats_results)
output_file = os.path.join(save_directory, "PCA_Growth_Statistics_Standard_TTest.csv")
summary_df.to_csv(output_file, index=False)

print("\n" + "="*60)
print("STANDARD STUDENT'S T-TEST SUMMARY (Equal Variance)")
print("="*60)
print(summary_df.to_string(index=False))
print("\n" + "="*60)
print(f"✅ Standard report saved to: {output_file}")

In [ ]:
#welch test with benjamin hoghebergcorreciton

In [ ]:
!pip install statsmodels

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
import os

# --- 1. SETUP & DATA PREPARATION ---
file_path = r'E:\Thesis3april\GrowthResults\Excel_Export\Growth_Metrics_Raw_Replicates.csv'
save_directory = r'E:\Thesis3april\GrowthResults'
df = pd.read_csv(file_path)

# Identifying Controls
control_names = ['no_sgrna', 'nosgrna', 'control', 'scr']
df['Is_Control'] = df['Gene_target'].str.lower().isin(control_names)

metrics = ['AUC'] # You can add 'Growth_rate (.h-1)' back if needed
mutants = df[~df['Is_Control']]['Gene_target'].unique()

summary_results = []
detailed_mutant_hits = []

# --- 2. STATISTICAL EXECUTION ---
for m in metrics:
    # --- A. CONTROL BASELINE (Effect of Xylose on Controls) ---
    c_no = df[(df['Is_Control']) & (df['Xylose'] == 'no xylose')][m].dropna()
    c_yes = df[(df['Is_Control']) & (df['Xylose'] == 'xylose')][m].dropna()
    
    if len(c_no) >= 2 and len(c_yes) >= 2:
        _, p_baseline = stats.ttest_ind(c_yes, c_no, equal_var=False)
        summary_results.append({'Metric': m, 'Test': 'Control Baseline', 'Condition': 'Xyl vs NoXyl', 'Value': p_baseline, 'Unit': 'p-value'})

    # --- B. MUTANT INTERNAL SHIFT & C. MUTANT VS CONTROL ---
    # We collect p-values for FDR correction
    p_internal = []
    p_vs_ctrl_no = []
    p_vs_ctrl_yes = []

    for mutant in mutants:
        m_no = df[(df['Gene_target'] == mutant) & (df['Xylose'] == 'no xylose')][m].dropna()
        m_yes = df[(df['Gene_target'] == mutant) & (df['Xylose'] == 'xylose')][m].dropna()
        ctrl_no = c_no
        ctrl_yes = c_yes

        # 1. Internal Shift (Mutant Xyl vs Mutant No Xyl)
        p_int = stats.ttest_ind(m_yes, m_no, equal_var=False, alternative='less')[1] if len(m_yes)>=2 and len(m_no)>=2 else np.nan
        p_internal.append(p_int)

        # 2. Mutant vs Control (No Xylose) - Checking for leakiness
        p_v_c_no = stats.ttest_ind(m_no, ctrl_no, equal_var=False, alternative='less')[1] if len(m_no)>=2 and len(ctrl_no)>=2 else np.nan
        p_vs_ctrl_no.append(p_v_c_no)

        # 3. Mutant vs Control (Xylose) - Checking for hits
        p_v_c_yes = stats.ttest_ind(m_yes, ctrl_yes, equal_var=False, alternative='less')[1] if len(m_yes)>=2 and len(ctrl_yes)>=2 else np.nan
        p_vs_ctrl_yes.append(p_v_c_yes)

        detailed_mutant_hits.append({
            'Gene': mutant, 'Metric': m, 
            'p_internal': p_int, 'p_vs_ctrl_noXyl': p_v_c_no, 'p_vs_ctrl_Xyl': p_v_c_yes
        })

    # --- 3. MULTIPLE TESTING CORRECTION (BH/FDR) ---
    def get_hit_percent(p_list):
        p_clean = [p for p in p_list if not np.isnan(p)]
        if not p_clean: return 0, 0
        # Nominal percent (uncorrected)
        nom = sum(1 for p in p_clean if p < 0.05) / len(p_clean) * 100
        # FDR percent (corrected)
        rej, _, _, _ = multipletests(p_clean, alpha=0.05, method='fdr_bh')
        fdr = sum(rej) / len(rej) * 100
        return nom, fdr

    # Calculate percentages for Summary
    nom_int, fdr_int = get_hit_percent(p_internal)
    nom_v_no, fdr_v_no = get_hit_percent(p_vs_ctrl_no)
    nom_v_yes, fdr_v_yes = get_hit_percent(p_vs_ctrl_yes)

    summary_results.extend([
        {'Metric': m, 'Test': 'Mutant Internal Shift', 'Condition': 'Xyl vs NoXyl', 'Value': nom_int, 'Unit': '% Hits (Nominal)'},
        {'Metric': m, 'Test': 'Mutant Internal Shift', 'Condition': 'Xyl vs NoXyl', 'Value': fdr_int, 'Unit': '% Hits (FDR)'},
        {'Metric': m, 'Test': 'Mutant vs Control', 'Condition': 'No Xylose', 'Value': fdr_v_no, 'Unit': '% Hits (FDR)'},
        {'Metric': m, 'Test': 'Mutant vs Control', 'Condition': 'Xylose', 'Value': fdr_v_yes, 'Unit': '% Hits (FDR)'}
    ])

# --- 4. SAVE & DISPLAY ---
summary_df = pd.DataFrame(summary_results)
detailed_df = pd.DataFrame(detailed_mutant_hits)

summary_df.to_csv(os.path.join(save_directory, "Growth_Stats_Summary_welchmethoghberg.csv"), index=False)
detailed_df.to_csv(os.path.join(save_directory, "Detailed_Gene_Pvalues_welchmethoghber.csv"), index=False)

print("\n" + "="*60)
print("STATISTICS SUMMARY (Welch's T-Test + BH Correction)")
print("="*60)
print(summary_df.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
import os

# --- 1. SETUP & DATA LOADING ---
file_path = r'E:\Thesis3april\GrowthResults\Excel_Export\Growth_Metrics_Raw_Replicates.csv'
save_directory = r'E:\Thesis3april\GrowthResults'
df = pd.read_csv(file_path)

# Identifying Controls (Adjust these names if they differ in your file)
control_names = ['no_sgrna', 'nosgrna', 'control', 'scr']
df['Is_Control'] = df['Gene_target'].str.lower().isin(control_names)

metric = 'AUC'
mutants = df[~df['Is_Control']]['Gene_target'].unique()

detailed_results = []

# --- 2. STATISTICAL EXECUTION (Welch's T-Test) ---
# We compare each Mutant (+Xylose) against the global Control pool (+Xylose)
c_yes = df[(df['Is_Control']) & (df['Xylose'] == 'xylose')][metric].dropna()

for mutant in mutants:
    # Filter for the specific mutant under induction
    m_yes = df[(df['Gene_target'] == mutant) & (df['Xylose'] == 'xylose')][metric].dropna()
    
    # Requirement: At least 2 replicates for each side of the T-test
    if len(m_yes) >= 2 and len(c_yes) >= 2:
        # Welch's T-test (equal_var=False)
        # 'less' finds mutants where AUC is significantly lower than Control
        t_stat, p_raw = stats.ttest_ind(m_yes, c_yes, equal_var=False, alternative='less')
        
        detailed_results.append({
            'Gene_target': mutant,
            'n_reps': len(m_yes),
            'Mean_AUC': m_yes.mean(),
            'p_raw': p_raw
        })

# Convert to DataFrame
results_df = pd.DataFrame(detailed_results)

# --- 3. MULTIPLE TESTING CORRECTION ---
if not results_df.empty:
    # Benjamini-Hochberg (FDR) correction
    rejected, p_corrected, _, _ = multipletests(results_df['p_raw'], alpha=0.05, method='fdr_bh')
    
    results_df['p_corrected'] = p_corrected
    results_df['Significant_Nominal'] = results_df['p_raw'] < 0.05
    results_df['Significant_FDR'] = rejected

    # --- 4. SUMMARY & SAVING ---
    total = len(results_df)
    nom_hits = results_df['Significant_Nominal'].sum()
    fdr_hits = results_df['Significant_FDR'].sum()

    print("\n" + "="*60)
    print("FINAL STATISTICAL RESULTS (Welch's T-test)")
    print("="*60)
    print(f"Total Mutants Evaluated:      {total}")
    print(f"Hits without Correction (p<0.05): {nom_hits} ({nom_hits/total*100:.1f}%)")
    print(f"Hits with FDR Correction (q<0.05): {fdr_hits} ({fdr_hits/total*100:.1f}%)")
    print("-" * 60)
    
    # Sort by significance
    results_df = results_df.sort_values('p_raw')
    
    output_file = os.path.join(save_directory, "AUC_FDR_Results_Summary_hoghberghcorrectionerbij.csv")
    results_df.to_csv(output_file, index=False)
    print(f"✅ Detailed report saved to: {output_file}")

In [ ]:
#distribuitons

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# --- 1. LOAD THE EXPORTED DATA ---
file_path = r'E:\Thesis3april\GrowthResults\Excel_Export\Growth_Metrics_Raw_Replicates.csv'
df = pd.read_csv(file_path)

def analyze_auc_distribution(data_df, metric='AUC'):
    # Set plot style
    sns.set_theme(style="whitegrid")
    
    # Create a figure with two subplots
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # --- PLOT 1: Histogram & KDE (Visual Shape) ---
    # We color by Xylose to see if the distribution shifts
    hist = sns.histplot(data=data_df, x=metric, hue='Xylose', kde=True, ax=axes[0], palette='viridis')
    axes[0].set_title(f'Histogram of {metric}')
    axes[0].set_box_aspect(1) # Force square
    
    # Move legend next to the plot
    sns.move_legend(axes[0], "upper left", bbox_to_anchor=(1, 1), title='Condition')

    # --- PLOT 2: Q-Q Plot (Normality Verification) ---
    # Dots on the line = Normal distribution
    res = stats.probplot(data_df[metric].dropna(), dist="norm", plot=axes[1])
    axes[1].get_lines()[0].set_markerfacecolor('#4C72B0')
    axes[1].get_lines()[0].set_markersize(4)
    axes[1].set_title(f'Normal Q-Q Plot: {metric}')
    axes[1].set_box_aspect(1) # Force square
    
    plt.tight_layout()
    plt.show()

    # --- 2. STATISTICAL TESTS ---
    print(f"\n--- Distribution Statistics for {metric} ---")
    for condition in data_df['Xylose'].unique():
        subset = data_df[data_df['Xylose'] == condition][metric].dropna()
        if len(subset) >= 3:
            # Shapiro-Wilk Test
            shapiro_p = stats.shapiro(subset).pvalue
            # Skewness (0 = perfectly symmetrical)
            skew_val = stats.skew(subset)
            
            print(f"Condition [{condition}]:")
            print(f"  - Shapiro-Wilk p-value: {shapiro_p:.5f}")
            print(f"  - Skewness: {skew_val:.2f}")
            
            if shapiro_p < 0.05:
                print("  ⚠️ Distribution is likely NOT normal.")
            else:
                print("  ✅ Distribution appears Gaussian.")

# Execute
analyze_auc_distribution(df, 'AUC')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# --- 1. SETUP & DATA LOADING ---
file_path = r'E:\Thesis3april\GrowthResults\Excel_Export\Growth_Metrics_Raw_Replicates.csv'
df = pd.read_csv(file_path)

# Define metric
metric = 'AUC'

# Define Controls (matching your previous logic)
control_names = ['no_sgrna', 'nosgrna', 'control', 'scr']
df['Is_Control'] = df['Gene_target'].str.lower().isin(control_names)

# Create a grouping column for easy plotting
df['Group'] = df.apply(lambda x: f"{'Control' if x['Is_Control'] else 'Mutant'} ({x['Xylose']})", axis=1)

def analyze_all_distributions(data_df, metric):
    sns.set_theme(style="whitegrid")
    
    # We want 4 groups: Control -Xyl, Control +Xyl, Mutant -Xyl, Mutant +Xyl
    groups = data_df['Group'].unique()
    
    # Create a figure grid: Rows = Groups, Cols = (Histogram, Q-Q Plot)
    fig, axes = plt.subplots(len(groups), 2, figsize=(12, 5 * len(groups)))

    for i, grp in enumerate(sorted(groups)):
        subset = data_df[data_df['Group'] == grp][metric].dropna()
        
        # --- Column 1: Histogram ---
        sns.histplot(subset, kde=True, ax=axes[i, 0], color='#4C72B0', element="step")
        axes[i, 0].set_title(f'Hist: {grp}')
        axes[i, 0].set_box_aspect(1) # Square plot
        
        # --- Column 2: Q-Q Plot ---
        stats.probplot(subset, dist="norm", plot=axes[i, 1])
        axes[i, 1].set_title(f'Q-Q: {grp}')
        axes[i, 1].set_box_aspect(1) # Square plot
        
        # Statistical Printout
        if len(subset) >= 3:
            shapiro_p = stats.shapiro(subset).pvalue
            skew_val = stats.skew(subset)
            print(f"{grp:25} | Shapiro p: {shapiro_p:.5f} | Skew: {skew_val:.2f}")

    plt.tight_layout()
    plt.show()

    # --- Combined Overlay Plot (With Legend to the Side) ---
    plt.figure(figsize=(10, 6))
    ax_combined = sns.histplot(data=data_df, x=metric, hue='Group', kde=True, element="poly")
    ax_combined.set_box_aspect(1) # Square plot
    sns.move_legend(ax_combined, "upper left", bbox_to_anchor=(1, 1), title='Experimental Groups')
    plt.title(f"Combined {metric} Distribution Overlap")
    plt.show()

# Execute
analyze_all_distributions(df, metric)

In [ ]:
!pip install seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

def check_auc_distribution(df, metric='AUC'):
    """
    Plots a histogram and Q-Q plot to check for normality.
    Ensures the plotting region is a square.
    """
    # Filter out NaNs for the calculation
    data = df[metric].dropna()
    
    # Create a figure with two subplots
    # We use a 1x2 grid, but set the figure size to ensure 
    # the individual plots feel square and proportional.
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    # 1. Histogram with KDE (Check for Bell Curve)
    sns.histplot(data, kde=True, ax=axes[0], color='#4C72B0')
    axes[0].set_title(f'Histogram of {metric}')
    axes[0].set_aspect('auto') # Standard for histograms
    
    # 2. Q-Q Plot (Check if points follow the red line)
    stats.probplot(data, dist="norm", plot=axes[1])
    axes[1].get_lines()[0].set_markerfacecolor('#C44E52') 
    axes[1].get_lines()[0].set_markersize(4)
    axes[1].set_title(f'Normal Q-Q Plot: {metric}')
    
    # Force the plotting area to be a square shape
    for ax in axes:
        ax.set_box_aspect(1) 

    plt.tight_layout()
    plt.show()

    # Formal Statistical Test (Shapiro-Wilk)
    stat, p = stats.shapiro(data)
    print(f"--- Normality Test for {metric} ---")
    print(f"Shapiro-Wilk Statistic: {stat:.4f}")
    print(f"p-value: {p:.5e}")
    
    if p < 0.05:
        print("Result: Data is significantly different from a normal distribution.")
        print("Advice: Consider a log-transformation or using a non-parametric test (like Mann-Whitney U).")
    else:
        print("Result: Data appears normally distributed. Your t-tests are safe to use.")

# --- Execution ---
# Assuming raw_reps_df is already defined from your previous steps
check_auc_distribution(raw_reps_df, metric='AUC')

In [ ]:
#statistics end

In [ ]:
#zonder gewone auc#dit is bij missing values 0 ingevuld

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re

# --- Ensure xylose column is consistent ---
growth_data_with_avgs["Xylose"] = growth_data_with_avgs["Xylose"].str.strip().str.lower()

# --- Select only mean and sd rows ---
mean_df = growth_data_with_avgs[growth_data_with_avgs["rep"] == "mean"]
sd_df = growth_data_with_avgs[growth_data_with_avgs["rep"] == "sd"]

# --- Merge means and SDs for AUC ratio calculation ---
merged_stats = pd.merge(
    mean_df[["Sample", "Xylose", "AUC", "Growth_category"]],
    sd_df[["Sample", "Xylose", "AUC"]],
    on=["Sample", "Xylose"],
    suffixes=("_mean", "_sd")
)

# --- Separate xylose and no xylose ---
df_xylose = merged_stats[merged_stats["Xylose"] == "xylose"].set_index("Sample")
df_noxylose = merged_stats[merged_stats["Xylose"] == "no xylose"].set_index("Sample")

# --- Keep only samples present in both conditions ---
common_samples = df_xylose.index.intersection(df_noxylose.index)
df_xylose = df_xylose.loc[common_samples]
df_noxylose = df_noxylose.loc[common_samples]

# --- Compute AUC ratio and propagated SD ---
ratio_mean = df_xylose["AUC_mean"] / df_noxylose["AUC_mean"]
ratio_sd = ratio_mean * np.sqrt(
    (df_xylose["AUC_sd"] / df_xylose["AUC_mean"])**2 +
    (df_noxylose["AUC_sd"] / df_noxylose["AUC_mean"])**2
)

# --- Combine into AUC ratio dataframe ---
auc_ratio_df = pd.DataFrame({
    "Sample": common_samples,
    "AUC_ratio_mean": ratio_mean,
    "AUC_ratio_sd": ratio_sd,
    "Growth_category": df_xylose["Growth_category"].values
}).reset_index(drop=True)

# --- Extract Gene target from the Sample column ---
def extract_gene_target(sample_str):
    match = re.search(r"Gene target:\s*([\w\-]+)", str(sample_str))
    return match.group(1) if match else sample_str

auc_ratio_df["Gene_target"] = auc_ratio_df["Sample"].apply(extract_gene_target)

# --- Add AUC1/AUC2 log ratio ---
mean_df["AUC1/AUC2"] = mean_df["AUC1"] / mean_df["AUC2"]
mean_df["log(AUC1/AUC2)"] = np.log(mean_df["AUC1/AUC2"])

# --- Merge AUC ratio into mean_df ---
mean_df = mean_df.merge(
    auc_ratio_df[["Sample", "AUC_ratio_mean"]],
    on="Sample",
    how="left"
)

# --- Define metrics for PCA (including AUC ratio) ---
metrics_for_pca = [
    # --- AUC-related ---
   
   
    "log(AUC1/AUC2)",
    "AUC_ratio_mean",
    
    # --- OD (Optical Density) Metrics ---
    "Max_OD",
    "Max_OD_Time (h)",
    "Final_OD",
    
    # --- Peak Metrics ---
    "Peak_Prominence_Max"
]

# --- Filter mean_df for xylose condition and chosen metrics ---
pca_df = mean_df[mean_df["Xylose"] == "xylose"].copy()
pca_df = pca_df[["Sample", "Gene_target", "Growth_category"] + metrics_for_pca].set_index("Sample")

# --- Replace missing metric values with 0 instead of dropping ---
pca_df_clean = pca_df.copy()
pca_df_clean[metrics_for_pca] = pca_df_clean[metrics_for_pca].fillna(0)
# List strains where Peak_Prominence_Max is 0
zero_peak_strains = pca_df_clean[pca_df_clean["Peak_Prominence_Max"] == 0]

print("Number of strains with Peak_Prominence_Max = 0:", len(zero_peak_strains))
print(zero_peak_strains.index.tolist())




# --- Standardize metrics ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_df_clean[metrics_for_pca])

# --- Run PCA ---
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# --- Create PCA result dataframe ---
pca_result_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"],
    index=pca_df_clean.index
)
pca_result_df["Gene_target"] = pca_df_clean["Gene_target"]

# --- Plot PCA with points in black except for no_sgRNA in red ---
import numpy as np
from scipy.spatial.distance import pdist, squareform

# --- Parameters for labeling ---
min_distance = 0.8   # distance threshold to consider "close"
max_cluster_size = 5  # clusters larger than this are not labeled
text_offset = 0.04    # offset for text from the points

positions = pca_result_df[["PC1", "PC2"]].to_numpy()
labels = pca_result_df["Gene_target"].to_numpy()

# Compute pairwise distances
dist_matrix = squareform(pdist(positions))

# Determine cluster sizes (number of neighbors within min_distance)
cluster_sizes = np.sum(dist_matrix < min_distance, axis=1) - 1  # subtract 1 to ignore self

# --- Plot PCA ---
plt.figure(figsize=(9, 7))

# Plot non-control samples (all genes except no_sgRNA) first
non_control = pca_result_df[pca_result_df["Gene_target"].str.lower() != "no_sgrna"]
plt.scatter(
    non_control["PC1"],
    non_control["PC2"],
    color='black',
    label='All genes',
    s=30,
    alpha=0.8,
    zorder=1  # background
)

# Plot control samples (no_sgRNA) last to appear in foreground
control = pca_result_df[pca_result_df["Gene_target"].str.lower() == "no_sgrna"]
if not control.empty:
    plt.scatter(
        control["PC1"],
        control["PC2"],
        color='red',
        label='no_sgRNA',
        s=30,
        alpha=0.8,
        zorder=2  # foreground
    )

# --- Add gene target labels with offset ---
for i, (x, y) in enumerate(positions):
    # Label if small cluster (<= max_cluster_size) or always for no_sgRNA, schrijf no_sgrna in keline letterso meht te laten werken
    # --- Add labels only for non-no_sgrna + small clusters ---

    if labels[i].lower() == "no_sgrna":
        continue

    if cluster_sizes[i] <= max_cluster_size or labels[i].lower() == "no_sgRNA":
        plt.text(x + text_offset, y + text_offset, labels[i], fontsize=8, alpha=0.8, zorder=3)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.title("PCA of Selected Metrics (xylose condition)")
#plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"pcaRed_BlackJuist1zonderauc.svg", format="svg")
plt.show()



print(f"Number of strains plotted in PCA: {len(pca_result_df)}")
print(f"Number of strains plotted in PCA zonder nosgrna: {len(pca_result_df[pca_result_df["Gene_target"].str.lower() != "no_sgrna"])}")


In [ ]:
#zonder gewone AUC

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
from scipy.spatial.distance import pdist, squareform

# --- Ensure xylose column is consistent ---
growth_data_with_avgs["Xylose"] = growth_data_with_avgs["Xylose"].str.strip().str.lower()

# --- Select only mean and sd rows ---
mean_df = growth_data_with_avgs[growth_data_with_avgs["rep"] == "mean"]
sd_df = growth_data_with_avgs[growth_data_with_avgs["rep"] == "sd"]

# --- Merge means and SDs for AUC ratio calculation ---
merged_stats = pd.merge(
    mean_df[["Sample", "Xylose", "AUC", "Growth_category"]],
    sd_df[["Sample", "Xylose", "AUC"]],
    on=["Sample", "Xylose"],
    suffixes=("_mean", "_sd")
)

# --- Separate xylose and no xylose ---
df_xylose = merged_stats[merged_stats["Xylose"] == "xylose"].set_index("Sample")
df_noxylose = merged_stats[merged_stats["Xylose"] == "no xylose"].set_index("Sample")

# --- Keep only samples present in both conditions ---
common_samples = df_xylose.index.intersection(df_noxylose.index)
df_xylose = df_xylose.loc[common_samples]
df_noxylose = df_noxylose.loc[common_samples]

# --- Compute AUC ratio and propagated SD ---
ratio_mean = df_xylose["AUC_mean"] / df_noxylose["AUC_mean"]
ratio_sd = ratio_mean * np.sqrt(
    (df_xylose["AUC_sd"] / df_xylose["AUC_mean"])**2 +
    (df_noxylose["AUC_sd"] / df_noxylose["AUC_mean"])**2
)

# --- Combine into AUC ratio dataframe ---
auc_ratio_df = pd.DataFrame({
    "Sample": common_samples,
    "AUC_ratio_mean": ratio_mean,
    "AUC_ratio_sd": ratio_sd,
    "Growth_category": df_xylose["Growth_category"].values
}).reset_index(drop=True)

# --- Extract Gene target from the Sample column ---
def extract_gene_target(sample_str):
    match = re.search(r"Gene target:\s*([\w\-]+)", str(sample_str))
    return match.group(1) if match else sample_str

auc_ratio_df["Gene_target"] = auc_ratio_df["Sample"].apply(extract_gene_target)

# --- Add AUC1/AUC2 log ratio ---
mean_df["AUC1/AUC2"] = mean_df["AUC1"] / mean_df["AUC2"]
mean_df["log(AUC1/AUC2)"] = np.log(mean_df["AUC1/AUC2"])

# --- Merge AUC ratio into mean_df ---
mean_df = mean_df.merge(
    auc_ratio_df[["Sample", "AUC_ratio_mean"]],
    on="Sample",
    how="left"
)

# --- Define metrics for PCA (including AUC ratio) ---
metrics_for_pca = [
    
    "log(AUC1/AUC2)",
    "AUC_ratio_mean",
    "Max_OD",
    "Max_OD_Time (h)",
    "Final_OD",
    "Peak_Prominence_Max"
]

# --- Filter mean_df for xylose condition and chosen metrics ---
pca_df = mean_df[mean_df["Xylose"] == "xylose"].copy()
pca_df = pca_df[["Sample", "Gene_target", "Growth_category"] + metrics_for_pca].set_index("Sample")

# --- Drop samples with missing values in PCA metrics ---

# --- Replace missing metric values with 0 instead of dropping ---
pca_df_clean = pca_df.copy()
pca_df_clean[metrics_for_pca] = pca_df_clean[metrics_for_pca].fillna(0)
# List strains where Peak_Prominence_Max is 0
zero_peak_strains = pca_df_clean[pca_df_clean["Peak_Prominence_Max"] == 0]

print("Number of strains with Peak_Prominence_Max = 0:", len(zero_peak_strains))
print(zero_peak_strains.index.tolist())

# --- Standardize metrics ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_df_clean[metrics_for_pca])

# --- Run PCA ---
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# --- Create PCA result dataframe ---
pca_result_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"],
    index=pca_df_clean.index
)
pca_result_df["Gene_target"] = pca_df_clean["Gene_target"]

# --- Parameters for labeling ---
min_distance = 0.8
max_cluster_size = 5
text_offset = 0.04

positions = pca_result_df[["PC1", "PC2"]].to_numpy()
labels = pca_result_df["Gene_target"].to_numpy()

# Compute pairwise distances
dist_matrix = squareform(pdist(positions))
cluster_sizes = np.sum(dist_matrix < min_distance, axis=1) - 1

# ==========================
#      PCA SCATTER PLOT
#   Color = AUC_ratio_mean
# ==========================

plt.figure(figsize=(9, 7))

# Get color values
color_values = pca_df_clean.loc[pca_result_df.index, "AUC_ratio_mean"]

scatter = plt.scatter(
    pca_result_df["PC1"],
    pca_result_df["PC2"],
    c=color_values,
    cmap="Reds",
    s=40,
    alpha=0.9,
    edgecolors="black",
    linewidths=0.3
)

# Add colorbar
cbar = plt.colorbar(scatter)
cbar.set_label("AUC ratio (xylose / no xylose)")

# --- Add gene target labels with offset ---
for i, (x, y) in enumerate(positions):

    # Skip labeling no_sgrna
    if labels[i].lower() == "no_sgrna":
        continue

    # Label only small clusters
    if cluster_sizes[i] <= max_cluster_size:
        plt.text(x + text_offset, y + text_offset, labels[i], fontsize=8, alpha=0.8)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.title("PCA of Selected Metrics (xylose condition)")
plt.tight_layout()
plt.savefig("pca_AUCratio_colored.svg", format="svg")
plt.show()


# ================================
#  GENERATE SEPARATE PCA PLOTS
# ================================

metrics_to_plot = [
    
    "log(AUC1/AUC2)",
    "AUC_ratio_mean",
    "Max_OD",
    "Max_OD_Time (h)",
    "Final_OD",
    "Peak_Prominence_Max"
]

for metric in metrics_to_plot:
    plt.figure(figsize=(9, 7))

    # Values for coloring
    color_values = pca_df_clean.loc[pca_result_df.index, metric]

    # Scatter plot
    scatter = plt.scatter(
        pca_result_df["PC1"],
        pca_result_df["PC2"],
        c=color_values,
        cmap="Reds",
        s=40,
        alpha=0.9,
        edgecolors="black",
        linewidths=0.3
    )

    # Colorbar
    cbar = plt.colorbar(scatter)
    cbar.set_label(metric)

    # Labeling logic
    for i, (x, y) in enumerate(positions):

       
           

        if cluster_sizes[i] <= max_cluster_size:
            plt.text(x + text_offset, y + text_offset, labels[i], fontsize=8, alpha=0.8)

    plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
    plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
    plt.title(f"PCA Colored by {metric}")

    plt.tight_layout()
    plt.savefig(f"PCA_{metric.replace('/', '_').replace(' ', '_')}.svg", format="svg")
    plt.show()
    
print(f"Number of strains plotted in PCA: {len(pca_result_df)}")
print(f"Number of strains plotted in PCA zondernosgrna: {len(pca_result_df[pca_result_df["Gene_target"].str.lower() != "no_sgrna"])}")



In [ ]:
#groter lettertype:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform

# --- GLOBAL FONT SETTINGS (Doubled) ---
plt.rcParams.update({
    'font.size': 18,          # Base font size
    'axes.titlesize': 24,      # Title font size
    'axes.labelsize': 20,      # X and Y label font size
    'xtick.labelsize': 16,     # X axis tick numbers
    'ytick.labelsize': 16,     # Y axis tick numbers
    'legend.fontsize': 16,     # Legend font size
    'figure.titlesize': 26     # Figure title
})

# [Existing data processing code remains the same...]
# (Keeping the logic until the plotting sections)

# --- Parameters for labeling ---
min_distance = 0.8
max_cluster_size = 5
text_offset = 0.04
gene_label_size = 16  # Doubled from 8

# ==========================
#     PCA SCATTER PLOT
#  Color = AUC_ratio_mean
# ==========================

plt.figure(figsize=(12, 9)) # Increased figure size slightly for larger fonts

color_values = pca_df_clean.loc[pca_result_df.index, "AUC_ratio_mean"]

scatter = plt.scatter(
    pca_result_df["PC1"],
    pca_result_df["PC2"],
    c=color_values,
    cmap="Reds",
    s=60, # Increased dot size slightly to match font
    alpha=0.9,
    edgecolors="black",
    linewidths=0.3
)

# Add colorbar
cbar = plt.colorbar(scatter)
cbar.set_label("AUC ratio (xylose / no xylose)", size=20)
cbar.ax.tick_params(labelsize=16)

# --- Add gene target labels ---
for i, (x, y) in enumerate(positions):
    if labels[i].lower() == "no_sgrna":
        continue
    if cluster_sizes[i] <= max_cluster_size:
        # Changed fontsize to gene_label_size variable
        plt.text(x + text_offset, y + text_offset, labels[i], fontsize=gene_label_size, alpha=0.8)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.title("PCA of Selected Metrics (xylose condition)")
plt.tight_layout()
plt.savefig("pca_AUCratio_colored.svg", format="svg")
plt.show()


# ================================
#  GENERATE SEPARATE PCA PLOTS
# ================================

metrics_to_plot = [
    "log(AUC1/AUC2)",
    "AUC_ratio_mean",
    "Max_OD",
    "Max_OD_Time (h)",
    "Final_OD",
    "Peak_Prominence_Max"
]

for metric in metrics_to_plot:
    plt.figure(figsize=(12, 9))

    color_values = pca_df_clean.loc[pca_result_df.index, metric]

    scatter = plt.scatter(
        pca_result_df["PC1"],
        pca_result_df["PC2"],
        c=color_values,
        cmap="Reds",
        s=60,
        alpha=0.9,
        edgecolors="black",
        linewidths=0.3
    )

    cbar = plt.colorbar(scatter)
    cbar.set_label(metric, size=20)
    cbar.ax.tick_params(labelsize=16)

    for i, (x, y) in enumerate(positions):
        if cluster_sizes[i] <= max_cluster_size:
            plt.text(x + text_offset, y + text_offset, labels[i], fontsize=gene_label_size, alpha=0.8)

    plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
    plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
    plt.title(f"PCA Colored by {metric}")

    plt.tight_layout()
    plt.savefig(f"PCA_{metric.replace('/', '_').replace(' ', '_')}.svg", format="svg")
    plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re

# --- Ensure xylose column is consistent ---
growth_data_with_avgs["Xylose"] = growth_data_with_avgs["Xylose"].str.strip().str.lower()

# --- Select only mean and sd rows ---
mean_df = growth_data_with_avgs[growth_data_with_avgs["rep"] == "mean"]
sd_df = growth_data_with_avgs[growth_data_with_avgs["rep"] == "sd"]

# --- Merge means and SDs for AUC ratio calculation ---
merged_stats = pd.merge(
    mean_df[["Sample", "Xylose", "AUC", "Growth_category"]],
    sd_df[["Sample", "Xylose", "AUC"]],
    on=["Sample", "Xylose"],
    suffixes=("_mean", "_sd")
)

# --- Separate xylose and no xylose ---
df_xylose = merged_stats[merged_stats["Xylose"] == "xylose"].set_index("Sample")
df_noxylose = merged_stats[merged_stats["Xylose"] == "no xylose"].set_index("Sample")

# --- Keep only samples present in both conditions ---
common_samples = df_xylose.index.intersection(df_noxylose.index)
df_xylose = df_xylose.loc[common_samples]
df_noxylose = df_noxylose.loc[common_samples]

# --- Compute AUC ratio and propagated SD ---
ratio_mean = df_xylose["AUC_mean"] / df_noxylose["AUC_mean"]
ratio_sd = ratio_mean * np.sqrt(
    (df_xylose["AUC_sd"] / df_xylose["AUC_mean"])**2 +
    (df_noxylose["AUC_sd"] / df_noxylose["AUC_mean"])**2
)

# --- Combine into AUC ratio dataframe ---
auc_ratio_df = pd.DataFrame({
    "Sample": common_samples,
    "AUC_ratio_mean": ratio_mean,
    "AUC_ratio_sd": ratio_sd,
    "Growth_category": df_xylose["Growth_category"].values
}).reset_index(drop=True)

# --- Extract Gene target from the Sample column ---
def extract_gene_target(sample_str):
    match = re.search(r"Gene target:\s*([\w\-]+)", str(sample_str))
    return match.group(1) if match else sample_str

auc_ratio_df["Gene_target"] = auc_ratio_df["Sample"].apply(extract_gene_target)

# --- Add AUC1/AUC2 log ratio ---
mean_df["AUC1/AUC2"] = mean_df["AUC1"] / mean_df["AUC2"]
mean_df["log(AUC1/AUC2)"] = np.log(mean_df["AUC1/AUC2"])

# --- Merge AUC ratio into mean_df ---
mean_df = mean_df.merge(
    auc_ratio_df[["Sample", "AUC_ratio_mean"]],
    on="Sample",
    how="left"
)

# --- Define metrics for PCA/t-SNE ---
metrics_for_pca = [
    "log(AUC1/AUC2)",
    "AUC_ratio_mean",
    "Max_OD",
    "Max_OD_Time (h)",
    "Final_OD",
    "Peak_Prominence_Max"
]

# --- Filter mean_df for xylose condition and chosen metrics ---
pca_df = mean_df[mean_df["Xylose"] == "xylose"].copy()
pca_df = pca_df[["Sample", "Gene_target", "Growth_category"] + metrics_for_pca].set_index("Sample")

# --- Replace missing metric values with 0 instead of dropping ---
pca_df_clean = pca_df.copy()
pca_df_clean[metrics_for_pca] = pca_df_clean[metrics_for_pca].fillna(0)

# List strains where Peak_Prominence_Max is 0
zero_peak_strains = pca_df_clean[pca_df_clean["Peak_Prominence_Max"] == 0]
print("Number of strains with Peak_Prominence_Max = 0:", len(zero_peak_strains))
print(zero_peak_strains.index.tolist())

# --- Standardize metrics ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_df_clean[metrics_for_pca])

# ============================================================
#                     t-SNE instead of PCA
# ============================================================

from sklearn.manifold import TSNE
from scipy.spatial.distance import pdist, squareform

tsne = TSNE(
    n_components=2,
    perplexity=15,
    learning_rate='auto',
    init='random',
    random_state=42
)
X_tsne = tsne.fit_transform(X_scaled)

# --- Create t-SNE result dataframe ---
tsne_result_df = pd.DataFrame(
    X_tsne,
    columns=["TSNE1", "TSNE2"],
    index=pca_df_clean.index
)
tsne_result_df["Gene_target"] = pca_df_clean["Gene_target"]

# --- Label clustering logic ---
positions = tsne_result_df[["TSNE1", "TSNE2"]].to_numpy()
labels = tsne_result_df["Gene_target"].to_numpy()

min_distance = 0.8
max_cluster_size = 5
text_offset = 0.04

dist_matrix = squareform(pdist(positions))
cluster_sizes = np.sum(dist_matrix < min_distance, axis=1) - 1

# --- Plot t-SNE ---
plt.figure(figsize=(9, 7))

# Black non-control points
non_control = tsne_result_df[tsne_result_df["Gene_target"].str.lower() != "no_sgrna"]
plt.scatter(
    non_control["TSNE1"],
    non_control["TSNE2"],
    color='black',
    label='All genes',
    s=30,
    alpha=0.8,
    zorder=1
)

# Red no_sgRNA points
control = tsne_result_df[tsne_result_df["Gene_target"].str.lower() == "no_sgrna"]
if not control.empty:
    plt.scatter(
        control["TSNE1"],
        control["TSNE2"],
        color='red',
        label='no_sgRNA',
        s=30,
        alpha=0.8,
        zorder=2
    )

# Add gene target labels
for i, (x, y) in enumerate(positions):
    if labels[i].lower() == "no_sgrna":
        continue
    if cluster_sizes[i] <= max_cluster_size:
        plt.text(x + text_offset, y + text_offset, labels[i], fontsize=8, alpha=0.8, zorder=3)

plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("t-SNE of Selected Metrics (xylose condition)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("tSNE_Red_Black.svg", format="svg")
plt.show()

print("Number of strains plotted in t-SNE:", len(tsne_result_df))
print("Number of strains plotted zonder no_sgRNA:", 
      len(tsne_result_df[tsne_result_df["Gene_target"].str.lower() != "no_sgrna"]))


In [ ]:
#zonder gewone auc en auc1/auc2, #dit is bij missing values 0 ingevuld

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re

# --- Ensure xylose column is consistent ---
growth_data_with_avgs["Xylose"] = growth_data_with_avgs["Xylose"].str.strip().str.lower()

# --- Select only mean and sd rows ---
mean_df = growth_data_with_avgs[growth_data_with_avgs["rep"] == "mean"]
sd_df = growth_data_with_avgs[growth_data_with_avgs["rep"] == "sd"]

# --- Merge means and SDs for AUC ratio calculation ---
merged_stats = pd.merge(
    mean_df[["Sample", "Xylose", "AUC", "Growth_category"]],
    sd_df[["Sample", "Xylose", "AUC"]],
    on=["Sample", "Xylose"],
    suffixes=("_mean", "_sd")
)

# --- Separate xylose and no xylose ---
df_xylose = merged_stats[merged_stats["Xylose"] == "xylose"].set_index("Sample")
df_noxylose = merged_stats[merged_stats["Xylose"] == "no xylose"].set_index("Sample")

# --- Keep only samples present in both conditions ---
common_samples = df_xylose.index.intersection(df_noxylose.index)
df_xylose = df_xylose.loc[common_samples]
df_noxylose = df_noxylose.loc[common_samples]

# --- Compute AUC ratio and propagated SD ---
ratio_mean = df_xylose["AUC_mean"] / df_noxylose["AUC_mean"]
ratio_sd = ratio_mean * np.sqrt(
    (df_xylose["AUC_sd"] / df_xylose["AUC_mean"])**2 +
    (df_noxylose["AUC_sd"] / df_noxylose["AUC_mean"])**2
)

# --- Combine into AUC ratio dataframe ---
auc_ratio_df = pd.DataFrame({
    "Sample": common_samples,
    "AUC_ratio_mean": ratio_mean,
    "AUC_ratio_sd": ratio_sd,
    "Growth_category": df_xylose["Growth_category"].values
}).reset_index(drop=True)

# --- Extract Gene target from the Sample column ---
def extract_gene_target(sample_str):
    match = re.search(r"Gene target:\s*([\w\-]+)", str(sample_str))
    return match.group(1) if match else sample_str

auc_ratio_df["Gene_target"] = auc_ratio_df["Sample"].apply(extract_gene_target)

# --- Add AUC1/AUC2 log ratio ---
mean_df["AUC1/AUC2"] = mean_df["AUC1"] / mean_df["AUC2"]
mean_df["log(AUC1/AUC2)"] = np.log(mean_df["AUC1/AUC2"])

# --- Merge AUC ratio into mean_df ---
mean_df = mean_df.merge(
    auc_ratio_df[["Sample", "AUC_ratio_mean"]],
    on="Sample",
    how="left"
)

# --- Define metrics for PCA (including AUC ratio) ---
metrics_for_pca = [
    # --- AUC-related ---
   
   
    
    "AUC_ratio_mean",
    
    # --- OD (Optical Density) Metrics ---
    "Max_OD",
    "Max_OD_Time (h)",
    "Final_OD",
    
    # --- Peak Metrics ---
    "Peak_Prominence_Max"
]

# --- Filter mean_df for xylose condition and chosen metrics ---
pca_df = mean_df[mean_df["Xylose"] == "xylose"].copy()
pca_df = pca_df[["Sample", "Gene_target", "Growth_category"] + metrics_for_pca].set_index("Sample")

# --- Replace missing metric values with 0 instead of dropping ---
pca_df_clean = pca_df.copy()
pca_df_clean[metrics_for_pca] = pca_df_clean[metrics_for_pca].fillna(0)
# List strains where Peak_Prominence_Max is 0
zero_peak_strains = pca_df_clean[pca_df_clean["Peak_Prominence_Max"] == 0]

print("Number of strains with Peak_Prominence_Max = 0:", len(zero_peak_strains))
print(zero_peak_strains.index.tolist())




# --- Standardize metrics ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_df_clean[metrics_for_pca])

# --- Run PCA ---
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# --- Create PCA result dataframe ---
pca_result_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"],
    index=pca_df_clean.index
)
pca_result_df["Gene_target"] = pca_df_clean["Gene_target"]

# --- Plot PCA with points in black except for no_sgRNA in red ---
import numpy as np
from scipy.spatial.distance import pdist, squareform

# --- Parameters for labeling ---
min_distance = 0.8   # distance threshold to consider "close"
max_cluster_size = 5  # clusters larger than this are not labeled
text_offset = 0.04    # offset for text from the points

positions = pca_result_df[["PC1", "PC2"]].to_numpy()
labels = pca_result_df["Gene_target"].to_numpy()

# Compute pairwise distances
dist_matrix = squareform(pdist(positions))

# Determine cluster sizes (number of neighbors within min_distance)
cluster_sizes = np.sum(dist_matrix < min_distance, axis=1) - 1  # subtract 1 to ignore self

# --- Plot PCA ---
plt.figure(figsize=(9, 7))

# Plot non-control samples (all genes except no_sgRNA) first
non_control = pca_result_df[pca_result_df["Gene_target"].str.lower() != "no_sgrna"]
plt.scatter(
    non_control["PC1"],
    non_control["PC2"],
    color='black',
    label='All genes',
    s=30,
    alpha=0.8,
    zorder=1  # background
)

# Plot control samples (no_sgRNA) last to appear in foreground
control = pca_result_df[pca_result_df["Gene_target"].str.lower() == "no_sgrna"]
if not control.empty:
    plt.scatter(
        control["PC1"],
        control["PC2"],
        color='red',
        label='no_sgRNA',
        s=30,
        alpha=0.8,
        zorder=2  # foreground
    )

# --- Add gene target labels with offset ---
for i, (x, y) in enumerate(positions):
    # Label if small cluster (<= max_cluster_size) or always for no_sgRNA, schrijf no_sgrna in keline letterso meht te laten werken
    # --- Add labels only for non-no_sgrna + small clusters ---

    if labels[i].lower() == "no_sgrna":
        continue

    if cluster_sizes[i] <= max_cluster_size or labels[i].lower() == "no_sgRNA":
        plt.text(x + text_offset, y + text_offset, labels[i], fontsize=8, alpha=0.8, zorder=3)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.title("PCA of Selected Metrics (xylose condition)")
#plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"pcaRed_BlackJuist1.svg", format="svg")
plt.show()



print(f"Number of strains plotted in PCA: {len(pca_result_df)}")
print(f"Number of strains plotted in PCA zonder nosgrna: {len(pca_result_df[pca_result_df["Gene_target"].str.lower() != "no_sgrna"])}")


In [ ]:
#dit is bij missing values 0 ingevuld

from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
from scipy.spatial.distance import pdist, squareform
from mpl_toolkits.mplot3d import Axes3D

# --- Ensure xylose column is consistent ---
growth_data_with_avgs["Xylose"] = growth_data_with_avgs["Xylose"].str.strip().str.lower()

# --- Select only mean and sd rows ---
mean_df = growth_data_with_avgs[growth_data_with_avgs["rep"] == "mean"]
sd_df = growth_data_with_avgs[growth_data_with_avgs["rep"] == "sd"]

# --- Merge means and SDs for AUC ratio calculation ---
merged_stats = pd.merge(
    mean_df[["Sample", "Xylose", "AUC", "Growth_category"]],
    sd_df[["Sample", "Xylose", "AUC"]],
    on=["Sample", "Xylose"],
    suffixes=("_mean", "_sd")
)

# --- Separate xylose and no xylose ---
df_xylose = merged_stats[merged_stats["Xylose"] == "xylose"].set_index("Sample")
df_noxylose = merged_stats[merged_stats["Xylose"] == "no xylose"].set_index("Sample")

# --- Keep only samples present in both conditions ---
common_samples = df_xylose.index.intersection(df_noxylose.index)
df_xylose = df_xylose.loc[common_samples]
df_noxylose = df_noxylose.loc[common_samples]

# --- Compute AUC ratio and propagated SD ---
ratio_mean = df_xylose["AUC_mean"] / df_noxylose["AUC_mean"]
ratio_sd = ratio_mean * np.sqrt(
    (df_xylose["AUC_sd"] / df_xylose["AUC_mean"])**2 +
    (df_noxylose["AUC_sd"] / df_noxylose["AUC_mean"])**2
)

# --- Combine into AUC ratio dataframe ---
auc_ratio_df = pd.DataFrame({
    "Sample": common_samples,
    "AUC_ratio_mean": ratio_mean,
    "AUC_ratio_sd": ratio_sd,
    "Growth_category": df_xylose["Growth_category"].values
}).reset_index(drop=True)

# --- Extract Gene target from the Sample column ---
def extract_gene_target(sample_str):
    match = re.search(r"Gene target:\s*([\w\-]+)", str(sample_str))
    return match.group(1) if match else sample_str

auc_ratio_df["Gene_target"] = auc_ratio_df["Sample"].apply(extract_gene_target)

# --- Add AUC1/AUC2 log ratio ---
mean_df["AUC1/AUC2"] = mean_df["AUC1"] / mean_df["AUC2"]
mean_df["log(AUC1/AUC2)"] = np.log(mean_df["AUC1/AUC2"])

# --- Merge AUC ratio into mean_df ---
mean_df = mean_df.merge(
    auc_ratio_df[["Sample", "AUC_ratio_mean"]],
    on="Sample",
    how="left"
)

# --- Define metrics for downstream analysis ---
metrics_for_plot = [
    "AUC_ratio_mean",
    "Max_OD_Time (h)",
    "Peak_Prominence_Max"
]

# --- Filter mean_df for xylose condition and chosen metrics ---
pca_df = mean_df[mean_df["Xylose"] == "xylose"].copy()
pca_df = pca_df[["Sample", "Gene_target", "Growth_category"] + metrics_for_plot].set_index("Sample")

# --- Replace missing metric values with 0 instead of dropping ---
pca_df_clean = pca_df.copy()
pca_df_clean[metrics_for_plot] = pca_df_clean[metrics_for_plot].fillna(0)

# List strains where Peak_Prominence_Max is 0
zero_peak_strains = pca_df_clean[pca_df_clean["Peak_Prominence_Max"] == 0]

print("Number of strains with Peak_Prominence_Max = 0:", len(zero_peak_strains))
print(zero_peak_strains.index.tolist())

# --- Prepare data for the 3D scatterplot ---
X = pca_df_clean["AUC_ratio_mean"].values
Y = pca_df_clean["Max_OD_Time (h)"].values
Z = pca_df_clean["Peak_Prominence_Max"].values
labels = pca_df_clean["Gene_target"].values

positions_3d = np.column_stack([X, Y, Z])

# --- Label logic (same as PCA labeling) ---
min_distance = 0.8
max_cluster_size = 5
text_offset = 0.02

dist_matrix = squareform(pdist(positions_3d))
cluster_sizes = np.sum(dist_matrix < min_distance, axis=1) - 1

# --- Create 3D plot ---
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Non-control = black
non_control = pca_df_clean[pca_df_clean["Gene_target"].str.lower() != "no_sgrna"]
ax.scatter(
    non_control["AUC_ratio_mean"],
    non_control["Max_OD_Time (h)"],
    non_control["Peak_Prominence_Max"],
    color='black',
    s=30,
    alpha=0.8,
    label="All genes"
)

# no_sgRNA = red
control = pca_df_clean[pca_df_clean["Gene_target"].str.lower() == "no_sgrna"]
ax.scatter(
    control["AUC_ratio_mean"],
    control["Max_OD_Time (h)"],
    control["Peak_Prominence_Max"],
    color='red',
    s=40,
    alpha=1.0,
    label="no_sgRNA"
)

# --- Add labels ---
for i, (x, y, z) in enumerate(positions_3d):
    if labels[i].lower() == "no_sgrna":
        continue
    if cluster_sizes[i] <= max_cluster_size:
        ax.text(
            x + text_offset,
            y + text_offset,
            z + text_offset,
            labels[i],
            fontsize=7
        )

# --- Axis labels ---
ax.set_xlabel("AUC_ratio_mean")
ax.set_ylabel("Max_OD_Time (h)")
ax.set_zlabel("Peak_Prominence_Max")

plt.title("3D Scatterplot of Growth Metrics")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re

# --- Ensure xylose column is consistent ---
growth_data_with_avgs["Xylose"] = growth_data_with_avgs["Xylose"].str.strip().str.lower()

# --- Select only mean and sd rows ---
mean_df = growth_data_with_avgs[growth_data_with_avgs["rep"] == "mean"]
sd_df = growth_data_with_avgs[growth_data_with_avgs["rep"] == "sd"]

# --- Merge means and SDs for AUC ratio calculation ---
merged_stats = pd.merge(
    mean_df[["Sample", "Xylose", "AUC", "Growth_category"]],
    sd_df[["Sample", "Xylose", "AUC"]],
    on=["Sample", "Xylose"],
    suffixes=("_mean", "_sd")
)

# --- Separate xylose and no xylose ---
df_xylose = merged_stats[merged_stats["Xylose"] == "xylose"].set_index("Sample")
df_noxylose = merged_stats[merged_stats["Xylose"] == "no xylose"].set_index("Sample")

# --- Keep only samples present in both conditions ---
common_samples = df_xylose.index.intersection(df_noxylose.index)
df_xylose = df_xylose.loc[common_samples]
df_noxylose = df_noxylose.loc[common_samples]

# --- Compute AUC ratio and propagated SD ---
ratio_mean = df_xylose["AUC_mean"] / df_noxylose["AUC_mean"]
ratio_sd = ratio_mean * np.sqrt(
    (df_xylose["AUC_sd"] / df_xylose["AUC_mean"])**2 +
    (df_noxylose["AUC_sd"] / df_noxylose["AUC_mean"])**2
)

# --- Combine into AUC ratio dataframe ---
auc_ratio_df = pd.DataFrame({
    "Sample": common_samples,
    "AUC_ratio_mean": ratio_mean,
    "AUC_ratio_sd": ratio_sd,
    "Growth_category": df_xylose["Growth_category"].values
}).reset_index(drop=True)

# --- Extract Gene target from the Sample column ---
def extract_gene_target(sample_str):
    match = re.search(r"Gene target:\s*([\w\-]+)", str(sample_str))
    return match.group(1) if match else sample_str

auc_ratio_df["Gene_target"] = auc_ratio_df["Sample"].apply(extract_gene_target)

# --- Add AUC1/AUC2 log ratio ---
mean_df["AUC1/AUC2"] = mean_df["AUC1"] / mean_df["AUC2"]
mean_df["log(AUC1/AUC2)"] = np.log(mean_df["AUC1/AUC2"])

# --- Merge AUC ratio into mean_df ---
mean_df = mean_df.merge(
    auc_ratio_df[["Sample", "AUC_ratio_mean"]],
    on="Sample",
    how="left"
)

# --- Define metrics for PCA (including AUC ratio) ---
metrics_for_pca = [
    # --- AUC-related ---
    
   
    "log(AUC1/AUC2)",
    "AUC_ratio_mean",
    
    # --- OD (Optical Density) Metrics ---
    "Max_OD",
    "Max_OD_Time (h)",
    "Final_OD",
    
    # --- Peak Metrics ---
    "Peak_Prominence_Max"
]

# --- Filter mean_df for xylose condition and chosen metrics ---
pca_df = mean_df[mean_df["Xylose"] == "xylose"].copy()
pca_df = pca_df[["Sample", "Gene_target", "Growth_category"] + metrics_for_pca].set_index("Sample")

# --- Drop samples with missing values in PCA metrics ---
# --- Replace missing metric values with 0 instead of dropping ---
pca_df_clean = pca_df.copy()
pca_df_clean[metrics_for_pca] = pca_df_clean[metrics_for_pca].fillna(0)
# List strains where Peak_Prominence_Max is 0
zero_peak_strains = pca_df_clean[pca_df_clean["Peak_Prominence_Max"] == 0]

print("Number of strains with Peak_Prominence_Max = 0:", len(zero_peak_strains))
print(zero_peak_strains.index.tolist())

# --- Standardize metrics ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_df_clean[metrics_for_pca])

# --- Run PCA ---
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# --- Create PCA result dataframe ---
pca_result_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"],
    index=pca_df_clean.index
)
pca_result_df["Gene_target"] = pca_df_clean["Gene_target"]

# --- Plot PCA with points in black except for no_sgRNA in red ---
import numpy as np
from scipy.spatial.distance import pdist, squareform

# --- Parameters for labeling ---
min_distance = 0.8   # distance threshold to consider "close"
max_cluster_size = 5  # clusters larger than this are not labeled
text_offset = 0.04    # offset for text from the points

positions = pca_result_df[["PC1", "PC2"]].to_numpy()
labels = pca_result_df["Gene_target"].to_numpy()

# Compute pairwise distances
dist_matrix = squareform(pdist(positions))

# Determine cluster sizes (number of neighbors within min_distance)
cluster_sizes = np.sum(dist_matrix < min_distance, axis=1) - 1  # subtract 1 to ignore self

# --- Plot PCA ---
plt.figure(figsize=(9, 7))

# Plot non-control samples (all genes except no_sgRNA) first
non_control = pca_result_df[pca_result_df["Gene_target"].str.lower() != "no_sgrna"]
plt.scatter(
    non_control["PC1"],
    non_control["PC2"],
    color='black',
    label='All genes',
    s=30,
    alpha=0.8,
    zorder=1  # background
)

# Plot control samples (no_sgRNA) last to appear in foreground
control = pca_result_df[pca_result_df["Gene_target"].str.lower() == "no_sgrna"]
if not control.empty:
    plt.scatter(
        control["PC1"],
        control["PC2"],
        color='red',
        
        s=30,
        alpha=0.8,
        zorder=2  # foreground
    )
# --- Highlight specific gene targets ---
highlight_genes = ["dxr", "sufs", "pgm", "pheS","ylan27_1","ylan27_2","acps50_1","acps50_2", "ftsz226","ftsz","mreC"]  # <--- write small letters

highlight_df = pca_result_df[pca_result_df["Gene_target"].str.lower().isin(highlight_genes)]

# Plot highlighted points in cyan
plt.scatter(
    highlight_df["PC1"],
    highlight_df["PC2"],
    color='cyan',
    s=30,
    edgecolor='black',
    linewidth=0.8,
    zorder=5,
    label="Highlighted genes"
)

# Force labeling of highlighted genes
force_label_indices = highlight_df.index.tolist()
for i, (x, y) in enumerate(positions):
    gt = labels[i].lower()

    if (i in force_label_indices) or (cluster_sizes[i] <= max_cluster_size) or (gt == "no_sgrna"):
        plt.text(x + text_offset, y + text_offset, labels[i], fontsize=8, alpha=0.9, zorder=6)



plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.title("PCA of Selected Metrics (xylose condition)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"pcaRed_BlackJuist1.svg", format="svg")
plt.show()


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re

# --- Ensure xylose column is consistent ---
growth_data_with_avgs["Xylose"] = growth_data_with_avgs["Xylose"].str.strip().str.lower()

# --- Select only mean and sd rows ---
mean_df = growth_data_with_avgs[growth_data_with_avgs["rep"] == "mean"]
sd_df = growth_data_with_avgs[growth_data_with_avgs["rep"] == "sd"]

# --- Merge means and SDs for AUC ratio calculation ---
merged_stats = pd.merge(
    mean_df[["Sample", "Xylose", "AUC", "Growth_category"]],
    sd_df[["Sample", "Xylose", "AUC"]],
    on=["Sample", "Xylose"],
    suffixes=("_mean", "_sd")
)

# --- Separate xylose and no xylose ---
df_xylose = merged_stats[merged_stats["Xylose"] == "xylose"].set_index("Sample")
df_noxylose = merged_stats[merged_stats["Xylose"] == "no xylose"].set_index("Sample")

# --- Keep only samples present in both conditions ---
common_samples = df_xylose.index.intersection(df_noxylose.index)
df_xylose = df_xylose.loc[common_samples]
df_noxylose = df_noxylose.loc[common_samples]

# --- Compute AUC ratio and propagated SD ---
ratio_mean = df_xylose["AUC_mean"] / df_noxylose["AUC_mean"]
ratio_sd = ratio_mean * np.sqrt(
    (df_xylose["AUC_sd"] / df_xylose["AUC_mean"])**2 +
    (df_noxylose["AUC_sd"] / df_noxylose["AUC_mean"])**2
)

# --- Combine into AUC ratio dataframe ---
auc_ratio_df = pd.DataFrame({
    "Sample": common_samples,
    "AUC_ratio_mean": ratio_mean,
    "AUC_ratio_sd": ratio_sd,
    "Growth_category": df_xylose["Growth_category"].values
}).reset_index(drop=True)

# --- Extract Gene target from the Sample column ---
def extract_gene_target(sample_str):
    match = re.search(r"Gene target:\s*([\w\-]+)", str(sample_str))
    return match.group(1) if match else sample_str

auc_ratio_df["Gene_target"] = auc_ratio_df["Sample"].apply(extract_gene_target)

# --- Add AUC1/AUC2 log ratio ---
mean_df["AUC1/AUC2"] = mean_df["AUC1"] / mean_df["AUC2"]
mean_df["log(AUC1/AUC2)"] = np.log(mean_df["AUC1/AUC2"])

# --- Merge AUC ratio into mean_df ---
mean_df = mean_df.merge(
    auc_ratio_df[["Sample", "AUC_ratio_mean"]],
    on="Sample",
    how="left"
)

# --- Define metrics for PCA (including AUC ratio) ---
metrics_for_pca = [
    # --- AUC-related ---
    "AUC",
    
    "log(AUC1/AUC2)",
    "AUC_ratio_mean",
    
    # --- OD (Optical Density) Metrics ---
    "Max_OD",
    "Max_OD_Time (h)",
    "Final_OD",
    
    # --- Peak Metrics ---
    "Peak_Prominence_Max"
]

# --- Filter mean_df for xylose condition and chosen metrics ---
pca_df = mean_df[mean_df["Xylose"] == "xylose"].copy()
pca_df = pca_df[["Sample", "Gene_target", "Growth_category"] + metrics_for_pca].set_index("Sample")

# --- Drop samples with missing values in PCA metrics ---
pca_df_clean = pca_df.dropna(subset=metrics_for_pca)

# --- Standardize metrics ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_df_clean[metrics_for_pca])

# --- Run PCA ---
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# --- Create PCA result dataframe ---
pca_result_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"],
    index=pca_df_clean.index
)
pca_result_df["Gene_target"] = pca_df_clean["Gene_target"]

# --- Plot PCA with points in black except for no_sgRNA in red ---
import numpy as np
from scipy.spatial.distance import pdist, squareform

# --- Parameters for labeling ---
min_distance = 0.8   # distance threshold to consider "close"
max_cluster_size = 5  # clusters larger than this are not labeled
text_offset = 0.04    # offset for text from the points

# --- Use Sample names as labels instead of Gene_target ---
positions = pca_result_df[["PC1", "PC2"]].to_numpy()
labels = pca_result_df.index.to_numpy()  # <-- sample names

# Compute pairwise distances
dist_matrix = squareform(pdist(positions))

# Determine cluster sizes (number of neighbors within min_distance)
cluster_sizes = np.sum(dist_matrix < min_distance, axis=1) - 1  # subtract 1 to ignore self

# --- Plot PCA ---
plt.figure(figsize=(9, 7))

# Plot non-control samples (all genes except no_sgRNA) first
non_control = pca_result_df[pca_result_df["Gene_target"].str.lower() != "no_sgrna"]
plt.scatter(
    non_control["PC1"],
    non_control["PC2"],
    color='black',
    label='All genes',
    s=30,
    alpha=0.8,
    zorder=1
)

# Plot control samples (no_sgRNA) last to appear in foreground
control = pca_result_df[pca_result_df["Gene_target"].str.lower() == "no_sgrna"]
if not control.empty:
    plt.scatter(
        control["PC1"],
        control["PC2"],
        color='red',
        label='no_sgRNA',
        s=30,
        alpha=0.8,
        zorder=2
    )

# --- Add sample name labels with offset ---
for i, (x, y) in enumerate(positions):
    # Label if small cluster (<= max_cluster_size) or always for no_sgRNA
    if cluster_sizes[i] <= max_cluster_size or pca_result_df["Gene_target"].iloc[i].lower() == "no_sgRNA":
        plt.text(x + text_offset, y + text_offset, labels[i], fontsize=8, alpha=0.8, zorder=3)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.title("PCA of Selected Metrics (xylose condition)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"pca.svg", format="svg")
plt.show()


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
#'AUC': auc,
        #'AUC1': auc1,
        #'AUC2': auc2,
        #'Max_OD': max_od,
        #'Max_OD_Time (h)': max_od_time / 60,  # convert minutes to hours
        #'Final_OD': last_od,
        ##'Max_Slope': max_slope,
        #'Decline_Rate': decline_rate,
        #'Var_dODdt': var_dODdt,
        # Peak summary metrics
        #'Num_Peaks': num_peaks,
        #'Mean_Peak_Height': mean_peak_height,
        #'Max_Peak_Height': max_peak_height,
        #'Mean_Peak_Width': mean_peak_width,
        #'Max_Peak_Width': max_peak_width,
       # 'First_Peak_Time': first_peak_time,
        #'Last_Peak_Time': last_peak_time
# --- Ensure xylose column is consistent ---
growth_data_with_avgs["Xylose"] = growth_data_with_avgs["Xylose"].str.strip().str.lower()

# --- Select only mean and sd rows ---
mean_df = growth_data_with_avgs[growth_data_with_avgs["rep"] == "mean"]
sd_df = growth_data_with_avgs[growth_data_with_avgs["rep"] == "sd"]

# --- Merge means and SDs for AUC ratio calculation ---
merged_stats = pd.merge(
    mean_df[["Sample", "Xylose", "AUC", "Growth_category"]],
    sd_df[["Sample", "Xylose", "AUC"]],
    on=["Sample", "Xylose"],
    suffixes=("_mean", "_sd")
)

# --- Separate xylose and no xylose ---
df_xylose = merged_stats[merged_stats["Xylose"] == "xylose"].set_index("Sample")
df_noxylose = merged_stats[merged_stats["Xylose"] == "no xylose"].set_index("Sample")

# --- Keep only samples present in both conditions ---
common_samples = df_xylose.index.intersection(df_noxylose.index)
df_xylose = df_xylose.loc[common_samples]
df_noxylose = df_noxylose.loc[common_samples]

# --- Compute AUC ratio and propagated SD ---
ratio_mean = df_xylose["AUC_mean"] / df_noxylose["AUC_mean"]
ratio_sd = ratio_mean * np.sqrt(
    (df_xylose["AUC_sd"] / df_xylose["AUC_mean"])**2 +
    (df_noxylose["AUC_sd"] / df_noxylose["AUC_mean"])**2
)

# --- Combine into AUC ratio dataframe ---
auc_ratio_df = pd.DataFrame({
    "Sample": common_samples,
    "AUC_ratio_mean": ratio_mean,
    "AUC_ratio_sd": ratio_sd,
    "Growth_category": df_xylose["Growth_category"].values
}).reset_index(drop=True)

# --- Extract Gene target from the Sample column ---
def extract_gene_target(sample_str):
    match = re.search(r"Gene target:\s*([\w\-]+)", str(sample_str))
    return match.group(1) if match else sample_str

auc_ratio_df["Gene_target"] = auc_ratio_df["Sample"].apply(extract_gene_target)

# --- Add AUC1/AUC2 log ratio ---
mean_df["AUC1/AUC2"] = mean_df["AUC1"] / mean_df["AUC2"]
mean_df["log(AUC1/AUC2)"] = np.log(mean_df["AUC1/AUC2"])

# --- Merge AUC ratio into mean_df ---
mean_df = mean_df.merge(
    auc_ratio_df[["Sample", "AUC_ratio_mean"]],
    on="Sample",
    how="left"
)

# --- Define metrics for PCA (including AUC ratio) ---
metrics_for_pca = [
    # --- AUC-related ---
    "AUC",
    
    "log(AUC1/AUC2)",
    "AUC_ratio_mean",
    
    # --- OD (Optical Density) Metrics ---
    "Max_OD",
    "Max_OD_Time (h)",
    "Final_OD",
    
    
    # --- Peak Metrics ---
    
    "Peak_Prominence_Max"
    
    
    
]


# --- Filter mean_df for xylose condition and chosen metrics ---
pca_df = mean_df[mean_df["Xylose"] == "xylose"].copy()
pca_df = pca_df[["Sample", "Gene_target", "Growth_category"] + metrics_for_pca].set_index("Sample")

# --- Drop samples with missing values in PCA metrics ---
pca_df_clean = pca_df.dropna(subset=metrics_for_pca)

# --- Standardize metrics ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_df_clean[metrics_for_pca])

# --- Run PCA ---
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# --- Create PCA result dataframe ---
pca_result_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"],
    index=pca_df_clean.index
)
pca_result_df["Gene_target"] = pca_df_clean["Gene_target"]
pca_result_df["Growth_category"] = pca_df_clean["Growth_category"]

# --- Define color map for Growth categories ---
categories = pca_result_df["Growth_category"].dropna().unique()
colors = plt.cm.tab10.colors  # up to 10 distinct colors
color_map = {cat: colors[i % len(colors)] for i, cat in enumerate(categories)}

# --- Plot PCA colored by Growth_category ---
# --- Plot PCA colored by Growth_category with selective labels ---
import numpy as np
from scipy.spatial.distance import pdist, squareform

# Parameters
min_distance = 0.7  # distance threshold to consider “close”
max_cluster_size = 5  # clusters larger than this are not labeled

positions = pca_result_df[["PC1", "PC2"]].to_numpy()
labels = pca_result_df["Gene_target"].to_numpy()

# Compute pairwise distances
dist_matrix = squareform(pdist(positions))

# Determine cluster sizes
cluster_sizes = np.sum(dist_matrix < min_distance, axis=1)  # counts self + neighbors

import numpy as np
from scipy.spatial.distance import pdist, squareform

# Parameters
min_distance = 0.8  # distance threshold to consider "close"
max_cluster_size = 5  # clusters larger than this are not labeled

positions = pca_result_df[["PC1", "PC2"]].to_numpy()
labels = pca_result_df["Gene_target"].to_numpy()

# Compute pairwise distances
dist_matrix = squareform(pdist(positions))

# Determine cluster sizes
cluster_sizes = np.sum(dist_matrix < min_distance, axis=1) - 1  # subtract 1 to ignore self

plt.figure(figsize=(10, 7))

categories = pca_result_df["Growth_category"].dropna().unique()
colors = plt.cm.tab10.colors

# Default category color map
color_map = {cat: colors[i % len(colors)] for i, cat in enumerate(categories)}

# --- Manually override the color for "linear growth curve" ---
if "Linear growth curve" in color_map:
    color_map["Linear growth curve"] = "purple"   # <-- manual override

# --- Remove red from categories automatically so no conflict with no_sgrna ---
for cat, c in color_map.items():
    if isinstance(c, tuple) and np.allclose(c, (1.0, 0.0, 0.0)):  # if tab10 assigned red
        color_map[cat] = colors[1]   # replace with another tab10 color

# --- Plot all non-no_sgrna points ---
for category in categories:
    subset = pca_result_df[
        (pca_result_df["Growth_category"] == category) &
        (pca_result_df["Gene_target"].str.lower() != "no_sgrna")
    ]
    plt.scatter(
        subset["PC1"],
        subset["PC2"],
        color=color_map[category],
        label=category,
        s=30,
        alpha=0.8
    )

# --- Plot no_sgrna points in red ---
nosg_subset = pca_result_df[pca_result_df["Gene_target"].str.lower() == "no_sgrna"]
plt.scatter(
    nosg_subset["PC1"],
    nosg_subset["PC2"],
    color="red",     # <-- remains red
    s=40,
    alpha=1.0,
    label="no_sgrna"
)
text_offset = 0.04
# --- Add labels only for non-no_sgrna + small clusters ---
for i, (x, y) in enumerate(positions):
    if labels[i].lower() == "no_sgrna":
        continue

    if cluster_sizes[i] <= max_cluster_size:
        #plt.text(x, y, labels[i], fontsize=8, alpha=0.8)
        plt.text(x + text_offset, y + text_offset, labels[i], fontsize=8, alpha=0.8, zorder=3)

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.title("PCA of Selected Metrics (xylose condition, colored by Growth Category)")

plt.legend(title="Growth Category", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"pcaGroxthcatjuist.svg",format='svg')
plt.show()




